In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from itertools import product
from pathlib import Path

from EpiLearnSpatialTemporal.dataset import UniversalDataset
from EpiLearnSpatialTemporal import metrics
from EpiLearnSpatialTemporal.utils import generate_dataset

In [ ]:
datasets = ['ILI2019', 'NCHSdeaths', 'JHUcase', 'HHShosp', 'AUcase', 'CAcase', 'CANpositivity', 'CHNGinpatient', 'CHNGoutpatient', 'CPRadmissions', 'DVcli']
#datasets = ['DVcli']
horizons = [1, 2, 4, 7, 10, 14, 28]
model_names = [
    "repeat_last",
    "ARIMA",
    "Dlinear",
    "AGCRN",
    "ColaGNN",
    "DCRNN",
    "EpiGNN",
    "MTGNN",
    "STGCN",
    "GTS",
    "StemGNN",
    "STNorm",
    "EARTH",
    "GraphWaveNet",
]

metric_cols = [
    "mse",
    "mae",
    "rmse",
    "mse_filtered",
    "mae_filtered",
    "rmse_filtered",
    "medse",
    "medae",
]

epi_options = [False, "ngm", "sir_incidence", "sir_percent"]
einn_options = [True, False]
filter_options = [True, False]
ti_options = [True, False]

### LOAD DATA

In [ ]:
def read_file(filename):
    df = pd.read_csv(filename, sep=',')
    for col in ["einn", "filter", "ti"]:
        df[col] = df[col].map(
            {"True": True, "False": False, True: True, False: False}
        )

    # mixed column: keep False as bool, keep model names as strings
    df["epi"] = df["epi"].map(
        {
            "False": False,
            False: False,
            "ngm": "ngm",
            "sir_incidence": "sir_percent",
            "sir_percent": "sir_percent",
        }
    )
    return df
results_dfs = {}
results_dfs_outbreak = {}
results_dfs_non_outbreak = {}
for dataset in datasets:
    filename = f'whole_results_submission2_{dataset}.csv'
    outbreak_filename = f'outbreak_results_submission2_{dataset}.csv'
    non_outbreak_filename = f'non_outbreak_results_submission2_{dataset}.csv'
    
    df = read_file(filename)
    results_dfs[dataset] = df
    
    df = read_file(outbreak_filename)
    results_dfs_outbreak[dataset] = df

    df = read_file(non_outbreak_filename)
    results_dfs_non_outbreak[dataset] = df

tusoai_filename = 'whole_results_tusoai_variants_all_datasets.csv'
tusoai_df = pd.read_csv(tusoai_filename, index_col=0, sep=',')

### PATCHES PLOT

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
from pathlib import Path
from matplotlib.patches import Patch

# =========================================================
# Editable parameters
# =========================================================

metric_cols = [
    #"mse",
    #"mae",
    "rmse",
    #"mse_filtered",
    #"rmse_filtered",
    #"mae_filtered",
    #"medse",
    #"medae",
]
group_cols = ["epi", "einn", "filter", "ti"]

# Candidate option order (used only for deterministic combo ordering)
epi_options = [False, "ngm", "sir_percent"]
einn_options = [False, True]
filter_options = [False, True]
ti_options = [False, True]

# Whether to exclude the all-False combo from the plot
EXCLUDE_ALL_FALSE_COMBO = True

# Figure / layout
FIGSIZE_HEIGHT = 12
FIGSIZE_WIDTH_PER_COMBO = 1.1
FIGSIZE_MIN_WIDTH = 12
TIGHT_LAYOUT = True

# Fonts
TITLE_FONTSIZE = 32
AXIS_LABEL_FONTSIZE = 40
XTICK_FONTSIZE = 32
YTICK_FONTSIZE = 32
LEGEND_FONTSIZE = 28

# Text
Y_LABEL = "Error relative to unpatched"
X_LABEL = ""
TITLE_ALL_COMBOS = "Epidemic patch utility"
TITLE_REDUCED_COMBOS = ""
ORIGINAL_LABEL = "All periods"
CONSENSUS_LABEL = "Outbreak periods"

# Plot appearance
ORIG_COLOR = "blue"
CONS_COLOR = "red"
BOX_ALPHA = 0.35
BOX_WIDTH = 0.30
BOX_OFFSET = 0.18
SHOW_FLIERS = False

DOT_ALPHA = 0.8
DOT_SIZE = 28
DOT_JITTER_SD = 0.03
DOT_RANDOM_SEED = 0

HLINE_Y = 1.0
HLINE_STYLE = "--"
HLINE_WIDTH = 1

# Axis limits / rotation
Y_LIM = (0, 2)
XTICK_ROTATION = 0

# Legend
LEGEND_LOC = "upper right"
SHOW_LEGEND = True

# Saving
SAVE_FIG = True
SAVE_DIR = Path(".")
SAVE_FILENAME_TEMPLATE = "patch_combos_super_boxplot_{mode}.png"
SAVE_DPI = 300
SAVE_BBOX_INCHES = "tight"

# Display
SHOW_FIG = True

# Custom x-tick names. Keys are (epi, einn, filter, ti)
# Any combo not listed here falls back to the automatic compact label.
XTICK_LABEL_OVERRIDES = {
    (False, False, False, True): "TID",
    (False, False, True, False): "Filter",
    (False, True, False, False): "EINN",
    ("ngm", False, False, False): "NGM",
    ("sir_percent", False, False, False): "SIR",
    ("ngm", True, False, False): "NGM +\n EINN",
    ("sir_percent", True, False, False): "SIR +\n EINN",
    # add more overrides here if you want
}

# =========================================================
# Helpers
# =========================================================

def is_all_false_combo(df: pd.DataFrame) -> pd.Series:
    return (
        (df["epi"] == False)
        & (df["einn"] == False)
        & (df["filter"] == False)
        & (df["ti"] == False)
    )


def count_false_values(epi, einn, filt, ti):
    return int(epi is False) + int(einn is False) + int(filt is False) + int(ti is False)


def make_compact_combo_label(epi, einn, filt, ti):
    """
    Only show active / non-false settings.
    """
    parts = []

    if epi is not False:
        parts.append(str(epi))

    if einn is True:
        parts.append("einn")

    if filt is True:
        parts.append("filter")

    if ti is True:
        parts.append("ti")

    if len(parts) == 0:
        return "none"

    return "\n".join(parts)


def get_combo_label(epi, einn, filt, ti, label_overrides):
    combo = (epi, einn, filt, ti)
    if combo in label_overrides:
        return label_overrides[combo]
    return make_compact_combo_label(epi, einn, filt, ti)


def combo_sort_key(combo):
    """
    Sort with most false values first.
    Ties are broken deterministically.
    """
    epi, einn, filt, ti = combo

    epi_rank_map = {False: 0}
    for i, val in enumerate(epi_options):
        if val not in epi_rank_map:
            epi_rank_map[val] = i + 1

    return (
        -count_false_values(epi, einn, filt, ti),   # most false first
        epi_rank_map.get(epi, 999),
        int(bool(einn)),
        int(bool(filt)),
        int(bool(ti)),
    )


def keep_combo(combo, show_all_combos: bool):
    """
    If show_all_combos is False, keep only:
      - combos with exactly 3 False values
      - (ngm, True, False, False)
      - (sir_percent, True, False, False)
    """
    epi, einn, filt, ti = combo

    if show_all_combos:
        return True

    if count_false_values(epi, einn, filt, ti) == 3:
        return True

    if combo == ("ngm", True, False, False):
        return True

    if combo == ("sir_percent", True, False, False):
        return True

    return False


def compute_method_combo_vs_false(df_in, metric_cols, group_cols):
    """
    From one results dataframe, compute one value per (setting combination, method):
      average across horizons of
      baseline_norm_avg(setting, horizon, method) /
      baseline_norm_avg(all-False, same horizon, same method)

    Assumes the dataframe contains '<metric>_mean' columns.
    Returns columns:
      [epi, einn, filter, ti, method, vs_false]
    """
    df = df_in.copy()

    metric_mean_cols = [f"{m}_mean" for m in metric_cols]

    missing_cols = [c for c in metric_mean_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing expected metric mean columns: {missing_cols}")

    rows = []

    # 1) Recompute baseline_norm_avg relative to repeat_last within each setting+horizon
    for _, df_group in df.groupby(group_cols, dropna=False):
        if not (df_group["method"] == "repeat_last").any():
            continue

        valid_horizons = (
            df_group.groupby("horizon")[metric_mean_cols]
            .apply(lambda x: x.notna().any().any())
        )
        valid_horizons = valid_horizons[valid_horizons].index

        df_sub = df_group[df_group["horizon"].isin(valid_horizons)].copy()
        if df_sub.empty:
            continue

        baseline = (
            df_sub[df_sub["method"] == "repeat_last"][["horizon"] + metric_mean_cols]
            .drop_duplicates(subset=["horizon"])
            .set_index("horizon")
        )
        if baseline.empty:
            continue

        baseline_renamed = baseline.rename(
            columns={m: f"{m}_baseline" for m in metric_mean_cols}
        )
        df_sub = df_sub.merge(
            baseline_renamed,
            left_on="horizon",
            right_index=True,
            how="left",
        )

        rel_cols = []
        for m in metric_mean_cols:
            rel_col = f"{m}_rel"
            df_sub[rel_col] = df_sub[m] / df_sub[f"{m}_baseline"]
            rel_cols.append(rel_col)

        df_sub[rel_cols] = df_sub[rel_cols].replace([np.inf, -np.inf], np.nan)

        # Average across metrics, skipping missing metrics within a row
        df_sub["baseline_norm_avg"] = df_sub[rel_cols].mean(axis=1, skipna=True)
        df_sub = df_sub.dropna(subset=["baseline_norm_avg"]).copy()

        if df_sub.empty:
            continue

        rows.append(df_sub)

    if not rows:
        raise ValueError("No valid combinations found with repeat_last baseline.")

    df_plot_all = pd.concat(rows, ignore_index=True)

    # 2) Get all-False reference for each method+horizon
    false_mask = is_all_false_combo(df_plot_all)
    df_false = df_plot_all[false_mask].copy()

    if df_false.empty:
        raise ValueError("No all-False setting found in dataframe.")

    false_ref = (
        df_false[["method", "horizon", "baseline_norm_avg"]]
        .drop_duplicates(subset=["method", "horizon"])
        .rename(columns={"baseline_norm_avg": "baseline_norm_avg_false"})
    )

    # 3) Compare each method to itself in the all-False setting for same horizon
    df_compare = df_plot_all.merge(
        false_ref,
        on=["method", "horizon"],
        how="inner",
    )

    df_compare["vs_false"] = (
        df_compare["baseline_norm_avg"] / df_compare["baseline_norm_avg_false"]
    )

    # One point per method per combination: average across horizons
    df_method_combo = (
        df_compare.groupby(group_cols + ["method"], dropna=False)["vs_false"]
        .mean()
        .reset_index()
    )

    return df_method_combo


def build_super_plot_df(results_dict, metric_cols, group_cols, dataset_label):
    """
    Build one long dataframe across all datasets.
    Each row is one dataset-method-combo point.
    """
    rows = []

    for dataset, df in results_dict.items():
        if df is None or len(df) == 0:
            continue

        try:
            df_method_combo = compute_method_combo_vs_false(
                df, metric_cols=metric_cols, group_cols=group_cols
            )
        except ValueError as e:
            print(f"Skipping {dataset} ({dataset_label}): {e}")
            continue

        df_method_combo = df_method_combo.copy()
        df_method_combo["dataset"] = dataset
        rows.append(df_method_combo)

    if not rows:
        return pd.DataFrame(columns=group_cols + ["method", "vs_false", "dataset"])

    return pd.concat(rows, ignore_index=True)


def make_super_boxplot(
    results_dfs,
    results_dfs_consensus,
    metric_cols,
    group_cols,
    show_all_combos: bool,
    xtick_label_overrides: dict,
):
    # -----------------------------------------------------
    # Build one long dataframe per source across datasets
    # -----------------------------------------------------
    df_super_orig = build_super_plot_df(
        results_dict=results_dfs,
        metric_cols=metric_cols,
        group_cols=group_cols,
        dataset_label="all",
    )

    df_super_cons = build_super_plot_df(
        results_dict=results_dfs_consensus,
        metric_cols=metric_cols,
        group_cols=group_cols,
        dataset_label="consensus",
    )

    if df_super_orig.empty and df_super_cons.empty:
        print("No plottable data found.")
        return

    # -----------------------------------------------------
    # Candidate combos
    # -----------------------------------------------------
    combo_order_all = list(product(epi_options, einn_options, filter_options, ti_options))

    if EXCLUDE_ALL_FALSE_COMBO:
        combo_order_all = [
            combo for combo in combo_order_all
            if not (combo[0] is False and combo[1] is False and combo[2] is False and combo[3] is False)
        ]

    combo_order_all = [
        combo for combo in combo_order_all
        if keep_combo(combo, show_all_combos=show_all_combos)
    ]

    # Keep only combos with any values in either source
    combo_order_plot = []
    for epi, einn, filt, ti in combo_order_all:
        mask_orig = (
            (df_super_orig["epi"] == epi)
            & (df_super_orig["einn"] == einn)
            & (df_super_orig["filter"] == filt)
            & (df_super_orig["ti"] == ti)
        )
        vals_orig = df_super_orig.loc[mask_orig, "vs_false"].dropna().to_numpy()

        mask_cons = (
            (df_super_cons["epi"] == epi)
            & (df_super_cons["einn"] == einn)
            & (df_super_cons["filter"] == filt)
            & (df_super_cons["ti"] == ti)
        )
        vals_cons = df_super_cons.loc[mask_cons, "vs_false"].dropna().to_numpy()

        if len(vals_orig) > 0 or len(vals_cons) > 0:
            combo_order_plot.append((epi, einn, filt, ti))

    combo_order_plot = sorted(combo_order_plot, key=combo_sort_key)

    if len(combo_order_plot) == 0:
        print("No plottable combinations after filtering.")
        return

    combo_labels = [
        get_combo_label(epi, einn, filt, ti, xtick_label_overrides)
        for epi, einn, filt, ti in combo_order_plot
    ]
    positions = np.arange(1, len(combo_order_plot) + 1)

    # -----------------------------------------------------
    # Boxplot data
    # -----------------------------------------------------
    orig_box_data = []
    orig_positions = []

    cons_box_data = []
    cons_positions = []

    for i, (epi, einn, filt, ti) in enumerate(combo_order_plot, start=1):
        mask_orig = (
            (df_super_orig["epi"] == epi)
            & (df_super_orig["einn"] == einn)
            & (df_super_orig["filter"] == filt)
            & (df_super_orig["ti"] == ti)
        )
        vals_orig = df_super_orig.loc[mask_orig, "vs_false"].dropna().to_numpy()
        if len(vals_orig) > 0:
            orig_box_data.append(vals_orig)
            orig_positions.append(i - BOX_OFFSET)

        mask_cons = (
            (df_super_cons["epi"] == epi)
            & (df_super_cons["einn"] == einn)
            & (df_super_cons["filter"] == filt)
            & (df_super_cons["ti"] == ti)
        )
        vals_cons = df_super_cons.loc[mask_cons, "vs_false"].dropna().to_numpy()
        if len(vals_cons) > 0:
            cons_box_data.append(vals_cons)
            cons_positions.append(i + BOX_OFFSET)

    # -----------------------------------------------------
    # Plot
    # -----------------------------------------------------
    fig_width = max(FIGSIZE_MIN_WIDTH, FIGSIZE_WIDTH_PER_COMBO * len(combo_order_plot))
    plt.figure(figsize=(fig_width, FIGSIZE_HEIGHT))

    # Original boxplots
    if len(orig_box_data) > 0:
        bp1 = plt.boxplot(
            orig_box_data,
            positions=orig_positions,
            widths=BOX_WIDTH,
            showfliers=SHOW_FLIERS,
            patch_artist=True,
        )

        for patch in bp1["boxes"]:
            patch.set_facecolor(ORIG_COLOR)
            patch.set_alpha(BOX_ALPHA)
        for item in bp1["medians"]:
            item.set_color(ORIG_COLOR)
        for item in bp1["whiskers"] + bp1["caps"]:
            item.set_color(ORIG_COLOR)

    # Consensus boxplots
    if len(cons_box_data) > 0:
        bp2 = plt.boxplot(
            cons_box_data,
            positions=cons_positions,
            widths=BOX_WIDTH,
            showfliers=SHOW_FLIERS,
            patch_artist=True,
        )

        for patch in bp2["boxes"]:
            patch.set_facecolor(CONS_COLOR)
            patch.set_alpha(BOX_ALPHA)
        for item in bp2["medians"]:
            item.set_color(CONS_COLOR)
        for item in bp2["whiskers"] + bp2["caps"]:
            item.set_color(CONS_COLOR)

    # Overlay dots: each point is one dataset-method combo
    rng = np.random.default_rng(DOT_RANDOM_SEED)

    for i, (epi, einn, filt, ti) in enumerate(combo_order_plot, start=1):
        mask_orig = (
            (df_super_orig["epi"] == epi)
            & (df_super_orig["einn"] == einn)
            & (df_super_orig["filter"] == filt)
            & (df_super_orig["ti"] == ti)
        )
        vals_orig = df_super_orig.loc[mask_orig, "vs_false"].dropna().to_numpy()
        if len(vals_orig) > 0:
            x = rng.normal(loc=i - BOX_OFFSET, scale=DOT_JITTER_SD, size=len(vals_orig))
            plt.scatter(x, vals_orig, color=ORIG_COLOR, alpha=DOT_ALPHA, s=DOT_SIZE)

        mask_cons = (
            (df_super_cons["epi"] == epi)
            & (df_super_cons["einn"] == einn)
            & (df_super_cons["filter"] == filt)
            & (df_super_cons["ti"] == ti)
        )
        vals_cons = df_super_cons.loc[mask_cons, "vs_false"].dropna().to_numpy()
        if len(vals_cons) > 0:
            x = rng.normal(loc=i + BOX_OFFSET, scale=DOT_JITTER_SD, size=len(vals_cons))
            plt.scatter(x, vals_cons, color=CONS_COLOR, alpha=DOT_ALPHA, s=DOT_SIZE)

    title = TITLE_ALL_COMBOS if show_all_combos else TITLE_REDUCED_COMBOS
    mode = "all_combos" if show_all_combos else "selected_combos"

    plt.axhline(HLINE_Y, linestyle=HLINE_STYLE, linewidth=HLINE_WIDTH)
    plt.xticks(positions, combo_labels, rotation=XTICK_ROTATION, fontsize=XTICK_FONTSIZE)
    plt.yticks(fontsize=YTICK_FONTSIZE)
    plt.ylabel(Y_LABEL, fontsize=AXIS_LABEL_FONTSIZE)
    plt.xlabel(X_LABEL, fontsize=AXIS_LABEL_FONTSIZE)
    plt.ylim(Y_LIM)
    plt.title(title, fontsize=TITLE_FONTSIZE)

    if SHOW_LEGEND:
        plt.legend(
            handles=[
                Patch(facecolor=ORIG_COLOR, edgecolor=ORIG_COLOR, alpha=BOX_ALPHA, label=ORIGINAL_LABEL),
                Patch(facecolor=CONS_COLOR, edgecolor=CONS_COLOR, alpha=BOX_ALPHA, label=CONSENSUS_LABEL),
            ],
            loc=LEGEND_LOC,
            fontsize=LEGEND_FONTSIZE,
        )

    if TIGHT_LAYOUT:
        plt.tight_layout()

    if SAVE_FIG:
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        save_path = SAVE_DIR / SAVE_FILENAME_TEMPLATE.format(mode=mode)
        plt.savefig(save_path, dpi=SAVE_DPI, bbox_inches=SAVE_BBOX_INCHES)
        print(f"Saved: {save_path}")

    if SHOW_FIG:
        plt.show()
    else:
        plt.close()


# =========================================================
# Make two super plots across all datasets
# =========================================================

make_super_boxplot(
    results_dfs=results_dfs,
    results_dfs_consensus=results_dfs_outbreak,
    metric_cols=metric_cols,
    group_cols=group_cols,
    show_all_combos=False,
    xtick_label_overrides=XTICK_LABEL_OVERRIDES,
)

### INDIVIDUAL EXAMPLES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =========================================================
# Editable inputs
# =========================================================

metric_cols = [
    # Standard point metrics
    "mse",
    "mae",
    "rmse",
    "medae",
    "medse",

    # Filtered point metrics
    "mse_filtered",
    "mae_filtered",
    "rmse_filtered",
]

METRIC_LABELS = {
    # Standard point metrics
    "mse": "MSE",
    "mae": "MAE",
    "rmse": "RMSE",
    "medae": "medAE",
    "medse": "medSE",

    # Filtered point metrics
    "mse_filtered": "MSE filtered",
    "mae_filtered": "MAE filtered",
    "rmse_filtered": "RMSE filtered",
}

group_cols = ["epi", "einn", "filter", "ti"]
CONFIG_COLS = ["epi", "einn", "filter", "ti"]

BASELINE_METHOD_NAME = "repeat_last"

# ---------------------------------------------------------
# Figure / save parameters
# ---------------------------------------------------------
SAVE_DIR = Path(".")
SAVE_DPI = 300
SAVE_BBOX_INCHES = "tight"
SHOW_FIGURES = True
CLOSE_AFTER_SAVE = True

# File names
FILENAME_ALL_FALSE_ALL = "{dataset}_{metric}_all_false_all.png"
FILENAME_ALL_FALSE_CONSENSUS = "{dataset}_{metric}_all_false_consensus.png"
FILENAME_BEST_ALL = "{dataset}_{metric}_best_config_all.png"
FILENAME_BEST_CONSENSUS = "{dataset}_{metric}_best_config_consensus.png"

# ---------------------------------------------------------
# Figure sizes
# ---------------------------------------------------------
FIGSIZE_ALL_FALSE = (10, 6)
FIGSIZE_BEST = (10, 6)

# ---------------------------------------------------------
# Font sizes
# ---------------------------------------------------------
TITLE_FONTSIZE = 24
SUBTITLE_FONTSIZE = 11
AXIS_LABEL_FONTSIZE = 22
XTICK_FONTSIZE = 22
YTICK_FONTSIZE = 16
LEGEND_FONTSIZE = 18

# ---------------------------------------------------------
# Text
# ---------------------------------------------------------
X_LABEL = "Horizon (weeks)"
Y_LABEL = "Error relative to baseline"

TITLE_TEMPLATE_ALL_FALSE = "{dataset} performance\n ({data_label}, {metric_label})"
TITLE_TEMPLATE_BEST = "{dataset} performance\n (best patch, {data_label}, {metric_label})"

DATA_LABEL_ALL = "all periods"
DATA_LABEL_CONSENSUS = "outbreak periods"

# Toggle whether best-config legend includes the configuration details
BEST_PLOT_LEGEND_INCLUDE_CONFIG = False

BEST_LABEL_TEMPLATE = (
    "{method} "
    "(epi={epi}, einn={einn}, filter={filter}, ti={ti})"
)

# ---------------------------------------------------------
# Axes / limits / ticks
# ---------------------------------------------------------
Y_LIM_ALL_FALSE = (0, 3)
Y_LIM_BEST = (0, 3)
XTICK_ROTATION = 0
SHOW_GRID = False
GRID_ALPHA = 0.25

# ---------------------------------------------------------
# Line / marker / CI style
# ---------------------------------------------------------
LINEWIDTH = 2.0
LINESTYLE = "-"
MARKER = "o"
MARKERSIZE = 5

CI_ALPHA = 0.18
CI_LINEWIDTH = 0.0

# Optional fixed method colors. Leave empty to use fallback palette.
METHOD_COLORS = {
    # "method_a": "tab:blue",
    # "method_b": "tab:orange",
}

# Fallback palette with 20 distinct colors
FALLBACK_COLOR_CYCLE = list(plt.get_cmap("tab20").colors)

# Reference line at baseline for all plots
SHOW_REPEAT_LAST_REFERENCE = True
REPEAT_LAST_REF_Y = 1.0
REPEAT_LAST_REF_COLOR = "black"
REPEAT_LAST_REF_LINESTYLE = "--"
REPEAT_LAST_REF_LINEWIDTH = 1.5

# Legend
SHOW_LEGEND = True
LEGEND_LOC = "upper left"
LEGEND_BBOX_TO_ANCHOR = (1.02, 1.0)
LEGEND_FRAMEON = True

# Tight layout
USE_TIGHT_LAYOUT = True


# =========================================================
# Helpers
# =========================================================

def get_metric_label(metric: str) -> str:
    return METRIC_LABELS.get(metric, metric)


def normalize_config_value(x):
    if pd.isna(x):
        return x
    if isinstance(x, str):
        if x == "True":
            return True
        if x == "False":
            return False
    return x


def standardize_config_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in CONFIG_COLS:
        if col in df.columns:
            df[col] = df[col].map(normalize_config_value)
    return df


def is_all_false_mask(df: pd.DataFrame) -> pd.Series:
    return (
        (df["epi"] == False)
        & (df["einn"] == False)
        & (df["filter"] == False)
        & (df["ti"] == False)
    )


def safe_divide(num, den):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)

    out = np.full(np.broadcast(num, den).shape, np.nan, dtype=float)

    valid = np.isfinite(num) & np.isfinite(den) & (den != 0)
    out[valid] = num[valid] / den[valid]

    return out


def get_method_color(method, fallback_idx=None):
    if method in METHOD_COLORS:
        return METHOD_COLORS[method]

    if fallback_idx is not None and len(FALLBACK_COLOR_CYCLE) > 0:
        return FALLBACK_COLOR_CYCLE[fallback_idx % len(FALLBACK_COLOR_CYCLE)]

    return None


def format_best_label(cfg_row, include_config=True):
    if not include_config:
        return str(cfg_row["method"])

    return BEST_LABEL_TEMPLATE.format(
        method=cfg_row["method"],
        epi=cfg_row["epi"],
        einn=cfg_row["einn"],
        filter=cfg_row["filter"],
        ti=cfg_row["ti"],
    )


def add_reference_line(ax, horizons):
    if SHOW_REPEAT_LAST_REFERENCE and len(horizons) > 0:
        ax.plot(
            horizons,
            np.full(len(horizons), REPEAT_LAST_REF_Y, dtype=float),
            linestyle=REPEAT_LAST_REF_LINESTYLE,
            linewidth=REPEAT_LAST_REF_LINEWIDTH,
            color=REPEAT_LAST_REF_COLOR,
            label="_nolegend_",
        )


# =========================================================
# Core computations
# =========================================================

def compute_baseline_norm_rows_with_ci_for_metric(
    df_in: pd.DataFrame,
    metric_col: str,
    group_cols: list[str],
) -> pd.DataFrame:
    """
    Recompute baseline-normalized metric relative to repeat_last
    within each setting + horizon.

    Input dataframe is expected to have:
      <metric_col>_mean
      <metric_col>_ci_low
      <metric_col>_ci_high

    Returns row-level dataframe with:
      baseline_norm_mean
      baseline_norm_ci_low
      baseline_norm_ci_high

    CI propagation here is approximate, using the stored metric CIs.
    """
    df = df_in.copy()

    metric_mean_col = f"{metric_col}_mean"
    metric_low_col = f"{metric_col}_ci_low"
    metric_high_col = f"{metric_col}_ci_high"

    needed = [metric_mean_col, metric_low_col, metric_high_col]
    missing = [c for c in needed if c not in df.columns]

    if missing:
        raise ValueError(
            f"For metric '{metric_col}', missing expected columns: {missing}"
        )

    rows = []

    for _, df_group in df.groupby(group_cols, dropna=False):
        if not (df_group["method"] == BASELINE_METHOD_NAME).any():
            continue

        valid_horizons = (
            df_group.groupby("horizon")[metric_mean_col]
            .apply(lambda x: x.notna().any())
        )
        valid_horizons = valid_horizons[valid_horizons].index

        df_sub = df_group[df_group["horizon"].isin(valid_horizons)].copy()

        if df_sub.empty:
            continue

        baseline_cols = [
            "horizon",
            metric_mean_col,
            metric_low_col,
            metric_high_col,
        ]

        baseline = (
            df_sub[df_sub["method"] == BASELINE_METHOD_NAME][baseline_cols]
            .drop_duplicates(subset=["horizon"])
            .set_index("horizon")
        )

        if baseline.empty:
            continue

        baseline = baseline.rename(
            columns={
                metric_mean_col: "baseline_mean",
                metric_low_col: "baseline_ci_low",
                metric_high_col: "baseline_ci_high",
            }
        )

        df_sub = df_sub.merge(
            baseline,
            left_on="horizon",
            right_index=True,
            how="left",
        )

        # Point estimate
        df_sub["baseline_norm_mean"] = safe_divide(
            df_sub[metric_mean_col].to_numpy(),
            df_sub["baseline_mean"].to_numpy(),
        )

        # Approximate conservative CI propagation for positive-valued metrics:
        # lower ~ low / baseline_high
        # upper ~ high / baseline_low
        df_sub["baseline_norm_ci_low"] = safe_divide(
            df_sub[metric_low_col].to_numpy(),
            df_sub["baseline_ci_high"].to_numpy(),
        )

        df_sub["baseline_norm_ci_high"] = safe_divide(
            df_sub[metric_high_col].to_numpy(),
            df_sub["baseline_ci_low"].to_numpy(),
        )

        df_sub[
            [
                "baseline_norm_mean",
                "baseline_norm_ci_low",
                "baseline_norm_ci_high",
            ]
        ] = df_sub[
            [
                "baseline_norm_mean",
                "baseline_norm_ci_low",
                "baseline_norm_ci_high",
            ]
        ].replace([np.inf, -np.inf], np.nan)

        df_sub = df_sub.dropna(subset=["baseline_norm_mean"]).copy()

        if not df_sub.empty:
            rows.append(df_sub)

    if not rows:
        raise ValueError(
            f"No valid combinations found with repeat_last baseline for metric '{metric_col}'."
        )

    return pd.concat(rows, ignore_index=True)


def compute_method_combo_vs_false_with_ci(df_plot_all, group_cols):
    """
    For each method and setting combination, compute average across horizons of:
      baseline_norm(setting, horizon, method) /
      baseline_norm(all-False, same horizon, same method)

    Returns:
      [epi, einn, filter, ti, method, vs_false_mean, vs_false_ci_low, vs_false_ci_high]
    """
    false_mask = is_all_false_mask(df_plot_all)

    df_false = df_plot_all[false_mask].copy()

    if df_false.empty:
        raise ValueError("No all-False setting found in dataframe.")

    false_ref = (
        df_false[
            [
                "method",
                "horizon",
                "baseline_norm_mean",
                "baseline_norm_ci_low",
                "baseline_norm_ci_high",
            ]
        ]
        .drop_duplicates(subset=["method", "horizon"])
        .rename(
            columns={
                "baseline_norm_mean": "baseline_norm_false_mean",
                "baseline_norm_ci_low": "baseline_norm_false_ci_low",
                "baseline_norm_ci_high": "baseline_norm_false_ci_high",
            }
        )
    )

    df_compare = df_plot_all.merge(
        false_ref,
        on=["method", "horizon"],
        how="inner",
    )

    df_compare["vs_false_mean"] = safe_divide(
        df_compare["baseline_norm_mean"].to_numpy(),
        df_compare["baseline_norm_false_mean"].to_numpy(),
    )

    df_compare["vs_false_ci_low"] = safe_divide(
        df_compare["baseline_norm_ci_low"].to_numpy(),
        df_compare["baseline_norm_false_ci_high"].to_numpy(),
    )

    df_compare["vs_false_ci_high"] = safe_divide(
        df_compare["baseline_norm_ci_high"].to_numpy(),
        df_compare["baseline_norm_false_ci_low"].to_numpy(),
    )

    df_method_combo = (
        df_compare
        .groupby(group_cols + ["method"], dropna=False)[
            ["vs_false_mean", "vs_false_ci_low", "vs_false_ci_high"]
        ]
        .mean()
        .reset_index()
    )

    return df_method_combo


def select_best_config_per_method(df_plot_all, df_method_combo, group_cols):
    """
    Select one config per method using the LOWEST vs_false_mean.
    repeat_last is excluded.
    """
    df_mc = df_method_combo[df_method_combo["method"] != BASELINE_METHOD_NAME].copy()

    if df_mc.empty:
        raise ValueError("No non-repeat_last methods available.")

    best_idx = df_mc.groupby("method")["vs_false_mean"].idxmin()
    best_configs = df_mc.loc[best_idx].reset_index(drop=True)

    best_rows = df_plot_all.merge(
        best_configs[["method"] + group_cols + ["vs_false_mean"]],
        on=["method"] + group_cols,
        how="inner",
    )

    return best_configs, best_rows


def get_all_false_rows(df_plot_all):
    df_false = df_plot_all[is_all_false_mask(df_plot_all)].copy()

    if df_false.empty:
        raise ValueError("No all-False rows found.")

    return df_false


# =========================================================
# Plotting
# =========================================================

def apply_common_axis_style(ax, horizons, ylim, title):
    ax.set_xlabel(X_LABEL, fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel(Y_LABEL, fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE)

    ax.set_xticks(horizons)
    ax.tick_params(axis="x", labelsize=XTICK_FONTSIZE, rotation=XTICK_ROTATION)
    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)

    ax.set_ylim(ylim)

    if SHOW_GRID:
        ax.grid(True, alpha=GRID_ALPHA)


def plot_all_false_horizons(
    df_false_rows,
    dataset,
    data_label,
    metric_label,
    save_path,
    ylim,
):
    fig, ax = plt.subplots(figsize=FIGSIZE_ALL_FALSE)

    horizons = sorted(df_false_rows["horizon"].dropna().unique())

    df_false_rows = df_false_rows[
        df_false_rows["method"] != BASELINE_METHOD_NAME
    ].copy()

    add_reference_line(ax, horizons)

    for idx, (method, g) in enumerate(df_false_rows.groupby("method")):
        g = g.sort_values("horizon")

        color = get_method_color(method, fallback_idx=idx)

        line, = ax.plot(
            g["horizon"],
            g["baseline_norm_mean"],
            marker=MARKER,
            markersize=MARKERSIZE,
            linestyle=LINESTYLE,
            linewidth=LINEWIDTH,
            color=color,
            label=method,
        )

        line_color = line.get_color()

        ax.fill_between(
            g["horizon"],
            g["baseline_norm_ci_low"],
            g["baseline_norm_ci_high"],
            color=line_color,
            alpha=CI_ALPHA,
            linewidth=CI_LINEWIDTH,
        )

    apply_common_axis_style(
        ax=ax,
        horizons=horizons,
        ylim=ylim,
        title=TITLE_TEMPLATE_ALL_FALSE.format(
            dataset=dataset,
            data_label=data_label,
            metric_label=metric_label,
        ),
    )

    if SHOW_LEGEND:
        ax.legend(
            loc=LEGEND_LOC,
            bbox_to_anchor=LEGEND_BBOX_TO_ANCHOR,
            fontsize=LEGEND_FONTSIZE,
            frameon=LEGEND_FRAMEON,
        )

    if USE_TIGHT_LAYOUT:
        plt.tight_layout()

    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=SAVE_DPI, bbox_inches=SAVE_BBOX_INCHES)
    print(f"Saved: {save_path}")

    if SHOW_FIGURES:
        plt.show()

    if CLOSE_AFTER_SAVE:
        plt.close(fig)


def plot_best_config_horizons_with_ci(
    df_best_rows,
    best_configs,
    dataset,
    data_label,
    metric_label,
    save_path,
    ylim,
):
    fig, ax = plt.subplots(figsize=FIGSIZE_BEST)

    horizons = sorted(df_best_rows["horizon"].dropna().unique())

    add_reference_line(ax, horizons)

    for idx, (method, g) in enumerate(df_best_rows.groupby("method")):
        g = g.sort_values("horizon")

        cfg = best_configs[best_configs["method"] == method].iloc[0]

        label = format_best_label(
            cfg,
            include_config=BEST_PLOT_LEGEND_INCLUDE_CONFIG,
        )

        color = get_method_color(method, fallback_idx=idx)

        line, = ax.plot(
            g["horizon"],
            g["baseline_norm_mean"],
            marker=MARKER,
            markersize=MARKERSIZE,
            linestyle=LINESTYLE,
            linewidth=LINEWIDTH,
            color=color,
            label=label,
        )

        line_color = line.get_color()

        ax.fill_between(
            g["horizon"],
            g["baseline_norm_ci_low"],
            g["baseline_norm_ci_high"],
            color=line_color,
            alpha=CI_ALPHA,
            linewidth=CI_LINEWIDTH,
        )

    apply_common_axis_style(
        ax=ax,
        horizons=horizons,
        ylim=ylim,
        title=TITLE_TEMPLATE_BEST.format(
            dataset=dataset,
            data_label=data_label,
            metric_label=metric_label,
        ),
    )

    if SHOW_LEGEND:
        ax.legend(
            loc=LEGEND_LOC,
            bbox_to_anchor=LEGEND_BBOX_TO_ANCHOR,
            fontsize=LEGEND_FONTSIZE,
            frameon=LEGEND_FRAMEON,
        )

    if USE_TIGHT_LAYOUT:
        plt.tight_layout()

    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=SAVE_DPI, bbox_inches=SAVE_BBOX_INCHES)
    print(f"Saved: {save_path}")

    if SHOW_FIGURES:
        plt.show()

    if CLOSE_AFTER_SAVE:
        plt.close(fig)


# =========================================================
# Per-dataset / per-metric driver
# =========================================================

def make_four_plots_for_dataset_and_metric(
    dataset,
    results_df_dataset,
    results_df_consensus_dataset,
    metric_col,
):
    metric_label = get_metric_label(metric_col)

    # ----------------------------
    # All-data summary rows
    # ----------------------------
    df_all = standardize_config_cols(results_df_dataset)

    df_plot_all_orig = compute_baseline_norm_rows_with_ci_for_metric(
        df_in=df_all,
        metric_col=metric_col,
        group_cols=group_cols,
    )

    df_method_combo_orig = compute_method_combo_vs_false_with_ci(
        df_plot_all_orig,
        group_cols=group_cols,
    )

    best_configs_orig, df_best_rows_orig = select_best_config_per_method(
        df_plot_all_orig,
        df_method_combo_orig,
        group_cols=group_cols,
    )

    df_false_orig = get_all_false_rows(df_plot_all_orig)

    # ----------------------------
    # Consensus / outbreak summary rows
    # ----------------------------
    df_cons = standardize_config_cols(results_df_consensus_dataset)

    df_plot_all_cons = compute_baseline_norm_rows_with_ci_for_metric(
        df_in=df_cons,
        metric_col=metric_col,
        group_cols=group_cols,
    )

    df_method_combo_cons = compute_method_combo_vs_false_with_ci(
        df_plot_all_cons,
        group_cols=group_cols,
    )

    best_configs_cons, df_best_rows_cons = select_best_config_per_method(
        df_plot_all_cons,
        df_method_combo_cons,
        group_cols=group_cols,
    )

    df_false_cons = get_all_false_rows(df_plot_all_cons)

    # ----------------------------
    # 1) all data, all-False config
    # ----------------------------
    plot_all_false_horizons(
        df_false_rows=df_false_orig,
        dataset=dataset,
        data_label=DATA_LABEL_ALL,
        metric_label=metric_label,
        save_path=SAVE_DIR / FILENAME_ALL_FALSE_ALL.format(
            dataset=dataset,
            metric=metric_col,
        ),
        ylim=Y_LIM_ALL_FALSE,
    )

    # ----------------------------
    # 2) consensus/outbreak, all-False config
    # ----------------------------
    plot_all_false_horizons(
        df_false_rows=df_false_cons,
        dataset=dataset,
        data_label=DATA_LABEL_CONSENSUS,
        metric_label=metric_label,
        save_path=SAVE_DIR / FILENAME_ALL_FALSE_CONSENSUS.format(
            dataset=dataset,
            metric=metric_col,
        ),
        ylim=Y_LIM_ALL_FALSE,
    )

    # ----------------------------
    # 3) all data, best config per method
    # ----------------------------
    plot_best_config_horizons_with_ci(
        df_best_rows=df_best_rows_orig,
        best_configs=best_configs_orig,
        dataset=dataset,
        data_label=DATA_LABEL_ALL,
        metric_label=metric_label,
        save_path=SAVE_DIR / FILENAME_BEST_ALL.format(
            dataset=dataset,
            metric=metric_col,
        ),
        ylim=Y_LIM_BEST,
    )

    # ----------------------------
    # 4) consensus/outbreak, best config per method
    # ----------------------------
    plot_best_config_horizons_with_ci(
        df_best_rows=df_best_rows_cons,
        best_configs=best_configs_cons,
        dataset=dataset,
        data_label=DATA_LABEL_CONSENSUS,
        metric_label=metric_label,
        save_path=SAVE_DIR / FILENAME_BEST_CONSENSUS.format(
            dataset=dataset,
            metric=metric_col,
        ),
        ylim=Y_LIM_BEST,
    )


# =========================================================
# Run over all datasets and all metrics
# =========================================================

results_dfs_consensus = results_dfs_outbreak
common_datasets = sorted(set(results_dfs) & set(results_dfs_consensus))

for dataset in common_datasets:
    df_all = results_dfs[dataset]
    df_cons = results_dfs_consensus[dataset]

    if df_all is None or len(df_all) == 0:
        print(f"Skipping {dataset}: results_dfs[{dataset}] is empty")
        continue

    if df_cons is None or len(df_cons) == 0:
        print(f"Skipping {dataset}: results_dfs_consensus[{dataset}] is empty")
        continue

    for metric_col in metric_cols:
        try:
            make_four_plots_for_dataset_and_metric(
                dataset=dataset,
                results_df_dataset=df_all,
                results_df_consensus_dataset=df_cons,
                metric_col=metric_col,
            )
        except Exception as e:
            print(f"Failed for dataset={dataset}, metric={metric_col}: {e}")

### WIN RATES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =========================================================
# Editable parameters
# =========================================================

# Data / columns
WIN_RATE_MEAN_COL = "win_rate_mean"
WIN_RATE_CI_LOW_COL = "win_rate_ci_low"
WIN_RATE_CI_HIGH_COL = "win_rate_ci_high"

METHOD_BASELINE_NAME = "repeat_last"   # excluded from plots
CONFIG_COLS = ["epi", "einn", "filter", "ti"]

# How to aggregate across horizons using the existing CIs
# Options:
#   "inverse_variance"  -> pooled estimate using CI-derived SEs
#   "simple_mean"       -> mean of means, mean of CI bounds
HORIZON_AGG_METHOD = "inverse_variance"

# ---------------------------------------------------------
# Figure / layout
# ---------------------------------------------------------
FIGSIZE = (14, 8)
BAR_GROUP_WIDTH = 0.8
BAR_ALPHA = 0.9

SHOW_GRID = False
GRID_ALPHA = 0.25
GRID_AXIS = "y"

YMIN = 0.0
YMAX = 0.6

# 50% reference line
REFERENCE_Y = 0.5
REFERENCE_LINESTYLE = "--"
REFERENCE_LINEWIDTH = 1.5
REFERENCE_LINECOLOR = "black"

# ---------------------------------------------------------
# Error bars / CI style
# ---------------------------------------------------------
SHOW_ERROR_BARS = True
ERROR_BAR_COLOR = "black"
ERROR_BAR_LINEWIDTH = 1.2
ERROR_BAR_CAPSIZE = 4
ERROR_BAR_CAPTHICK = 1.2

# ---------------------------------------------------------
# Fonts
# ---------------------------------------------------------
TITLE_FONTSIZE = 24
AXIS_LABEL_FONTSIZE = 24
XTICK_FONTSIZE = 22
YTICK_FONTSIZE = 18
LEGEND_FONTSIZE = 18

# ---------------------------------------------------------
# Text
# ---------------------------------------------------------
TITLE_ALL = "Win rate over baseline (all periods)"
TITLE_CONSENSUS = "Win rate over baseline (outbreak periods)"
X_LABEL = ""
Y_LABEL = "Win rate"
REFERENCE_LABEL = "50%"

# ---------------------------------------------------------
# X tick / labels
# ---------------------------------------------------------
XTICK_ROTATION = 45
XTICK_HA = "right"

# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------
SHOW_LEGEND = True
LEGEND_LOC = "upper left"
LEGEND_BBOX_TO_ANCHOR = (1.02, 1.0)
LEGEND_FRAMEON = True
LEGEND_TITLE = ""

# ---------------------------------------------------------
# Method colors
# Leave empty to use fallback palette.
# ---------------------------------------------------------
METHOD_COLORS = {}

# Same 20-color fallback palette as the other plotting code
FALLBACK_COLOR_CYCLE = list(plt.get_cmap("tab20").colors)

# ---------------------------------------------------------
# Save / show
# ---------------------------------------------------------
SAVE_FIGURES = True
SAVE_DIR = Path(".")
SAVE_DPI = 300
SAVE_BBOX_INCHES = "tight"
FILENAME_ALL = "win_rate_all_false_all_data.png"
FILENAME_CONSENSUS = "win_rate_all_false_consensus.png"

SHOW_FIGURES = True
CLOSE_FIGURES = True


# =========================================================
# Helpers
# =========================================================

def normalize_config_value(x):
    """
    Make CSV-loaded config values robust for comparisons.
    """
    if pd.isna(x):
        return x
    if isinstance(x, str):
        if x == "True":
            return True
        if x == "False":
            return False
    return x


def standardize_config_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in CONFIG_COLS:
        if col in df.columns:
            df[col] = df[col].map(normalize_config_value)
    return df


def is_all_false_mask(df: pd.DataFrame) -> pd.Series:
    return (
        (df["epi"] == False)
        & (df["einn"] == False)
        & (df["filter"] == False)
        & (df["ti"] == False)
    )


def clip01(x):
    return np.clip(x, 0.0, 1.0) if np.isfinite(x) else x


def ci_to_se(ci_low, ci_high):
    """
    Recover SE from a symmetric 95% CI width.
    """
    if not (np.isfinite(ci_low) and np.isfinite(ci_high)):
        return np.nan
    width = ci_high - ci_low
    if width <= 0:
        return np.nan
    return width / (2 * 1.96)


def aggregate_estimates_with_ci(
    means: np.ndarray,
    ci_lows: np.ndarray,
    ci_highs: np.ndarray,
    method: str = "inverse_variance",
):
    """
    Aggregate horizon-level estimates with existing CIs.

    Returns:
      pooled_mean, pooled_ci_low, pooled_ci_high
    """
    means = np.asarray(means, dtype=float)
    ci_lows = np.asarray(ci_lows, dtype=float)
    ci_highs = np.asarray(ci_highs, dtype=float)

    valid = np.isfinite(means) & np.isfinite(ci_lows) & np.isfinite(ci_highs)
    means = means[valid]
    ci_lows = ci_lows[valid]
    ci_highs = ci_highs[valid]

    if len(means) == 0:
        return np.nan, np.nan, np.nan

    if len(means) == 1:
        return clip01(means[0]), clip01(ci_lows[0]), clip01(ci_highs[0])

    if method == "simple_mean":
        pooled_mean = means.mean()
        pooled_ci_low = ci_lows.mean()
        pooled_ci_high = ci_highs.mean()
        return clip01(pooled_mean), clip01(pooled_ci_low), clip01(pooled_ci_high)

    if method == "inverse_variance":
        ses = np.array([ci_to_se(lo, hi) for lo, hi in zip(ci_lows, ci_highs)], dtype=float)
        valid_se = np.isfinite(ses) & (ses > 0)

        if not valid_se.any():
            pooled_mean = means.mean()
            pooled_ci_low = ci_lows.mean()
            pooled_ci_high = ci_highs.mean()
            return clip01(pooled_mean), clip01(pooled_ci_low), clip01(pooled_ci_high)

        means = means[valid_se]
        ses = ses[valid_se]

        weights = 1.0 / (ses ** 2)
        pooled_mean = np.sum(weights * means) / np.sum(weights)
        pooled_se = np.sqrt(1.0 / np.sum(weights))
        pooled_ci_low = pooled_mean - 1.96 * pooled_se
        pooled_ci_high = pooled_mean + 1.96 * pooled_se

        return clip01(pooled_mean), clip01(pooled_ci_low), clip01(pooled_ci_high)

    raise ValueError(f"Unsupported HORIZON_AGG_METHOD={method}")


def get_method_color(method, idx):
    if method in METHOD_COLORS:
        return METHOD_COLORS[method]
    if len(FALLBACK_COLOR_CYCLE) > 0:
        return FALLBACK_COLOR_CYCLE[idx % len(FALLBACK_COLOR_CYCLE)]
    return None


def build_plot_df(results_dict: dict, baseline_method_name: str) -> pd.DataFrame:
    """
    Returns one row per dataset x method for the all-False config,
    aggregated across horizons using the stored CIs.

    Output columns:
      dataset, method, win_rate, win_rate_ci_low, win_rate_ci_high
    """
    rows = []

    for dataset, df in results_dict.items():
        if df is None or len(df) == 0:
            continue

        df = standardize_config_cols(df)

        needed_cols = [
            "dataset",
            "method",
            "horizon",
            WIN_RATE_MEAN_COL,
            WIN_RATE_CI_LOW_COL,
            WIN_RATE_CI_HIGH_COL,
            "epi",
            "einn",
            "filter",
            "ti",
        ]
        missing = [c for c in needed_cols if c not in df.columns]
        if missing:
            print(f"Skipping {dataset}: missing columns {missing}")
            continue

        df_sub = df[is_all_false_mask(df)].copy()
        if df_sub.empty:
            print(f"Skipping {dataset}: no all-False rows found")
            continue

        df_sub = df_sub[df_sub["method"] != baseline_method_name].copy()
        if df_sub.empty:
            print(f"Skipping {dataset}: no non-baseline methods after filtering")
            continue

        for method, g in df_sub.groupby("method", dropna=False):
            pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
                means=g[WIN_RATE_MEAN_COL].to_numpy(dtype=float),
                ci_lows=g[WIN_RATE_CI_LOW_COL].to_numpy(dtype=float),
                ci_highs=g[WIN_RATE_CI_HIGH_COL].to_numpy(dtype=float),
                method=HORIZON_AGG_METHOD,
            )

            rows.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "win_rate": pooled_mean,
                    "win_rate_ci_low": pooled_ci_low,
                    "win_rate_ci_high": pooled_ci_high,
                }
            )

    if not rows:
        return pd.DataFrame(
            columns=[
                "dataset",
                "method",
                "win_rate",
                "win_rate_ci_low",
                "win_rate_ci_high",
            ]
        )

    return pd.DataFrame(rows)


def plot_win_rate_barplot(
    plot_df: pd.DataFrame,
    title: str,
    save_filename: str | None = None,
):
    if plot_df.empty:
        print(f"No data to plot for: {title}")
        return

    datasets_order = list(plot_df["dataset"].drop_duplicates())
    methods_order = list(plot_df["method"].drop_duplicates())

    x = np.arange(len(datasets_order))
    n_methods = len(methods_order)

    if n_methods == 0:
        print(f"No methods to plot for: {title}")
        return

    bar_width = BAR_GROUP_WIDTH / n_methods

    fig, ax = plt.subplots(figsize=FIGSIZE)

    for i, method in enumerate(methods_order):
        vals = []
        yerr_low = []
        yerr_high = []

        for dataset in datasets_order:
            sub = plot_df[
                (plot_df["dataset"] == dataset)
                & (plot_df["method"] == method)
            ]

            if sub.empty:
                vals.append(np.nan)
                yerr_low.append(np.nan)
                yerr_high.append(np.nan)
            else:
                row = sub.iloc[0]
                val = row["win_rate"]
                ci_low = row["win_rate_ci_low"]
                ci_high = row["win_rate_ci_high"]

                vals.append(val)
                yerr_low.append(val - ci_low if pd.notna(val) and pd.notna(ci_low) else np.nan)
                yerr_high.append(ci_high - val if pd.notna(val) and pd.notna(ci_high) else np.nan)

        vals = np.asarray(vals, dtype=float)
        yerr = np.vstack([
            np.asarray(yerr_low, dtype=float),
            np.asarray(yerr_high, dtype=float),
        ])

        offsets = x - (BAR_GROUP_WIDTH / 2) + (i + 0.5) * bar_width
        color = get_method_color(method, i)

        ax.bar(
            offsets,
            vals,
            width=bar_width,
            label=method,
            alpha=BAR_ALPHA,
            color=color,
            yerr=yerr if SHOW_ERROR_BARS else None,
            ecolor=ERROR_BAR_COLOR,
            error_kw={
                "elinewidth": ERROR_BAR_LINEWIDTH,
                "capsize": ERROR_BAR_CAPSIZE,
                "capthick": ERROR_BAR_CAPTHICK,
            } if SHOW_ERROR_BARS else None,
        )

    ax.axhline(
        REFERENCE_Y,
        linestyle=REFERENCE_LINESTYLE,
        linewidth=REFERENCE_LINEWIDTH,
        color=REFERENCE_LINECOLOR,
        label=REFERENCE_LABEL if not SHOW_LEGEND else None,
    )

    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    ax.set_xlabel(X_LABEL, fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel(Y_LABEL, fontsize=AXIS_LABEL_FONTSIZE)

    ax.set_xticks(x)
    ax.set_xticklabels(
        datasets_order,
        rotation=XTICK_ROTATION,
        ha=XTICK_HA,
        fontsize=XTICK_FONTSIZE,
    )
    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)
    ax.set_ylim(YMIN, YMAX)

    if SHOW_GRID:
        ax.grid(True, axis=GRID_AXIS, alpha=GRID_ALPHA)

    if SHOW_LEGEND:
        ax.legend(
            loc=LEGEND_LOC,
            bbox_to_anchor=LEGEND_BBOX_TO_ANCHOR,
            fontsize=LEGEND_FONTSIZE,
            frameon=LEGEND_FRAMEON,
            title=LEGEND_TITLE,
        )

    plt.tight_layout()

    if SAVE_FIGURES and save_filename is not None:
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        save_path = SAVE_DIR / save_filename
        plt.savefig(save_path, dpi=SAVE_DPI, bbox_inches=SAVE_BBOX_INCHES)
        print(f"Saved: {save_path}")

    if SHOW_FIGURES:
        plt.show()

    if CLOSE_FIGURES:
        plt.close(fig)


# =========================================================
# Build and plot
# =========================================================

plot_df_all = build_plot_df(
    results_dict=results_dfs,
    baseline_method_name=METHOD_BASELINE_NAME,
)

plot_df_consensus = build_plot_df(
    results_dict=results_dfs_consensus,
    baseline_method_name=METHOD_BASELINE_NAME,
)

plot_win_rate_barplot(
    plot_df=plot_df_all,
    title=TITLE_ALL,
    save_filename=FILENAME_ALL,
)

plot_win_rate_barplot(
    plot_df=plot_df_consensus,
    title=TITLE_CONSENSUS,
    save_filename=FILENAME_CONSENSUS,
)

### PERFORMANCE

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =========================================================
# Editable parameters
# =========================================================

metric_cols = [
    # Standard point metrics
    "mse",
    "mae",
    "rmse",
    "medae",
    "medse",

    # Filtered point metrics
    "mse_filtered",
    "mae_filtered",
    "rmse_filtered",
]

METRIC_LABELS = {
    # Standard point metrics
    "mse": "MSE",
    "mae": "MAE",
    "rmse": "RMSE",
    "medae": "medAE",
    "medse": "medSE",

    # Filtered point metrics
    "mse_filtered": "MSE filtered",
    "mae_filtered": "MAE filtered",
    "rmse_filtered": "RMSE filtered",
}

group_cols = ["epi", "einn", "filter", "ti"]
CONFIG_COLS = ["epi", "einn", "filter", "ti"]

BASELINE_METHOD_NAME = "repeat_last"

# How to aggregate across horizons using existing CIs
# Options:
#   "inverse_variance"
#   "simple_mean"
HORIZON_AGG_METHOD = "inverse_variance"

# Whether to include repeat_last bars
INCLUDE_REPEAT_LAST = False

# ---------------------------------------------------------
# Figure / layout
# ---------------------------------------------------------
FIGSIZE = (14, 8)
BAR_GROUP_WIDTH = 0.8
BAR_ALPHA = 0.9

SHOW_GRID = False
GRID_ALPHA = 0.25
GRID_AXIS = "y"

YMIN = 0.0
YMAX = 2.0

# Reference line at naive baseline
REFERENCE_Y = 1.0
REFERENCE_LINESTYLE = "--"
REFERENCE_LINEWIDTH = 1.5
REFERENCE_LINECOLOR = "black"

# ---------------------------------------------------------
# Error bars / CI style
# ---------------------------------------------------------
SHOW_ERROR_BARS = True
ERROR_BAR_COLOR = "black"
ERROR_BAR_LINEWIDTH = 1.2
ERROR_BAR_CAPSIZE = 4
ERROR_BAR_CAPTHICK = 1.2

# ---------------------------------------------------------
# Fonts
# ---------------------------------------------------------
TITLE_FONTSIZE = 24
AXIS_LABEL_FONTSIZE = 24
XTICK_FONTSIZE = 22
YTICK_FONTSIZE = 18
LEGEND_FONTSIZE = 18

# ---------------------------------------------------------
# Text
# ---------------------------------------------------------
TITLE_ALL_TEMPLATE = "Overall performance (all periods, {metric_label})"
TITLE_CONSENSUS_TEMPLATE = "Overall performance (outbreak periods, {metric_label})"

X_LABEL = ""
Y_LABEL = "Error relative to baseline"
REFERENCE_LABEL = "repeat_last = 1"

# ---------------------------------------------------------
# X tick / labels
# ---------------------------------------------------------
XTICK_ROTATION = 45
XTICK_HA = "right"

# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------
SHOW_LEGEND = True
LEGEND_LOC = "upper left"
LEGEND_BBOX_TO_ANCHOR = (1.02, 1.0)
LEGEND_FRAMEON = True
LEGEND_TITLE = ""

# ---------------------------------------------------------
# Method colors
# Leave empty to use the shared fallback palette.
# ---------------------------------------------------------
METHOD_COLORS = {
    # "AGCRN": "tab:blue",
    # "DCRNN": "tab:orange",
}

FALLBACK_COLOR_CYCLE = list(plt.get_cmap("tab20").colors)

# ---------------------------------------------------------
# Save / show
# ---------------------------------------------------------
SAVE_FIGURES = True
SAVE_DIR = Path(".")
SAVE_DPI = 300
SAVE_BBOX_INCHES = "tight"

FILENAME_ALL_TEMPLATE = "baseline_norm_all_false_{metric}_all_data.png"
FILENAME_CONSENSUS_TEMPLATE = "baseline_norm_all_false_{metric}_consensus.png"

SHOW_FIGURES = True
CLOSE_FIGURES = True


# =========================================================
# Helpers
# =========================================================

def get_metric_label(metric: str) -> str:
    """
    Display labels for metric names.

    Examples:
      mse -> MSE
      medae -> medAE
      mse_filtered -> MSE filtered
    """
    return METRIC_LABELS.get(metric, metric)


def normalize_config_value(x):
    if pd.isna(x):
        return x
    if isinstance(x, str):
        if x == "True":
            return True
        if x == "False":
            return False
    return x


def standardize_config_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in CONFIG_COLS:
        if col in df.columns:
            df[col] = df[col].map(normalize_config_value)
    return df


def is_all_false_mask(df: pd.DataFrame) -> pd.Series:
    return (
        (df["epi"] == False)
        & (df["einn"] == False)
        & (df["filter"] == False)
        & (df["ti"] == False)
    )


def safe_divide(num, den):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)

    out = np.full(np.broadcast(num, den).shape, np.nan, dtype=float)

    valid = np.isfinite(num) & np.isfinite(den) & (den != 0)
    out[valid] = num[valid] / den[valid]

    return out


def ci_to_se(ci_low, ci_high):
    if not (np.isfinite(ci_low) and np.isfinite(ci_high)):
        return np.nan

    width = ci_high - ci_low
    if width <= 0:
        return np.nan

    return width / (2 * 1.96)


def aggregate_estimates_with_ci(
    means: np.ndarray,
    ci_lows: np.ndarray,
    ci_highs: np.ndarray,
    method: str = "inverse_variance",
):
    """
    Aggregate horizon-level estimates with existing CIs.

    Returns:
      pooled_mean, pooled_ci_low, pooled_ci_high
    """
    means = np.asarray(means, dtype=float)
    ci_lows = np.asarray(ci_lows, dtype=float)
    ci_highs = np.asarray(ci_highs, dtype=float)

    valid = np.isfinite(means) & np.isfinite(ci_lows) & np.isfinite(ci_highs)
    means = means[valid]
    ci_lows = ci_lows[valid]
    ci_highs = ci_highs[valid]

    if len(means) == 0:
        return np.nan, np.nan, np.nan

    if len(means) == 1:
        return means[0], ci_lows[0], ci_highs[0]

    if method == "simple_mean":
        return means.mean(), ci_lows.mean(), ci_highs.mean()

    if method == "inverse_variance":
        ses = np.array(
            [ci_to_se(lo, hi) for lo, hi in zip(ci_lows, ci_highs)],
            dtype=float,
        )

        valid_se = np.isfinite(ses) & (ses > 0)

        if not valid_se.any():
            return means.mean(), ci_lows.mean(), ci_highs.mean()

        means = means[valid_se]
        ses = ses[valid_se]

        weights = 1.0 / (ses ** 2)

        pooled_mean = np.sum(weights * means) / np.sum(weights)
        pooled_se = np.sqrt(1.0 / np.sum(weights))

        pooled_ci_low = pooled_mean - 1.96 * pooled_se
        pooled_ci_high = pooled_mean + 1.96 * pooled_se

        return pooled_mean, pooled_ci_low, pooled_ci_high

    raise ValueError(f"Unsupported HORIZON_AGG_METHOD={method}")


def get_method_color(method, idx):
    if method in METHOD_COLORS:
        return METHOD_COLORS[method]

    if len(FALLBACK_COLOR_CYCLE) > 0:
        return FALLBACK_COLOR_CYCLE[idx % len(FALLBACK_COLOR_CYCLE)]

    return None


def make_yerr(val, ci_low, ci_high):
    """
    Matplotlib expects nonnegative asymmetric error sizes.
    """
    if pd.isna(val) or pd.isna(ci_low) or pd.isna(ci_high):
        return np.nan, np.nan

    low_err = max(val - ci_low, 0.0)
    high_err = max(ci_high - val, 0.0)

    return low_err, high_err


# =========================================================
# Core computation
# =========================================================

def compute_baseline_norm_rows_with_ci_for_metric(
    df_in: pd.DataFrame,
    metric_col: str,
    group_cols: list[str],
) -> pd.DataFrame:
    """
    Recompute baseline-normalized performance relative to repeat_last
    within each setting + horizon for one metric.

    Input dataframe is expected to have:
      <metric_col>_mean
      <metric_col>_ci_low
      <metric_col>_ci_high

    Returns row-level dataframe with:
      baseline_norm_mean
      baseline_norm_ci_low
      baseline_norm_ci_high
    """
    df = df_in.copy()

    metric_mean_col = f"{metric_col}_mean"
    metric_low_col = f"{metric_col}_ci_low"
    metric_high_col = f"{metric_col}_ci_high"

    needed = [metric_mean_col, metric_low_col, metric_high_col]
    missing = [c for c in needed if c not in df.columns]

    if missing:
        raise ValueError(
            f"For metric '{metric_col}', missing expected columns: {missing}"
        )

    rows = []

    for _, df_group in df.groupby(group_cols, dropna=False):
        if not (df_group["method"] == BASELINE_METHOD_NAME).any():
            continue

        valid_horizons = (
            df_group.groupby("horizon")[metric_mean_col]
            .apply(lambda x: x.notna().any())
        )
        valid_horizons = valid_horizons[valid_horizons].index

        df_sub = df_group[df_group["horizon"].isin(valid_horizons)].copy()

        if df_sub.empty:
            continue

        baseline_cols = [
            "horizon",
            metric_mean_col,
            metric_low_col,
            metric_high_col,
        ]

        baseline = (
            df_sub[df_sub["method"] == BASELINE_METHOD_NAME][baseline_cols]
            .drop_duplicates(subset=["horizon"])
            .set_index("horizon")
        )

        if baseline.empty:
            continue

        baseline = baseline.rename(
            columns={
                metric_mean_col: "baseline_mean",
                metric_low_col: "baseline_ci_low",
                metric_high_col: "baseline_ci_high",
            }
        )

        df_sub = df_sub.merge(
            baseline,
            left_on="horizon",
            right_index=True,
            how="left",
        )

        df_sub["baseline_norm_mean"] = safe_divide(
            df_sub[metric_mean_col].to_numpy(),
            df_sub["baseline_mean"].to_numpy(),
        )

        # Conservative CI propagation:
        # lower bound uses model lower / baseline upper
        # upper bound uses model upper / baseline lower
        df_sub["baseline_norm_ci_low"] = safe_divide(
            df_sub[metric_low_col].to_numpy(),
            df_sub["baseline_ci_high"].to_numpy(),
        )

        df_sub["baseline_norm_ci_high"] = safe_divide(
            df_sub[metric_high_col].to_numpy(),
            df_sub["baseline_ci_low"].to_numpy(),
        )

        df_sub[
            [
                "baseline_norm_mean",
                "baseline_norm_ci_low",
                "baseline_norm_ci_high",
            ]
        ] = df_sub[
            [
                "baseline_norm_mean",
                "baseline_norm_ci_low",
                "baseline_norm_ci_high",
            ]
        ].replace([np.inf, -np.inf], np.nan)

        df_sub = df_sub.dropna(subset=["baseline_norm_mean"]).copy()

        if not df_sub.empty:
            rows.append(df_sub)

    if not rows:
        raise ValueError(
            f"No valid combinations found with repeat_last baseline for metric '{metric_col}'."
        )

    return pd.concat(rows, ignore_index=True)


def build_plot_df(
    results_dict: dict,
    metric_col: str,
) -> pd.DataFrame:
    """
    Returns one row per dataset x method for the all-False config,
    aggregated across horizons using the stored/propagated CIs.

    Output columns:
      dataset
      method
      metric
      performance
      performance_ci_low
      performance_ci_high
    """
    rows = []

    for dataset, df in results_dict.items():
        if df is None or len(df) == 0:
            continue

        df = standardize_config_cols(df)

        try:
            df_plot_all = compute_baseline_norm_rows_with_ci_for_metric(
                df_in=df,
                metric_col=metric_col,
                group_cols=group_cols,
            )
        except ValueError as e:
            print(f"Skipping {dataset}, metric={metric_col}: {e}")
            continue

        df_false = df_plot_all[is_all_false_mask(df_plot_all)].copy()

        if df_false.empty:
            print(f"Skipping {dataset}, metric={metric_col}: no all-False rows found")
            continue

        if not INCLUDE_REPEAT_LAST:
            df_false = df_false[df_false["method"] != BASELINE_METHOD_NAME].copy()

        if df_false.empty:
            print(
                f"Skipping {dataset}, metric={metric_col}: "
                "no methods to plot after filtering"
            )
            continue

        for method, g in df_false.groupby("method", dropna=False):
            pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
                means=g["baseline_norm_mean"].to_numpy(dtype=float),
                ci_lows=g["baseline_norm_ci_low"].to_numpy(dtype=float),
                ci_highs=g["baseline_norm_ci_high"].to_numpy(dtype=float),
                method=HORIZON_AGG_METHOD,
            )

            rows.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "metric": metric_col,
                    "performance": pooled_mean,
                    "performance_ci_low": pooled_ci_low,
                    "performance_ci_high": pooled_ci_high,
                }
            )

    return pd.DataFrame(
        rows,
        columns=[
            "dataset",
            "method",
            "metric",
            "performance",
            "performance_ci_low",
            "performance_ci_high",
        ],
    )


# =========================================================
# Plotting
# =========================================================

def plot_performance_barplot(
    plot_df: pd.DataFrame,
    title: str,
    save_filename: str | None = None,
):
    if plot_df.empty:
        print(f"No data to plot for: {title}")
        return

    datasets_order = list(plot_df["dataset"].drop_duplicates())
    methods_order = list(plot_df["method"].drop_duplicates())

    x = np.arange(len(datasets_order))
    n_methods = len(methods_order)

    if n_methods == 0:
        print(f"No methods to plot for: {title}")
        return

    bar_width = BAR_GROUP_WIDTH / n_methods

    fig, ax = plt.subplots(figsize=FIGSIZE)

    for i, method in enumerate(methods_order):
        vals = []
        yerr_low = []
        yerr_high = []

        for dataset in datasets_order:
            sub = plot_df[
                (plot_df["dataset"] == dataset)
                & (plot_df["method"] == method)
            ]

            if sub.empty:
                vals.append(np.nan)
                yerr_low.append(np.nan)
                yerr_high.append(np.nan)
            else:
                row = sub.iloc[0]

                val = row["performance"]
                ci_low = row["performance_ci_low"]
                ci_high = row["performance_ci_high"]

                low_err, high_err = make_yerr(val, ci_low, ci_high)

                vals.append(val)
                yerr_low.append(low_err)
                yerr_high.append(high_err)

        vals = np.asarray(vals, dtype=float)
        yerr = np.vstack(
            [
                np.asarray(yerr_low, dtype=float),
                np.asarray(yerr_high, dtype=float),
            ]
        )

        offsets = x - (BAR_GROUP_WIDTH / 2) + (i + 0.5) * bar_width
        color = get_method_color(method, i)

        ax.bar(
            offsets,
            vals,
            width=bar_width,
            label=method,
            alpha=BAR_ALPHA,
            color=color,
            yerr=yerr if SHOW_ERROR_BARS else None,
            ecolor=ERROR_BAR_COLOR,
            error_kw={
                "elinewidth": ERROR_BAR_LINEWIDTH,
                "capsize": ERROR_BAR_CAPSIZE,
                "capthick": ERROR_BAR_CAPTHICK,
            }
            if SHOW_ERROR_BARS
            else None,
        )

    ax.axhline(
        REFERENCE_Y,
        linestyle=REFERENCE_LINESTYLE,
        linewidth=REFERENCE_LINEWIDTH,
        color=REFERENCE_LINECOLOR,
    )

    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    ax.set_xlabel(X_LABEL, fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel(Y_LABEL, fontsize=AXIS_LABEL_FONTSIZE)

    ax.set_xticks(x)
    ax.set_xticklabels(
        datasets_order,
        rotation=XTICK_ROTATION,
        ha=XTICK_HA,
        fontsize=XTICK_FONTSIZE,
    )

    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)
    ax.set_ylim(YMIN, YMAX)

    if SHOW_GRID:
        ax.grid(True, axis=GRID_AXIS, alpha=GRID_ALPHA)

    if SHOW_LEGEND:
        ax.legend(
            loc=LEGEND_LOC,
            bbox_to_anchor=LEGEND_BBOX_TO_ANCHOR,
            fontsize=LEGEND_FONTSIZE,
            frameon=LEGEND_FRAMEON,
            title=LEGEND_TITLE,
        )

    plt.tight_layout()

    if SAVE_FIGURES and save_filename is not None:
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        save_path = SAVE_DIR / save_filename
        plt.savefig(save_path, dpi=SAVE_DPI, bbox_inches=SAVE_BBOX_INCHES)
        print(f"Saved: {save_path}")

    if SHOW_FIGURES:
        plt.show()

    if CLOSE_FIGURES:
        plt.close(fig)


# =========================================================
# Build and plot for each metric
# =========================================================

plot_dfs_all = {}
plot_dfs_consensus = {}

for metric_col in metric_cols:
    metric_display = get_metric_label(metric_col)

    plot_df_all = build_plot_df(
        results_dict=results_dfs,
        metric_col=metric_col,
    )

    plot_df_consensus = build_plot_df(
        results_dict=results_dfs_consensus,
        metric_col=metric_col,
    )

    plot_dfs_all[metric_col] = plot_df_all
    plot_dfs_consensus[metric_col] = plot_df_consensus

    plot_performance_barplot(
        plot_df=plot_df_all,
        title=TITLE_ALL_TEMPLATE.format(metric_label=metric_display),
        save_filename=FILENAME_ALL_TEMPLATE.format(metric=metric_col),
    )

    plot_performance_barplot(
        plot_df=plot_df_consensus,
        title=TITLE_CONSENSUS_TEMPLATE.format(metric_label=metric_display),
        save_filename=FILENAME_CONSENSUS_TEMPLATE.format(metric=metric_col),
    )

In [ ]:
# =========================================================
# Outbreak degradation statistics
# =========================================================
# Assumes these already exist from the code above:
#   plot_df_all
#   plot_df_consensus
#
# Interpretation:
#   performance_all      = baseline-normalized error over all periods
#   performance_outbreak = baseline-normalized error over outbreak periods
#
# Since lower is better:
#   outbreak_minus_all > 0  => method is worse in outbreaks
#   outbreak_over_all  > 1  => method is worse in outbreaks
# =========================================================

from math import erfc, sqrt

OUTBREAK_LABEL = "outbreak"
ALL_LABEL = "all"

SAVE_COMPARISON_TABLES = True
FILENAME_OUTBREAK_COMPARISON = "outbreak_vs_all_method_dataset_stats.csv"
FILENAME_OUTBREAK_METHOD_SUMMARY = "outbreak_vs_all_method_summary.csv"

# Plot settings for degradation plot
PLOT_OUTBREAK_DEGRADATION = True
FILENAME_OUTBREAK_DEGRADATION_PLOT = "outbreak_degradation_by_method.png"
TITLE_OUTBREAK_DEGRADATION = "Outbreak degradation by method"
Y_LABEL_OUTBREAK_DEGRADATION = "Outbreak RMSE - all-period RMSE\n(relative to repeat_last)"
REFERENCE_DEGRADATION_Y = 0.0


def normal_sf(z):
    """
    Survival function for standard normal: P(Z >= z).
    Avoids needing scipy.
    """
    if not np.isfinite(z):
        return np.nan
    return 0.5 * erfc(z / sqrt(2.0))


def se_from_ci(ci_low, ci_high):
    """
    Convert 95% CI to approximate standard error.
    """
    if not (np.isfinite(ci_low) and np.isfinite(ci_high)):
        return np.nan
    width = ci_high - ci_low
    if width <= 0:
        return np.nan
    return width / (2 * 1.96)


def add_se_column(df, prefix="performance"):
    df = df.copy()
    df[f"{prefix}_se"] = [
        se_from_ci(lo, hi)
        for lo, hi in zip(df[f"{prefix}_ci_low"], df[f"{prefix}_ci_high"])
    ]
    return df


def compare_outbreak_vs_all(plot_df_all, plot_df_outbreak):
    """
    One row per dataset x method.

    Computes:
      outbreak_minus_all
      outbreak_over_all
      approximate CIs
      approximate one-sided and two-sided p-values

    Note:
      The p-values assume independent all/outbreak estimates.
      Because all-period estimates include outbreak periods, this is an approximation.
      The effect sizes are the most important output.
    """
    all_df = plot_df_all.rename(
        columns={
            "performance": "performance_all",
            "performance_ci_low": "performance_all_ci_low",
            "performance_ci_high": "performance_all_ci_high",
        }
    )

    outbreak_df = plot_df_outbreak.rename(
        columns={
            "performance": "performance_outbreak",
            "performance_ci_low": "performance_outbreak_ci_low",
            "performance_ci_high": "performance_outbreak_ci_high",
        }
    )

    all_df = add_se_column(all_df, prefix="performance_all")
    outbreak_df = add_se_column(outbreak_df, prefix="performance_outbreak")

    compare = all_df.merge(
        outbreak_df,
        on=["dataset", "method"],
        how="inner",
        validate="one_to_one",
    )

    if compare.empty:
        raise ValueError("No matching dataset x method rows found between all and outbreak data.")

    # Difference: positive means worse in outbreaks
    compare["outbreak_minus_all"] = (
        compare["performance_outbreak"] - compare["performance_all"]
    )

    compare["outbreak_minus_all_se"] = np.sqrt(
        compare["performance_outbreak_se"] ** 2
        + compare["performance_all_se"] ** 2
    )

    compare["outbreak_minus_all_ci_low"] = (
        compare["outbreak_minus_all"] - 1.96 * compare["outbreak_minus_all_se"]
    )
    compare["outbreak_minus_all_ci_high"] = (
        compare["outbreak_minus_all"] + 1.96 * compare["outbreak_minus_all_se"]
    )

    compare["outbreak_minus_all_z"] = (
        compare["outbreak_minus_all"] / compare["outbreak_minus_all_se"]
    )

    # One-sided p-value for hypothesis: outbreak performance is worse than all-period performance
    compare["p_outbreak_worse_one_sided"] = compare["outbreak_minus_all_z"].map(normal_sf)

    # Two-sided p-value for any difference
    compare["p_any_difference_two_sided"] = (
        2.0 * compare["outbreak_minus_all_z"].abs().map(normal_sf)
    )

    # Ratio: >1 means worse in outbreaks
    compare["outbreak_over_all"] = safe_divide(
        compare["performance_outbreak"],
        compare["performance_all"],
    )

    # Delta-method CI for log ratio
    valid_ratio = (
        np.isfinite(compare["performance_outbreak"])
        & np.isfinite(compare["performance_all"])
        & (compare["performance_outbreak"] > 0)
        & (compare["performance_all"] > 0)
        & np.isfinite(compare["performance_outbreak_se"])
        & np.isfinite(compare["performance_all_se"])
    )

    compare["log_outbreak_over_all"] = np.nan
    compare["log_outbreak_over_all_se"] = np.nan

    compare.loc[valid_ratio, "log_outbreak_over_all"] = np.log(
        compare.loc[valid_ratio, "outbreak_over_all"]
    )

    compare.loc[valid_ratio, "log_outbreak_over_all_se"] = np.sqrt(
        (compare.loc[valid_ratio, "performance_outbreak_se"]
         / compare.loc[valid_ratio, "performance_outbreak"]) ** 2
        +
        (compare.loc[valid_ratio, "performance_all_se"]
         / compare.loc[valid_ratio, "performance_all"]) ** 2
    )

    compare["outbreak_over_all_ci_low"] = np.exp(
        compare["log_outbreak_over_all"] - 1.96 * compare["log_outbreak_over_all_se"]
    )
    compare["outbreak_over_all_ci_high"] = np.exp(
        compare["log_outbreak_over_all"] + 1.96 * compare["log_outbreak_over_all_se"]
    )

    compare["pct_change_in_outbreaks"] = 100.0 * (
        compare["outbreak_over_all"] - 1.0
    )

    compare["worse_in_outbreaks"] = compare["outbreak_minus_all"] > 0
    compare["significantly_worse_in_outbreaks_approx"] = (
        compare["p_outbreak_worse_one_sided"] < 0.05
    )

    sort_cols = ["method", "dataset"]
    compare = compare.sort_values(sort_cols).reset_index(drop=True)

    return compare


def summarize_outbreak_degradation_by_method(compare_df):
    """
    Aggregates outbreak degradation across datasets for each method.

    Uses inverse-variance weighting on:
      1. outbreak_minus_all
      2. log(outbreak_over_all)

    Positive pooled_outbreak_minus_all means the method is worse in outbreaks.
    pooled_outbreak_over_all > 1 means the method is worse in outbreaks.
    """
    rows = []

    for method, g in compare_df.groupby("method", dropna=False):
        # ---------- Difference summary ----------
        valid_diff = (
            np.isfinite(g["outbreak_minus_all"])
            & np.isfinite(g["outbreak_minus_all_se"])
            & (g["outbreak_minus_all_se"] > 0)
        )

        if valid_diff.any():
            gd = g.loc[valid_diff].copy()
            weights = 1.0 / (gd["outbreak_minus_all_se"] ** 2)

            pooled_delta = np.sum(weights * gd["outbreak_minus_all"]) / np.sum(weights)
            pooled_delta_se = np.sqrt(1.0 / np.sum(weights))
            pooled_delta_ci_low = pooled_delta - 1.96 * pooled_delta_se
            pooled_delta_ci_high = pooled_delta + 1.96 * pooled_delta_se
            pooled_delta_z = pooled_delta / pooled_delta_se
            pooled_delta_p_one_sided = normal_sf(pooled_delta_z)
            pooled_delta_p_two_sided = 2.0 * normal_sf(abs(pooled_delta_z))
        else:
            pooled_delta = np.nan
            pooled_delta_se = np.nan
            pooled_delta_ci_low = np.nan
            pooled_delta_ci_high = np.nan
            pooled_delta_z = np.nan
            pooled_delta_p_one_sided = np.nan
            pooled_delta_p_two_sided = np.nan

        # ---------- Ratio summary ----------
        valid_ratio = (
            np.isfinite(g["log_outbreak_over_all"])
            & np.isfinite(g["log_outbreak_over_all_se"])
            & (g["log_outbreak_over_all_se"] > 0)
        )

        if valid_ratio.any():
            gr = g.loc[valid_ratio].copy()
            weights = 1.0 / (gr["log_outbreak_over_all_se"] ** 2)

            pooled_log_ratio = np.sum(weights * gr["log_outbreak_over_all"]) / np.sum(weights)
            pooled_log_ratio_se = np.sqrt(1.0 / np.sum(weights))

            pooled_ratio = np.exp(pooled_log_ratio)
            pooled_ratio_ci_low = np.exp(pooled_log_ratio - 1.96 * pooled_log_ratio_se)
            pooled_ratio_ci_high = np.exp(pooled_log_ratio + 1.96 * pooled_log_ratio_se)
            pooled_pct_change = 100.0 * (pooled_ratio - 1.0)
        else:
            pooled_ratio = np.nan
            pooled_ratio_ci_low = np.nan
            pooled_ratio_ci_high = np.nan
            pooled_pct_change = np.nan

        rows.append(
            {
                "method": method,
                "n_datasets": g["dataset"].nunique(),

                # Difference scale
                "pooled_outbreak_minus_all": pooled_delta,
                "pooled_outbreak_minus_all_se": pooled_delta_se,
                "pooled_outbreak_minus_all_ci_low": pooled_delta_ci_low,
                "pooled_outbreak_minus_all_ci_high": pooled_delta_ci_high,
                "p_outbreak_worse_one_sided": pooled_delta_p_one_sided,
                "p_any_difference_two_sided": pooled_delta_p_two_sided,

                # Ratio scale
                "pooled_outbreak_over_all": pooled_ratio,
                "pooled_outbreak_over_all_ci_low": pooled_ratio_ci_low,
                "pooled_outbreak_over_all_ci_high": pooled_ratio_ci_high,
                "pooled_pct_change_in_outbreaks": pooled_pct_change,

                # Simple descriptive checks
                "num_datasets_worse_in_outbreaks": int(g["worse_in_outbreaks"].sum()),
                "frac_datasets_worse_in_outbreaks": g["worse_in_outbreaks"].mean(),
                "mean_dataset_level_pct_change": g["pct_change_in_outbreaks"].mean(),
                "median_dataset_level_pct_change": g["pct_change_in_outbreaks"].median(),
            }
        )

    summary = pd.DataFrame(rows)

    # Worst-looking methods first
    summary = summary.sort_values(
        ["pooled_outbreak_minus_all", "pooled_outbreak_over_all"],
        ascending=[False, False],
    ).reset_index(drop=True)

    return summary


def print_outbreak_summary(compare_df, summary_df, top_n=20):
    print("\n" + "=" * 80)
    print("Dataset x method outbreak-vs-all comparison")
    print("Positive outbreak_minus_all means worse in outbreaks.")
    print("Ratio outbreak_over_all > 1 means worse in outbreaks.")
    print("=" * 80)

    display_cols = [
        "dataset",
        "method",
        "performance_all",
        "performance_outbreak",
        "outbreak_minus_all",
        "outbreak_minus_all_ci_low",
        "outbreak_minus_all_ci_high",
        "outbreak_over_all",
        "outbreak_over_all_ci_low",
        "outbreak_over_all_ci_high",
        "pct_change_in_outbreaks",
        "p_outbreak_worse_one_sided",
        "significantly_worse_in_outbreaks_approx",
    ]

    print(compare_df[display_cols].round(4).to_string(index=False))

    print("\n" + "=" * 80)
    print("Method-level summary across datasets")
    print("=" * 80)

    method_cols = [
        "method",
        "n_datasets",
        "pooled_outbreak_minus_all",
        "pooled_outbreak_minus_all_ci_low",
        "pooled_outbreak_minus_all_ci_high",
        "pooled_outbreak_over_all",
        "pooled_outbreak_over_all_ci_low",
        "pooled_outbreak_over_all_ci_high",
        "pooled_pct_change_in_outbreaks",
        "p_outbreak_worse_one_sided",
        "num_datasets_worse_in_outbreaks",
        "frac_datasets_worse_in_outbreaks",
    ]

    print(summary_df[method_cols].head(top_n).round(4).to_string(index=False))


def plot_method_degradation_summary(summary_df, save_filename=None):
    """
    Bar plot of method-level pooled outbreak-minus-all degradation.

    Above zero = worse in outbreaks.
    Below zero = better in outbreaks.
    """
    if summary_df.empty:
        print("No method summary to plot.")
        return

    dfp = summary_df.copy()
    dfp = dfp.sort_values("pooled_outbreak_minus_all", ascending=False)

    methods = dfp["method"].astype(str).to_list()
    vals = dfp["pooled_outbreak_minus_all"].to_numpy(dtype=float)

    yerr_low = vals - dfp["pooled_outbreak_minus_all_ci_low"].to_numpy(dtype=float)
    yerr_high = dfp["pooled_outbreak_minus_all_ci_high"].to_numpy(dtype=float) - vals
    yerr = np.vstack([yerr_low, yerr_high])

    x = np.arange(len(methods))

    fig, ax = plt.subplots(figsize=FIGSIZE)

    colors = [get_method_color(method, i) for i, method in enumerate(methods)]

    ax.bar(
        x,
        vals,
        alpha=BAR_ALPHA,
        color=colors,
        yerr=yerr if SHOW_ERROR_BARS else None,
        ecolor=ERROR_BAR_COLOR,
        error_kw={
            "elinewidth": ERROR_BAR_LINEWIDTH,
            "capsize": ERROR_BAR_CAPSIZE,
            "capthick": ERROR_BAR_CAPTHICK,
        } if SHOW_ERROR_BARS else None,
    )

    ax.axhline(
        REFERENCE_DEGRADATION_Y,
        linestyle=REFERENCE_LINESTYLE,
        linewidth=REFERENCE_LINEWIDTH,
        color=REFERENCE_LINECOLOR,
    )

    ax.set_title(TITLE_OUTBREAK_DEGRADATION, fontsize=TITLE_FONTSIZE)
    ax.set_ylabel(Y_LABEL_OUTBREAK_DEGRADATION, fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_xticks(x)
    ax.set_xticklabels(
        methods,
        rotation=XTICK_ROTATION,
        ha=XTICK_HA,
        fontsize=XTICK_FONTSIZE,
    )
    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)

    if SHOW_GRID:
        ax.grid(True, axis=GRID_AXIS, alpha=GRID_ALPHA)

    plt.tight_layout()

    if SAVE_FIGURES and save_filename is not None:
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        save_path = SAVE_DIR / save_filename
        plt.savefig(save_path, dpi=SAVE_DPI, bbox_inches=SAVE_BBOX_INCHES)
        print(f"Saved: {save_path}")

    if SHOW_FIGURES:
        plt.show()

    if CLOSE_FIGURES:
        plt.close(fig)


# =========================================================
# Run outbreak-vs-all analysis
# =========================================================

outbreak_compare_df = compare_outbreak_vs_all(
    plot_df_all=plot_df_all,
    plot_df_outbreak=plot_df_consensus,
)

outbreak_method_summary_df = summarize_outbreak_degradation_by_method(
    outbreak_compare_df
)

print_outbreak_summary(
    outbreak_compare_df,
    outbreak_method_summary_df,
)

if SAVE_COMPARISON_TABLES:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    comparison_path = SAVE_DIR / FILENAME_OUTBREAK_COMPARISON
    summary_path = SAVE_DIR / FILENAME_OUTBREAK_METHOD_SUMMARY

    outbreak_compare_df.to_csv(comparison_path, index=False)
    outbreak_method_summary_df.to_csv(summary_path, index=False)

    print(f"\nSaved: {comparison_path}")
    print(f"Saved: {summary_path}")

if PLOT_OUTBREAK_DEGRADATION:
    plot_method_degradation_summary(
        outbreak_method_summary_df,
        save_filename=FILENAME_OUTBREAK_DEGRADATION_PLOT,
    )

In [ ]:
# =========================================================
# Percent outbreak degradation plot by method
# =========================================================
# Requires outbreak_method_summary_df from the previous code.
#
# Y-axis interpretation:
#   0%    = same relative error in outbreaks and all periods
#   +25%  = method is 25% worse in outbreaks than all periods
#   -10%  = method is 10% better in outbreaks than all periods
#
# Formula:
#   100 * (outbreak_error_relative_to_baseline /
#          all_period_error_relative_to_baseline - 1)
# =========================================================

PLOT_OUTBREAK_PERCENT_DEGRADATION = True

FILENAME_OUTBREAK_PERCENT_DEGRADATION_PLOT = (
    "outbreak_error_relative_to_all_periods_percent_by_method.png"
)

TITLE_OUTBREAK_PERCENT_DEGRADATION = (
    "Outbreak performance"
)

Y_LABEL_OUTBREAK_PERCENT_DEGRADATION = (
    "Additional outbreak error (%)"
)

REFERENCE_PERCENT_Y = 0.0


def build_method_color_index_map_alphabetical(
    df: pd.DataFrame,
    method_col: str = "method",
) -> dict:
    """
    Build stable method -> color index mapping using alphabetical method order.

    This keeps each method's color fixed even if the plot sorts bars
    by percent degradation.
    """
    if df.empty or method_col not in df.columns:
        return {}

    methods = (
        df[method_col]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .sort_values()
        .to_list()
    )

    return {
        method: idx
        for idx, method in enumerate(methods)
    }


def plot_method_percent_outbreak_degradation(
    summary_df: pd.DataFrame,
    method_to_color_idx: dict | None = None,
    save_filename: str | None = None,
):
    """
    Bar plot of percent outbreak degradation by method.

    Uses:
      pooled_outbreak_over_all
      pooled_outbreak_over_all_ci_low
      pooled_outbreak_over_all_ci_high

    Converts those ratios to percentages:
      percent = 100 * (ratio - 1)

    Positive values mean worse in outbreaks.
    Negative values mean better in outbreaks.

    Bars are sorted by degradation, but colors are assigned using
    alphabetical method order.
    """
    if summary_df.empty:
        print("No method summary to plot.")
        return

    needed_cols = [
        "method",
        "pooled_outbreak_over_all",
        "pooled_outbreak_over_all_ci_low",
        "pooled_outbreak_over_all_ci_high",
    ]

    missing = [c for c in needed_cols if c not in summary_df.columns]
    if missing:
        raise ValueError(f"Missing expected columns in summary_df: {missing}")

    if method_to_color_idx is None:
        method_to_color_idx = build_method_color_index_map_alphabetical(
            summary_df,
            method_col="method",
        )

    dfp = summary_df.copy()

    dfp["method"] = dfp["method"].astype(str)

    dfp["percent_outbreak_relative_to_all"] = (
        100.0 * (dfp["pooled_outbreak_over_all"] - 1.0)
    )

    dfp["percent_ci_low"] = (
        100.0 * (dfp["pooled_outbreak_over_all_ci_low"] - 1.0)
    )

    dfp["percent_ci_high"] = (
        100.0 * (dfp["pooled_outbreak_over_all_ci_high"] - 1.0)
    )

    # Put the most outbreak-degraded methods first.
    # Colors remain fixed because they use alphabetical method_to_color_idx.
    dfp = dfp.sort_values(
        "percent_outbreak_relative_to_all",
        ascending=False,
    ).reset_index(drop=True)

    methods = dfp["method"].to_list()
    vals = dfp["percent_outbreak_relative_to_all"].to_numpy(dtype=float)

    yerr_low = vals - dfp["percent_ci_low"].to_numpy(dtype=float)
    yerr_high = dfp["percent_ci_high"].to_numpy(dtype=float) - vals

    yerr_low = np.where(np.isfinite(yerr_low), yerr_low, np.nan)
    yerr_high = np.where(np.isfinite(yerr_high), yerr_high, np.nan)

    yerr = np.vstack([yerr_low, yerr_high])

    x = np.arange(len(methods))

    fig, ax = plt.subplots(figsize=(8, 8))

    colors = [
        get_method_color(
            method,
            method_to_color_idx.get(method, i),
        )
        for i, method in enumerate(methods)
    ]

    ax.bar(
        x,
        vals,
        alpha=BAR_ALPHA,
        color=colors,
        yerr=yerr if SHOW_ERROR_BARS else None,
        ecolor=ERROR_BAR_COLOR,
        error_kw={
            "elinewidth": ERROR_BAR_LINEWIDTH,
            "capsize": ERROR_BAR_CAPSIZE,
            "capthick": ERROR_BAR_CAPTHICK,
        } if SHOW_ERROR_BARS else None,
    )

    ax.axhline(
        REFERENCE_PERCENT_Y,
        linestyle=REFERENCE_LINESTYLE,
        linewidth=REFERENCE_LINEWIDTH,
        color=REFERENCE_LINECOLOR,
    )

    ax.set_title(
        TITLE_OUTBREAK_PERCENT_DEGRADATION,
        fontsize=TITLE_FONTSIZE,
    )

    ax.set_ylabel(
        Y_LABEL_OUTBREAK_PERCENT_DEGRADATION,
        fontsize=AXIS_LABEL_FONTSIZE,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        methods,
        rotation=XTICK_ROTATION,
        ha="right",
        fontsize=XTICK_FONTSIZE,
    )

    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)

    # Format y-axis as percentages.
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda y, _: f"{y:.0f}%")
    )

    if SHOW_GRID:
        ax.grid(True, axis=GRID_AXIS, alpha=GRID_ALPHA)

    plt.tight_layout()

    if SAVE_FIGURES and save_filename is not None:
        SAVE_DIR.mkdir(parents=True, exist_ok=True)

        save_path = SAVE_DIR / save_filename

        plt.savefig(
            save_path,
            dpi=SAVE_DPI,
            bbox_inches=SAVE_BBOX_INCHES,
        )

        print(f"Saved: {save_path}")

    if SHOW_FIGURES:
        plt.show()

    if CLOSE_FIGURES:
        plt.close(fig)


# =========================================================
# Build alphabetical color map and plot
# =========================================================

outbreak_degradation_method_to_color_idx = (
    build_method_color_index_map_alphabetical(
        outbreak_method_summary_df,
        method_col="method",
    )
)

if PLOT_OUTBREAK_PERCENT_DEGRADATION:
    plot_method_percent_outbreak_degradation(
        outbreak_method_summary_df,
        method_to_color_idx=outbreak_degradation_method_to_color_idx,
        save_filename=FILENAME_OUTBREAK_PERCENT_DEGRADATION_PLOT,
    )

In [ ]:
FIGSIZE

### BEST PATCHES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =========================================================
# Editable parameters
# =========================================================

# Choose which results dict to use:
#   results_dfs
#   results_dfs_consensus
RESULTS_DICT = results_dfs

metric_cols = [
    # Standard point metrics
    "mse",
    "mae",
    "rmse",
    "medae",
    "medse",

    # Filtered point metrics
    "mse_filtered",
    "mae_filtered",
    "rmse_filtered",
]

METRIC_LABELS = {
    # Standard point metrics
    "mse": "MSE",
    "mae": "MAE",
    "rmse": "RMSE",
    "medae": "medAE",
    "medse": "medSE",

    # Filtered point metrics
    "mse_filtered": "MSE filtered",
    "mae_filtered": "MAE filtered",
    "rmse_filtered": "RMSE filtered",
}

group_cols = ["epi", "einn", "filter", "ti"]
CONFIG_COLS = ["epi", "einn", "filter", "ti"]

BASELINE_METHOD_NAME = "repeat_last"
INCLUDE_REPEAT_LAST = False

# How to aggregate across horizons using existing CIs
# Options:
#   "inverse_variance"
#   "simple_mean"
HORIZON_AGG_METHOD = "inverse_variance"

# ---------------------------------------------------------
# Figure / layout
# ---------------------------------------------------------
FIGSIZE_WIN = (14, 8)
FIGSIZE_PERF = (14, 8)

BAR_GROUP_WIDTH = 0.8
BAR_ALPHA = 0.9

SHOW_GRID = False
GRID_ALPHA = 0.25
GRID_AXIS = "y"

# Win-rate axis
WIN_YMIN = 0.0
WIN_YMAX = 1.0

# Performance axis
PERF_YMIN = 0.0
PERF_YMAX = 2.0

# Reference lines
WIN_REFERENCE_Y = 0.5
WIN_REFERENCE_LINESTYLE = "--"
WIN_REFERENCE_LINEWIDTH = 1.5
WIN_REFERENCE_LINECOLOR = "black"

PERF_REFERENCE_Y = 1.0
PERF_REFERENCE_LINESTYLE = "--"
PERF_REFERENCE_LINEWIDTH = 1.5
PERF_REFERENCE_LINECOLOR = "black"

# ---------------------------------------------------------
# Error bars / CI style
# ---------------------------------------------------------
SHOW_ERROR_BARS = True
ERROR_BAR_COLOR = "black"
ERROR_BAR_LINEWIDTH = 1.2
ERROR_BAR_CAPSIZE = 4
ERROR_BAR_CAPTHICK = 1.2

# ---------------------------------------------------------
# Fonts
# ---------------------------------------------------------
TITLE_FONTSIZE = 24
AXIS_LABEL_FONTSIZE = 24
XTICK_FONTSIZE = 22
YTICK_FONTSIZE = 18
LEGEND_FONTSIZE = 18

# ---------------------------------------------------------
# Text
# ---------------------------------------------------------
WIN_TITLE_TEMPLATE = "Win rate (best patches, all periods, {metric_label})"
PERF_TITLE_TEMPLATE = "Overall performance (best patches, all periods, {metric_label})"

X_LABEL = ""
WIN_Y_LABEL = "Win rate"
PERF_Y_LABEL = "Error relative to baseline"

WIN_REFERENCE_LABEL = "50%"
PERF_REFERENCE_LABEL = "repeat_last = 1"

# ---------------------------------------------------------
# X tick / labels
# ---------------------------------------------------------
XTICK_ROTATION = 45
XTICK_HA = "right"

# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------
SHOW_LEGEND = True
LEGEND_LOC = "upper left"
LEGEND_BBOX_TO_ANCHOR = (1.02, 1.0)
LEGEND_FRAMEON = True
LEGEND_TITLE = ""

# ---------------------------------------------------------
# Method colors
# Leave empty to use the shared 20-color fallback palette.
# ---------------------------------------------------------
METHOD_COLORS = {
    # "AGCRN": "tab:blue",
    # "DCRNN": "tab:orange",
}

FALLBACK_COLOR_CYCLE = list(plt.get_cmap("tab20").colors)

# ---------------------------------------------------------
# Save / show
# ---------------------------------------------------------
SAVE_FIGURES = True
SAVE_DIR = Path(".")
SAVE_DPI = 300
SAVE_BBOX_INCHES = "tight"

WIN_FILENAME_TEMPLATE = "best_patch_win_rate_across_datasets_all_{metric}.png"
PERF_FILENAME_TEMPLATE = "best_patch_performance_across_datasets_all_{metric}.png"

SHOW_FIGURES = True
CLOSE_FIGURES = True


# =========================================================
# Helpers
# =========================================================

def get_metric_label(metric: str) -> str:
    return METRIC_LABELS.get(metric, metric)


def normalize_config_value(x):
    if pd.isna(x):
        return x
    if isinstance(x, str):
        if x == "True":
            return True
        if x == "False":
            return False
    return x


def standardize_config_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in CONFIG_COLS:
        if col in df.columns:
            df[col] = df[col].map(normalize_config_value)
    return df


def safe_divide(num, den):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)

    out = np.full(np.broadcast(num, den).shape, np.nan, dtype=float)

    valid = np.isfinite(num) & np.isfinite(den) & (den != 0)
    out[valid] = num[valid] / den[valid]

    return out


def ci_to_se(ci_low, ci_high):
    if not (np.isfinite(ci_low) and np.isfinite(ci_high)):
        return np.nan

    width = ci_high - ci_low
    if width <= 0:
        return np.nan

    return width / (2 * 1.96)


def clip01(x):
    return np.clip(x, 0.0, 1.0) if np.isfinite(x) else x


def make_yerr(val, ci_low, ci_high):
    """
    Matplotlib expects nonnegative asymmetric error sizes.
    """
    if pd.isna(val) or pd.isna(ci_low) or pd.isna(ci_high):
        return np.nan, np.nan

    low_err = max(val - ci_low, 0.0)
    high_err = max(ci_high - val, 0.0)

    return low_err, high_err


def aggregate_estimates_with_ci(
    means: np.ndarray,
    ci_lows: np.ndarray,
    ci_highs: np.ndarray,
    method: str = "inverse_variance",
    clip_to_01: bool = False,
):
    """
    Aggregate horizon-level estimates with existing CIs.

    Returns:
      pooled_mean, pooled_ci_low, pooled_ci_high
    """
    means = np.asarray(means, dtype=float)
    ci_lows = np.asarray(ci_lows, dtype=float)
    ci_highs = np.asarray(ci_highs, dtype=float)

    valid = np.isfinite(means) & np.isfinite(ci_lows) & np.isfinite(ci_highs)

    means = means[valid]
    ci_lows = ci_lows[valid]
    ci_highs = ci_highs[valid]

    if len(means) == 0:
        return np.nan, np.nan, np.nan

    if len(means) == 1:
        pooled_mean = means[0]
        pooled_ci_low = ci_lows[0]
        pooled_ci_high = ci_highs[0]

    elif method == "simple_mean":
        pooled_mean = means.mean()
        pooled_ci_low = ci_lows.mean()
        pooled_ci_high = ci_highs.mean()

    elif method == "inverse_variance":
        ses = np.array(
            [ci_to_se(lo, hi) for lo, hi in zip(ci_lows, ci_highs)],
            dtype=float,
        )

        valid_se = np.isfinite(ses) & (ses > 0)

        if not valid_se.any():
            pooled_mean = means.mean()
            pooled_ci_low = ci_lows.mean()
            pooled_ci_high = ci_highs.mean()
        else:
            means = means[valid_se]
            ses = ses[valid_se]

            weights = 1.0 / (ses ** 2)

            pooled_mean = np.sum(weights * means) / np.sum(weights)
            pooled_se = np.sqrt(1.0 / np.sum(weights))

            pooled_ci_low = pooled_mean - 1.96 * pooled_se
            pooled_ci_high = pooled_mean + 1.96 * pooled_se

    else:
        raise ValueError(f"Unsupported HORIZON_AGG_METHOD={method}")

    if clip_to_01:
        pooled_mean = clip01(pooled_mean)
        pooled_ci_low = clip01(pooled_ci_low)
        pooled_ci_high = clip01(pooled_ci_high)

    return pooled_mean, pooled_ci_low, pooled_ci_high


def get_method_color(method, idx):
    if method in METHOD_COLORS:
        return METHOD_COLORS[method]

    if len(FALLBACK_COLOR_CYCLE) > 0:
        return FALLBACK_COLOR_CYCLE[idx % len(FALLBACK_COLOR_CYCLE)]

    return None


# =========================================================
# Core computation
# =========================================================

def compute_baseline_norm_rows_with_ci_for_metric(
    df_in: pd.DataFrame,
    metric_col: str,
    group_cols: list[str],
) -> pd.DataFrame:
    """
    Recompute baseline-normalized performance relative to repeat_last
    within each setting + horizon for one metric.

    Input dataframe is expected to have:
      <metric_col>_mean
      <metric_col>_ci_low
      <metric_col>_ci_high

    Returns row-level dataframe with:
      baseline_norm_mean
      baseline_norm_ci_low
      baseline_norm_ci_high
    """
    df = df_in.copy()

    metric_mean_col = f"{metric_col}_mean"
    metric_low_col = f"{metric_col}_ci_low"
    metric_high_col = f"{metric_col}_ci_high"

    needed = [metric_mean_col, metric_low_col, metric_high_col]
    missing = [c for c in needed if c not in df.columns]

    if missing:
        raise ValueError(
            f"For metric '{metric_col}', missing expected columns: {missing}"
        )

    rows = []

    for _, df_group in df.groupby(group_cols, dropna=False):
        if not (df_group["method"] == BASELINE_METHOD_NAME).any():
            continue

        valid_horizons = (
            df_group.groupby("horizon")[metric_mean_col]
            .apply(lambda x: x.notna().any())
        )
        valid_horizons = valid_horizons[valid_horizons].index

        df_sub = df_group[df_group["horizon"].isin(valid_horizons)].copy()

        if df_sub.empty:
            continue

        baseline_cols = [
            "horizon",
            metric_mean_col,
            metric_low_col,
            metric_high_col,
        ]

        baseline = (
            df_sub[df_sub["method"] == BASELINE_METHOD_NAME][baseline_cols]
            .drop_duplicates(subset=["horizon"])
            .set_index("horizon")
        )

        if baseline.empty:
            continue

        baseline = baseline.rename(
            columns={
                metric_mean_col: "baseline_mean",
                metric_low_col: "baseline_ci_low",
                metric_high_col: "baseline_ci_high",
            }
        )

        df_sub = df_sub.merge(
            baseline,
            left_on="horizon",
            right_index=True,
            how="left",
        )

        df_sub["baseline_norm_mean"] = safe_divide(
            df_sub[metric_mean_col].to_numpy(),
            df_sub["baseline_mean"].to_numpy(),
        )

        # Conservative CI propagation:
        # lower bound uses model lower / baseline upper
        # upper bound uses model upper / baseline lower
        df_sub["baseline_norm_ci_low"] = safe_divide(
            df_sub[metric_low_col].to_numpy(),
            df_sub["baseline_ci_high"].to_numpy(),
        )

        df_sub["baseline_norm_ci_high"] = safe_divide(
            df_sub[metric_high_col].to_numpy(),
            df_sub["baseline_ci_low"].to_numpy(),
        )

        df_sub[
            [
                "baseline_norm_mean",
                "baseline_norm_ci_low",
                "baseline_norm_ci_high",
            ]
        ] = df_sub[
            [
                "baseline_norm_mean",
                "baseline_norm_ci_low",
                "baseline_norm_ci_high",
            ]
        ].replace([np.inf, -np.inf], np.nan)

        df_sub = df_sub.dropna(subset=["baseline_norm_mean"]).copy()

        if not df_sub.empty:
            rows.append(df_sub)

    if not rows:
        raise ValueError(
            f"No valid combinations found with repeat_last baseline for metric '{metric_col}'."
        )

    return pd.concat(rows, ignore_index=True)


def build_best_config_tables(
    results_dict: dict,
    metric_col: str,
):
    """
    For each dataset x method:
      1) compute performance relative to repeat_last for one metric
      2) aggregate performance across horizons for each config
      3) select best config, meaning lowest pooled performance
      4) aggregate win_rate across horizons for that chosen config

    Returns:
      best_perf_df:
        dataset, method, metric, performance, performance_ci_low, performance_ci_high

      best_win_df:
        dataset, method, metric, win_rate, win_rate_ci_low, win_rate_ci_high

      best_configs_df:
        dataset, method, metric, epi, einn, filter, ti
    """
    perf_rows = []
    win_rows = []
    cfg_rows = []

    for dataset, df in results_dict.items():
        if df is None or len(df) == 0:
            continue

        df = standardize_config_cols(df)

        try:
            df_perf = compute_baseline_norm_rows_with_ci_for_metric(
                df_in=df,
                metric_col=metric_col,
                group_cols=group_cols,
            )
        except ValueError as e:
            print(f"Skipping {dataset}, metric={metric_col}: {e}")
            continue

        if not INCLUDE_REPEAT_LAST:
            df_perf = df_perf[df_perf["method"] != BASELINE_METHOD_NAME].copy()
            df = df[df["method"] != BASELINE_METHOD_NAME].copy()

        if df_perf.empty or df.empty:
            print(f"Skipping {dataset}, metric={metric_col}: no methods left after filtering")
            continue

        # ---------------------------------------------
        # Aggregate performance across horizons for each config
        # ---------------------------------------------
        config_perf_rows = []

        for keys, g in df_perf.groupby(["method"] + group_cols, dropna=False):
            method = keys[0]
            cfg = keys[1:]

            pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
                means=g["baseline_norm_mean"].to_numpy(dtype=float),
                ci_lows=g["baseline_norm_ci_low"].to_numpy(dtype=float),
                ci_highs=g["baseline_norm_ci_high"].to_numpy(dtype=float),
                method=HORIZON_AGG_METHOD,
                clip_to_01=False,
            )

            config_perf_rows.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "metric": metric_col,
                    "epi": cfg[0],
                    "einn": cfg[1],
                    "filter": cfg[2],
                    "ti": cfg[3],
                    "performance": pooled_mean,
                    "performance_ci_low": pooled_ci_low,
                    "performance_ci_high": pooled_ci_high,
                }
            )

        config_perf_df = pd.DataFrame(config_perf_rows)

        if config_perf_df.empty:
            print(f"Skipping {dataset}, metric={metric_col}: no valid config-level performance rows")
            continue

        # ---------------------------------------------
        # Best config per method = lowest performance
        # ---------------------------------------------
        best_idx = config_perf_df.groupby("method")["performance"].idxmin()
        best_configs_df = config_perf_df.loc[best_idx].reset_index(drop=True)

        for _, best_row in best_configs_df.iterrows():
            method = best_row["method"]
            epi = best_row["epi"]
            einn = best_row["einn"]
            filt = best_row["filter"]
            ti = best_row["ti"]

            cfg_rows.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "metric": metric_col,
                    "epi": epi,
                    "einn": einn,
                    "filter": filt,
                    "ti": ti,
                }
            )

            perf_rows.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "metric": metric_col,
                    "performance": best_row["performance"],
                    "performance_ci_low": best_row["performance_ci_low"],
                    "performance_ci_high": best_row["performance_ci_high"],
                }
            )

            # -----------------------------------------
            # Aggregate win_rate across horizons for chosen config
            # -----------------------------------------
            g_win = df[
                (df["method"] == method)
                & (df["epi"] == epi)
                & (df["einn"] == einn)
                & (df["filter"] == filt)
                & (df["ti"] == ti)
            ].copy()

            if g_win.empty:
                win_rows.append(
                    {
                        "dataset": dataset,
                        "method": method,
                        "metric": metric_col,
                        "win_rate": np.nan,
                        "win_rate_ci_low": np.nan,
                        "win_rate_ci_high": np.nan,
                    }
                )
                continue

            pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
                means=g_win["win_rate_mean"].to_numpy(dtype=float),
                ci_lows=g_win["win_rate_ci_low"].to_numpy(dtype=float),
                ci_highs=g_win["win_rate_ci_high"].to_numpy(dtype=float),
                method=HORIZON_AGG_METHOD,
                clip_to_01=True,
            )

            win_rows.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "metric": metric_col,
                    "win_rate": pooled_mean,
                    "win_rate_ci_low": pooled_ci_low,
                    "win_rate_ci_high": pooled_ci_high,
                }
            )

    best_perf_df = pd.DataFrame(perf_rows)
    best_win_df = pd.DataFrame(win_rows)
    best_configs_df = pd.DataFrame(cfg_rows)

    return best_perf_df, best_win_df, best_configs_df


# =========================================================
# Plotting
# =========================================================

def plot_grouped_barplot(
    plot_df: pd.DataFrame,
    value_col: str,
    ci_low_col: str,
    ci_high_col: str,
    title: str,
    y_label: str,
    reference_y: float,
    reference_label: str,
    reference_linestyle: str,
    reference_linewidth: float,
    reference_linecolor: str,
    ylim: tuple[float, float],
    figsize: tuple[float, float],
    save_filename: str | None = None,
):
    if plot_df.empty:
        print(f"No data to plot for: {title}")
        return

    datasets_order = list(plot_df["dataset"].drop_duplicates())
    methods_order = list(plot_df["method"].drop_duplicates())

    x = np.arange(len(datasets_order))
    n_methods = len(methods_order)

    if n_methods == 0:
        print(f"No methods to plot for: {title}")
        return

    bar_width = BAR_GROUP_WIDTH / n_methods

    fig, ax = plt.subplots(figsize=figsize)

    for i, method in enumerate(methods_order):
        vals = []
        yerr_low = []
        yerr_high = []

        for dataset in datasets_order:
            sub = plot_df[
                (plot_df["dataset"] == dataset)
                & (plot_df["method"] == method)
            ]

            if sub.empty:
                vals.append(np.nan)
                yerr_low.append(np.nan)
                yerr_high.append(np.nan)
            else:
                row = sub.iloc[0]

                val = row[value_col]
                ci_low = row[ci_low_col]
                ci_high = row[ci_high_col]

                low_err, high_err = make_yerr(val, ci_low, ci_high)

                vals.append(val)
                yerr_low.append(low_err)
                yerr_high.append(high_err)

        vals = np.asarray(vals, dtype=float)
        yerr = np.vstack(
            [
                np.asarray(yerr_low, dtype=float),
                np.asarray(yerr_high, dtype=float),
            ]
        )

        offsets = x - (BAR_GROUP_WIDTH / 2) + (i + 0.5) * bar_width
        color = get_method_color(method, i)

        ax.bar(
            offsets,
            vals,
            width=bar_width,
            label=method,
            alpha=BAR_ALPHA,
            color=color,
            yerr=yerr if SHOW_ERROR_BARS else None,
            ecolor=ERROR_BAR_COLOR,
            error_kw={
                "elinewidth": ERROR_BAR_LINEWIDTH,
                "capsize": ERROR_BAR_CAPSIZE,
                "capthick": ERROR_BAR_CAPTHICK,
            }
            if SHOW_ERROR_BARS
            else None,
        )

    ax.axhline(
        reference_y,
        linestyle=reference_linestyle,
        linewidth=reference_linewidth,
        color=reference_linecolor,
        label=reference_label if not SHOW_LEGEND else None,
    )

    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    ax.set_xlabel(X_LABEL, fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel(y_label, fontsize=AXIS_LABEL_FONTSIZE)

    ax.set_xticks(x)
    ax.set_xticklabels(
        datasets_order,
        rotation=XTICK_ROTATION,
        ha=XTICK_HA,
        fontsize=XTICK_FONTSIZE,
    )

    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)
    ax.set_ylim(ylim)

    if SHOW_GRID:
        ax.grid(True, axis=GRID_AXIS, alpha=GRID_ALPHA)

    if SHOW_LEGEND:
        ax.legend(
            loc=LEGEND_LOC,
            bbox_to_anchor=LEGEND_BBOX_TO_ANCHOR,
            fontsize=LEGEND_FONTSIZE,
            frameon=LEGEND_FRAMEON,
            title=LEGEND_TITLE,
        )

    plt.tight_layout()

    if SAVE_FIGURES and save_filename is not None:
        SAVE_DIR.mkdir(parents=True, exist_ok=True)

        save_path = SAVE_DIR / save_filename
        plt.savefig(save_path, dpi=SAVE_DPI, bbox_inches=SAVE_BBOX_INCHES)

        print(f"Saved: {save_path}")

    if SHOW_FIGURES:
        plt.show()

    if CLOSE_FIGURES:
        plt.close(fig)


# =========================================================
# Build best-config tables and plot for each metric
# =========================================================

best_perf_dfs = {}
best_win_dfs = {}
best_configs_dfs = {}

for metric_col in metric_cols:
    metric_display = get_metric_label(metric_col)

    best_perf_df, best_win_df, best_configs_df = build_best_config_tables(
        results_dict=RESULTS_DICT,
        metric_col=metric_col,
    )

    best_perf_dfs[metric_col] = best_perf_df
    best_win_dfs[metric_col] = best_win_df
    best_configs_dfs[metric_col] = best_configs_df

    plot_grouped_barplot(
        plot_df=best_win_df,
        value_col="win_rate",
        ci_low_col="win_rate_ci_low",
        ci_high_col="win_rate_ci_high",
        title=WIN_TITLE_TEMPLATE.format(metric_label=metric_display),
        y_label=WIN_Y_LABEL,
        reference_y=WIN_REFERENCE_Y,
        reference_label=WIN_REFERENCE_LABEL,
        reference_linestyle=WIN_REFERENCE_LINESTYLE,
        reference_linewidth=WIN_REFERENCE_LINEWIDTH,
        reference_linecolor=WIN_REFERENCE_LINECOLOR,
        ylim=(WIN_YMIN, WIN_YMAX),
        figsize=FIGSIZE_WIN,
        save_filename=WIN_FILENAME_TEMPLATE.format(metric=metric_col),
    )

    plot_grouped_barplot(
        plot_df=best_perf_df,
        value_col="performance",
        ci_low_col="performance_ci_low",
        ci_high_col="performance_ci_high",
        title=PERF_TITLE_TEMPLATE.format(metric_label=metric_display),
        y_label=PERF_Y_LABEL,
        reference_y=PERF_REFERENCE_Y,
        reference_label=PERF_REFERENCE_LABEL,
        reference_linestyle=PERF_REFERENCE_LINESTYLE,
        reference_linewidth=PERF_REFERENCE_LINEWIDTH,
        reference_linecolor=PERF_REFERENCE_LINECOLOR,
        ylim=(PERF_YMIN, PERF_YMAX),
        figsize=FIGSIZE_PERF,
        save_filename=PERF_FILENAME_TEMPLATE.format(metric=metric_col),
    )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# =========================================================
# Editable parameters
# =========================================================

# Outbreak-only results dictionary
# If your notebook uses a different outbreak variable name, change this line.
RESULTS_DICT = results_dfs_outbreak

metric_cols = [
    # Standard point metrics
    "mse",
    "mae",
    "rmse",
    "medae",
    "medse",

    # Filtered point metrics
    "mse_filtered",
    "mae_filtered",
    "rmse_filtered",
]

METRIC_LABELS = {
    # Standard point metrics
    "mse": "MSE",
    "mae": "MAE",
    "rmse": "RMSE",
    "medae": "medAE",
    "medse": "medSE",

    # Filtered point metrics
    "mse_filtered": "MSE filtered",
    "mae_filtered": "MAE filtered",
    "rmse_filtered": "RMSE filtered",
}

group_cols = ["epi", "einn", "filter", "ti"]
CONFIG_COLS = ["epi", "einn", "filter", "ti"]

BASELINE_METHOD_NAME = "repeat_last"
INCLUDE_REPEAT_LAST = False

# How to aggregate across horizons using existing CIs
# Options:
#   "inverse_variance"
#   "simple_mean"
HORIZON_AGG_METHOD = "inverse_variance"

# ---------------------------------------------------------
# Figure / layout
# ---------------------------------------------------------
FIGSIZE_WIN = (14, 8)
FIGSIZE_PERF = (14, 8)

BAR_GROUP_WIDTH = 0.8
BAR_ALPHA = 0.9

SHOW_GRID = False
GRID_ALPHA = 0.25
GRID_AXIS = "y"

# Win-rate axis
WIN_YMIN = 0.0
WIN_YMAX = 1.0

# Performance axis
PERF_YMIN = 0.0
PERF_YMAX = 2.0

# Reference lines
WIN_REFERENCE_Y = 0.5
WIN_REFERENCE_LINESTYLE = "--"
WIN_REFERENCE_LINEWIDTH = 1.5
WIN_REFERENCE_LINECOLOR = "black"

PERF_REFERENCE_Y = 1.0
PERF_REFERENCE_LINESTYLE = "--"
PERF_REFERENCE_LINEWIDTH = 1.5
PERF_REFERENCE_LINECOLOR = "black"

# ---------------------------------------------------------
# Error bars / CI style
# ---------------------------------------------------------
SHOW_ERROR_BARS = True
ERROR_BAR_COLOR = "black"
ERROR_BAR_LINEWIDTH = 1.2
ERROR_BAR_CAPSIZE = 4
ERROR_BAR_CAPTHICK = 1.2

# ---------------------------------------------------------
# Fonts
# ---------------------------------------------------------
TITLE_FONTSIZE = 24
AXIS_LABEL_FONTSIZE = 24
XTICK_FONTSIZE = 22
YTICK_FONTSIZE = 18
LEGEND_FONTSIZE = 18

# ---------------------------------------------------------
# Text
# ---------------------------------------------------------
WIN_TITLE_TEMPLATE = "Win rate (best patches, outbreak periods, {metric_label})"
PERF_TITLE_TEMPLATE = "Overall performance (best patches, outbreak periods, {metric_label})"

X_LABEL = ""
WIN_Y_LABEL = "Win rate"
PERF_Y_LABEL = "Error relative to baseline"

WIN_REFERENCE_LABEL = "50%"
PERF_REFERENCE_LABEL = "repeat_last = 1"

# ---------------------------------------------------------
# X tick / labels
# ---------------------------------------------------------
XTICK_ROTATION = 45
XTICK_HA = "right"

# ---------------------------------------------------------
# Legend
# ---------------------------------------------------------
SHOW_LEGEND = True
LEGEND_LOC = "upper left"
LEGEND_BBOX_TO_ANCHOR = (1.02, 1.0)
LEGEND_FRAMEON = True
LEGEND_TITLE = ""

# ---------------------------------------------------------
# Method colors
# Leave empty to use the shared 20-color fallback palette.
# ---------------------------------------------------------
METHOD_COLORS = {
    # "AGCRN": "tab:blue",
    # "DCRNN": "tab:orange",
}

FALLBACK_COLOR_CYCLE = list(plt.get_cmap("tab20").colors)

# ---------------------------------------------------------
# Save / show
# ---------------------------------------------------------
SAVE_FIGURES = True
SAVE_DIR = Path(".")
SAVE_DPI = 300
SAVE_BBOX_INCHES = "tight"

WIN_FILENAME_TEMPLATE = "best_patch_win_rate_across_datasets_outbreak_{metric}.png"
PERF_FILENAME_TEMPLATE = "best_patch_performance_across_datasets_outbreak_{metric}.png"

SHOW_FIGURES = True
CLOSE_FIGURES = True


# =========================================================
# Helpers
# =========================================================

def get_metric_label(metric: str) -> str:
    return METRIC_LABELS.get(metric, metric)


def normalize_config_value(x):
    if pd.isna(x):
        return x
    if isinstance(x, str):
        if x == "True":
            return True
        if x == "False":
            return False
    return x


def standardize_config_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in CONFIG_COLS:
        if col in df.columns:
            df[col] = df[col].map(normalize_config_value)
    return df


def safe_divide(num, den):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)

    out = np.full(np.broadcast(num, den).shape, np.nan, dtype=float)

    valid = np.isfinite(num) & np.isfinite(den) & (den != 0)
    out[valid] = num[valid] / den[valid]

    return out


def ci_to_se(ci_low, ci_high):
    if not (np.isfinite(ci_low) and np.isfinite(ci_high)):
        return np.nan

    width = ci_high - ci_low
    if width <= 0:
        return np.nan

    return width / (2 * 1.96)


def clip01(x):
    return np.clip(x, 0.0, 1.0) if np.isfinite(x) else x


def make_yerr(val, ci_low, ci_high):
    """
    Matplotlib expects nonnegative asymmetric error sizes.
    """
    if pd.isna(val) or pd.isna(ci_low) or pd.isna(ci_high):
        return np.nan, np.nan

    low_err = max(val - ci_low, 0.0)
    high_err = max(ci_high - val, 0.0)

    return low_err, high_err


def aggregate_estimates_with_ci(
    means: np.ndarray,
    ci_lows: np.ndarray,
    ci_highs: np.ndarray,
    method: str = "inverse_variance",
    clip_to_01: bool = False,
):
    """
    Aggregate horizon-level estimates with existing CIs.

    Returns:
      pooled_mean, pooled_ci_low, pooled_ci_high
    """
    means = np.asarray(means, dtype=float)
    ci_lows = np.asarray(ci_lows, dtype=float)
    ci_highs = np.asarray(ci_highs, dtype=float)

    valid = np.isfinite(means) & np.isfinite(ci_lows) & np.isfinite(ci_highs)

    means = means[valid]
    ci_lows = ci_lows[valid]
    ci_highs = ci_highs[valid]

    if len(means) == 0:
        return np.nan, np.nan, np.nan

    if len(means) == 1:
        pooled_mean = means[0]
        pooled_ci_low = ci_lows[0]
        pooled_ci_high = ci_highs[0]

    elif method == "simple_mean":
        pooled_mean = means.mean()
        pooled_ci_low = ci_lows.mean()
        pooled_ci_high = ci_highs.mean()

    elif method == "inverse_variance":
        ses = np.array(
            [ci_to_se(lo, hi) for lo, hi in zip(ci_lows, ci_highs)],
            dtype=float,
        )

        valid_se = np.isfinite(ses) & (ses > 0)

        if not valid_se.any():
            pooled_mean = means.mean()
            pooled_ci_low = ci_lows.mean()
            pooled_ci_high = ci_highs.mean()
        else:
            means = means[valid_se]
            ses = ses[valid_se]

            weights = 1.0 / (ses ** 2)

            pooled_mean = np.sum(weights * means) / np.sum(weights)
            pooled_se = np.sqrt(1.0 / np.sum(weights))

            pooled_ci_low = pooled_mean - 1.96 * pooled_se
            pooled_ci_high = pooled_mean + 1.96 * pooled_se

    else:
        raise ValueError(f"Unsupported HORIZON_AGG_METHOD={method}")

    if clip_to_01:
        pooled_mean = clip01(pooled_mean)
        pooled_ci_low = clip01(pooled_ci_low)
        pooled_ci_high = clip01(pooled_ci_high)

    return pooled_mean, pooled_ci_low, pooled_ci_high


def get_method_color(method, idx):
    if method in METHOD_COLORS:
        return METHOD_COLORS[method]

    if len(FALLBACK_COLOR_CYCLE) > 0:
        return FALLBACK_COLOR_CYCLE[idx % len(FALLBACK_COLOR_CYCLE)]

    return None


# =========================================================
# Core computation
# =========================================================

def compute_baseline_norm_rows_with_ci_for_metric(
    df_in: pd.DataFrame,
    metric_col: str,
    group_cols: list[str],
) -> pd.DataFrame:
    """
    Recompute baseline-normalized performance relative to repeat_last
    within each setting + horizon for one metric.

    Input dataframe is expected to have:
      <metric_col>_mean
      <metric_col>_ci_low
      <metric_col>_ci_high

    Returns row-level dataframe with:
      baseline_norm_mean
      baseline_norm_ci_low
      baseline_norm_ci_high
    """
    df = df_in.copy()

    metric_mean_col = f"{metric_col}_mean"
    metric_low_col = f"{metric_col}_ci_low"
    metric_high_col = f"{metric_col}_ci_high"

    needed = [metric_mean_col, metric_low_col, metric_high_col]
    missing = [c for c in needed if c not in df.columns]

    if missing:
        raise ValueError(
            f"For metric '{metric_col}', missing expected columns: {missing}"
        )

    rows = []

    for _, df_group in df.groupby(group_cols, dropna=False):
        if not (df_group["method"] == BASELINE_METHOD_NAME).any():
            continue

        valid_horizons = (
            df_group.groupby("horizon")[metric_mean_col]
            .apply(lambda x: x.notna().any())
        )
        valid_horizons = valid_horizons[valid_horizons].index

        df_sub = df_group[df_group["horizon"].isin(valid_horizons)].copy()

        if df_sub.empty:
            continue

        baseline_cols = [
            "horizon",
            metric_mean_col,
            metric_low_col,
            metric_high_col,
        ]

        baseline = (
            df_sub[df_sub["method"] == BASELINE_METHOD_NAME][baseline_cols]
            .drop_duplicates(subset=["horizon"])
            .set_index("horizon")
        )

        if baseline.empty:
            continue

        baseline = baseline.rename(
            columns={
                metric_mean_col: "baseline_mean",
                metric_low_col: "baseline_ci_low",
                metric_high_col: "baseline_ci_high",
            }
        )

        df_sub = df_sub.merge(
            baseline,
            left_on="horizon",
            right_index=True,
            how="left",
        )

        df_sub["baseline_norm_mean"] = safe_divide(
            df_sub[metric_mean_col].to_numpy(),
            df_sub["baseline_mean"].to_numpy(),
        )

        # Conservative CI propagation:
        # lower bound uses model lower / baseline upper
        # upper bound uses model upper / baseline lower
        df_sub["baseline_norm_ci_low"] = safe_divide(
            df_sub[metric_low_col].to_numpy(),
            df_sub["baseline_ci_high"].to_numpy(),
        )

        df_sub["baseline_norm_ci_high"] = safe_divide(
            df_sub[metric_high_col].to_numpy(),
            df_sub["baseline_ci_low"].to_numpy(),
        )

        df_sub[
            [
                "baseline_norm_mean",
                "baseline_norm_ci_low",
                "baseline_norm_ci_high",
            ]
        ] = df_sub[
            [
                "baseline_norm_mean",
                "baseline_norm_ci_low",
                "baseline_norm_ci_high",
            ]
        ].replace([np.inf, -np.inf], np.nan)

        df_sub = df_sub.dropna(subset=["baseline_norm_mean"]).copy()

        if not df_sub.empty:
            rows.append(df_sub)

    if not rows:
        raise ValueError(
            f"No valid combinations found with repeat_last baseline for metric '{metric_col}'."
        )

    return pd.concat(rows, ignore_index=True)


def build_best_config_tables(
    results_dict: dict,
    metric_col: str,
):
    """
    For each dataset x method:
      1) compute performance relative to repeat_last for one metric
      2) aggregate performance across horizons for each config
      3) select best config, meaning lowest pooled performance
      4) aggregate win_rate across horizons for that chosen config

    Returns:
      best_perf_df:
        dataset, method, metric, performance, performance_ci_low, performance_ci_high

      best_win_df:
        dataset, method, metric, win_rate, win_rate_ci_low, win_rate_ci_high

      best_configs_df:
        dataset, method, metric, epi, einn, filter, ti
    """
    perf_rows = []
    win_rows = []
    cfg_rows = []

    for dataset, df in results_dict.items():
        if df is None or len(df) == 0:
            continue

        df = standardize_config_cols(df)

        try:
            df_perf = compute_baseline_norm_rows_with_ci_for_metric(
                df_in=df,
                metric_col=metric_col,
                group_cols=group_cols,
            )
        except ValueError as e:
            print(f"Skipping {dataset}, metric={metric_col}: {e}")
            continue

        if not INCLUDE_REPEAT_LAST:
            df_perf = df_perf[df_perf["method"] != BASELINE_METHOD_NAME].copy()
            df = df[df["method"] != BASELINE_METHOD_NAME].copy()

        if df_perf.empty or df.empty:
            print(f"Skipping {dataset}, metric={metric_col}: no methods left after filtering")
            continue

        # ---------------------------------------------
        # Aggregate performance across horizons for each config
        # ---------------------------------------------
        config_perf_rows = []

        for keys, g in df_perf.groupby(["method"] + group_cols, dropna=False):
            method = keys[0]
            cfg = keys[1:]

            pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
                means=g["baseline_norm_mean"].to_numpy(dtype=float),
                ci_lows=g["baseline_norm_ci_low"].to_numpy(dtype=float),
                ci_highs=g["baseline_norm_ci_high"].to_numpy(dtype=float),
                method=HORIZON_AGG_METHOD,
                clip_to_01=False,
            )

            config_perf_rows.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "metric": metric_col,
                    "epi": cfg[0],
                    "einn": cfg[1],
                    "filter": cfg[2],
                    "ti": cfg[3],
                    "performance": pooled_mean,
                    "performance_ci_low": pooled_ci_low,
                    "performance_ci_high": pooled_ci_high,
                }
            )

        config_perf_df = pd.DataFrame(config_perf_rows)

        if config_perf_df.empty:
            print(f"Skipping {dataset}, metric={metric_col}: no valid config-level performance rows")
            continue

        # ---------------------------------------------
        # Best config per method = lowest performance
        # ---------------------------------------------
        best_idx = config_perf_df.groupby("method")["performance"].idxmin()
        best_configs_df = config_perf_df.loc[best_idx].reset_index(drop=True)

        for _, best_row in best_configs_df.iterrows():
            method = best_row["method"]
            epi = best_row["epi"]
            einn = best_row["einn"]
            filt = best_row["filter"]
            ti = best_row["ti"]

            cfg_rows.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "metric": metric_col,
                    "epi": epi,
                    "einn": einn,
                    "filter": filt,
                    "ti": ti,
                }
            )

            perf_rows.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "metric": metric_col,
                    "performance": best_row["performance"],
                    "performance_ci_low": best_row["performance_ci_low"],
                    "performance_ci_high": best_row["performance_ci_high"],
                }
            )

            # -----------------------------------------
            # Aggregate win_rate across horizons for chosen config
            # -----------------------------------------
            g_win = df[
                (df["method"] == method)
                & (df["epi"] == epi)
                & (df["einn"] == einn)
                & (df["filter"] == filt)
                & (df["ti"] == ti)
            ].copy()

            if g_win.empty:
                win_rows.append(
                    {
                        "dataset": dataset,
                        "method": method,
                        "metric": metric_col,
                        "win_rate": np.nan,
                        "win_rate_ci_low": np.nan,
                        "win_rate_ci_high": np.nan,
                    }
                )
                continue

            pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
                means=g_win["win_rate_mean"].to_numpy(dtype=float),
                ci_lows=g_win["win_rate_ci_low"].to_numpy(dtype=float),
                ci_highs=g_win["win_rate_ci_high"].to_numpy(dtype=float),
                method=HORIZON_AGG_METHOD,
                clip_to_01=True,
            )

            win_rows.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "metric": metric_col,
                    "win_rate": pooled_mean,
                    "win_rate_ci_low": pooled_ci_low,
                    "win_rate_ci_high": pooled_ci_high,
                }
            )

    best_perf_df = pd.DataFrame(perf_rows)
    best_win_df = pd.DataFrame(win_rows)
    best_configs_df = pd.DataFrame(cfg_rows)

    return best_perf_df, best_win_df, best_configs_df


# =========================================================
# Plotting
# =========================================================

def plot_grouped_barplot(
    plot_df: pd.DataFrame,
    value_col: str,
    ci_low_col: str,
    ci_high_col: str,
    title: str,
    y_label: str,
    reference_y: float,
    reference_label: str,
    reference_linestyle: str,
    reference_linewidth: float,
    reference_linecolor: str,
    ylim: tuple[float, float],
    figsize: tuple[float, float],
    save_filename: str | None = None,
):
    if plot_df.empty:
        print(f"No data to plot for: {title}")
        return

    datasets_order = list(plot_df["dataset"].drop_duplicates())
    methods_order = list(plot_df["method"].drop_duplicates())

    x = np.arange(len(datasets_order))
    n_methods = len(methods_order)

    if n_methods == 0:
        print(f"No methods to plot for: {title}")
        return

    bar_width = BAR_GROUP_WIDTH / n_methods

    fig, ax = plt.subplots(figsize=figsize)

    for i, method in enumerate(methods_order):
        vals = []
        yerr_low = []
        yerr_high = []

        for dataset in datasets_order:
            sub = plot_df[
                (plot_df["dataset"] == dataset)
                & (plot_df["method"] == method)
            ]

            if sub.empty:
                vals.append(np.nan)
                yerr_low.append(np.nan)
                yerr_high.append(np.nan)
            else:
                row = sub.iloc[0]

                val = row[value_col]
                ci_low = row[ci_low_col]
                ci_high = row[ci_high_col]

                low_err, high_err = make_yerr(val, ci_low, ci_high)

                vals.append(val)
                yerr_low.append(low_err)
                yerr_high.append(high_err)

        vals = np.asarray(vals, dtype=float)
        yerr = np.vstack(
            [
                np.asarray(yerr_low, dtype=float),
                np.asarray(yerr_high, dtype=float),
            ]
        )

        offsets = x - (BAR_GROUP_WIDTH / 2) + (i + 0.5) * bar_width
        color = get_method_color(method, i)

        ax.bar(
            offsets,
            vals,
            width=bar_width,
            label=method,
            alpha=BAR_ALPHA,
            color=color,
            yerr=yerr if SHOW_ERROR_BARS else None,
            ecolor=ERROR_BAR_COLOR,
            error_kw={
                "elinewidth": ERROR_BAR_LINEWIDTH,
                "capsize": ERROR_BAR_CAPSIZE,
                "capthick": ERROR_BAR_CAPTHICK,
            }
            if SHOW_ERROR_BARS
            else None,
        )

    ax.axhline(
        reference_y,
        linestyle=reference_linestyle,
        linewidth=reference_linewidth,
        color=reference_linecolor,
        label=reference_label if not SHOW_LEGEND else None,
    )

    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    ax.set_xlabel(X_LABEL, fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel(y_label, fontsize=AXIS_LABEL_FONTSIZE)

    ax.set_xticks(x)
    ax.set_xticklabels(
        datasets_order,
        rotation=XTICK_ROTATION,
        ha=XTICK_HA,
        fontsize=XTICK_FONTSIZE,
    )

    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)
    ax.set_ylim(ylim)

    if SHOW_GRID:
        ax.grid(True, axis=GRID_AXIS, alpha=GRID_ALPHA)

    if SHOW_LEGEND:
        ax.legend(
            loc=LEGEND_LOC,
            bbox_to_anchor=LEGEND_BBOX_TO_ANCHOR,
            fontsize=LEGEND_FONTSIZE,
            frameon=LEGEND_FRAMEON,
            title=LEGEND_TITLE,
        )

    plt.tight_layout()

    if SAVE_FIGURES and save_filename is not None:
        SAVE_DIR.mkdir(parents=True, exist_ok=True)

        save_path = SAVE_DIR / save_filename
        plt.savefig(save_path, dpi=SAVE_DPI, bbox_inches=SAVE_BBOX_INCHES)

        print(f"Saved: {save_path}")

    if SHOW_FIGURES:
        plt.show()

    if CLOSE_FIGURES:
        plt.close(fig)


# =========================================================
# Build best-config tables and plot for each metric
# =========================================================

best_perf_dfs = {}
best_win_dfs = {}
best_configs_dfs = {}

for metric_col in metric_cols:
    metric_display = get_metric_label(metric_col)

    best_perf_df, best_win_df, best_configs_df = build_best_config_tables(
        results_dict=RESULTS_DICT,
        metric_col=metric_col,
    )

    best_perf_dfs[metric_col] = best_perf_df
    best_win_dfs[metric_col] = best_win_df
    best_configs_dfs[metric_col] = best_configs_df

    plot_grouped_barplot(
        plot_df=best_win_df,
        value_col="win_rate",
        ci_low_col="win_rate_ci_low",
        ci_high_col="win_rate_ci_high",
        title=WIN_TITLE_TEMPLATE.format(metric_label=metric_display),
        y_label=WIN_Y_LABEL,
        reference_y=WIN_REFERENCE_Y,
        reference_label=WIN_REFERENCE_LABEL,
        reference_linestyle=WIN_REFERENCE_LINESTYLE,
        reference_linewidth=WIN_REFERENCE_LINEWIDTH,
        reference_linecolor=WIN_REFERENCE_LINECOLOR,
        ylim=(WIN_YMIN, WIN_YMAX),
        figsize=FIGSIZE_WIN,
        save_filename=WIN_FILENAME_TEMPLATE.format(metric=metric_col),
    )

    plot_grouped_barplot(
        plot_df=best_perf_df,
        value_col="performance",
        ci_low_col="performance_ci_low",
        ci_high_col="performance_ci_high",
        title=PERF_TITLE_TEMPLATE.format(metric_label=metric_display),
        y_label=PERF_Y_LABEL,
        reference_y=PERF_REFERENCE_Y,
        reference_label=PERF_REFERENCE_LABEL,
        reference_linestyle=PERF_REFERENCE_LINESTYLE,
        reference_linewidth=PERF_REFERENCE_LINEWIDTH,
        reference_linecolor=PERF_REFERENCE_LINECOLOR,
        ylim=(PERF_YMIN, PERF_YMAX),
        figsize=FIGSIZE_PERF,
        save_filename=PERF_FILENAME_TEMPLATE.format(metric=metric_col),
    )

In [ ]:
# =========================================================
# Percent RMSE vs RMSE filtered comparison by method
# =========================================================
# Requires:
#   - build_best_config_tables(...)
#   - aggregate_estimates_with_ci(...)
#   - safe_divide(...)
#   - get_method_color(...)
#   - plotting style constants already defined above
#
# Y-axis interpretation:
#   0%    = same relative error for RMSE and RMSE filtered
#   +25%  = RMSE is 25% worse than RMSE filtered
#   -10%  = RMSE is 10% better than RMSE filtered
#
# Formula:
#   100 * (rmse_error_relative_to_baseline /
#          rmse_filtered_error_relative_to_baseline - 1)
# =========================================================

PLOT_RMSE_VS_FILTERED_PERCENT_COMPARISON = True

FILENAME_RMSE_VS_FILTERED_PERCENT_COMPARISON_PLOT = (
    "rmse_vs_rmse_filtered_percent_by_method.png"
)

TITLE_RMSE_VS_FILTERED_PERCENT_COMPARISON = (
    "Performance w/out zeros and outliers"
)

Y_LABEL_RMSE_VS_FILTERED_PERCENT_COMPARISON = (
    "Additional error using filtered RMSE"
)

REFERENCE_PERCENT_Y = 0.0

# Optional explicit method order for color assignment.
# Leave as None to use first-seen method order from the dataset-level table.
METHOD_COLOR_ORDER_FOR_RMSE_FILTERED_PLOT = None


def build_method_color_index_map(
    *dfs: pd.DataFrame,
    method_col: str = "method",
    explicit_order: list | None = None,
) -> dict:
    """
    Build a stable method -> color index mapping.

    This keeps method colors consistent with the original method ordering
    even if the final plot is sorted differently.
    """
    if explicit_order is not None:
        return {
            method: idx
            for idx, method in enumerate(explicit_order)
        }

    method_series = []

    for df in dfs:
        if df is not None and not df.empty and method_col in df.columns:
            method_series.append(df[method_col])

    if not method_series:
        return {}

    method_order = (
        pd.concat(method_series, ignore_index=True)
        .dropna()
        .drop_duplicates()
        .to_list()
    )

    return {
        method: idx
        for idx, method in enumerate(method_order)
    }


def build_rmse_vs_filtered_dataset_table(results_dict: dict):
    """
    Build per-dataset RMSE vs RMSE filtered comparison table.

    For each dataset x method:
      1) compute best-patch performance for RMSE
      2) compute best-patch performance for RMSE filtered
      3) compare RMSE / RMSE filtered

    Returns columns including:
      dataset, method,
      rmse_performance,
      rmse_filtered_performance,
      rmse_over_filtered,
      rmse_over_filtered_ci_low,
      rmse_over_filtered_ci_high
    """
    rmse_perf_df, _, rmse_cfg_df = build_best_config_tables(
        results_dict=results_dict,
        metric_col="rmse",
    )

    rmse_filtered_perf_df, _, rmse_filtered_cfg_df = build_best_config_tables(
        results_dict=results_dict,
        metric_col="rmse_filtered",
    )

    if rmse_perf_df.empty:
        raise ValueError("No RMSE performance rows found.")

    if rmse_filtered_perf_df.empty:
        raise ValueError("No RMSE filtered performance rows found.")

    rmse_perf_df = rmse_perf_df.rename(
        columns={
            "performance": "rmse_performance",
            "performance_ci_low": "rmse_performance_ci_low",
            "performance_ci_high": "rmse_performance_ci_high",
        }
    )

    rmse_filtered_perf_df = rmse_filtered_perf_df.rename(
        columns={
            "performance": "rmse_filtered_performance",
            "performance_ci_low": "rmse_filtered_performance_ci_low",
            "performance_ci_high": "rmse_filtered_performance_ci_high",
        }
    )

    rmse_cfg_df = rmse_cfg_df.rename(
        columns={
            "epi": "rmse_best_epi",
            "einn": "rmse_best_einn",
            "filter": "rmse_best_filter",
            "ti": "rmse_best_ti",
        }
    )

    rmse_filtered_cfg_df = rmse_filtered_cfg_df.rename(
        columns={
            "epi": "rmse_filtered_best_epi",
            "einn": "rmse_filtered_best_einn",
            "filter": "rmse_filtered_best_filter",
            "ti": "rmse_filtered_best_ti",
        }
    )

    compare_df = rmse_perf_df.merge(
        rmse_filtered_perf_df[
            [
                "dataset",
                "method",
                "rmse_filtered_performance",
                "rmse_filtered_performance_ci_low",
                "rmse_filtered_performance_ci_high",
            ]
        ],
        on=["dataset", "method"],
        how="inner",
    )

    # Conservative ratio CI propagation:
    # lower = rmse low / rmse_filtered high
    # upper = rmse high / rmse_filtered low
    compare_df["rmse_over_filtered"] = safe_divide(
        compare_df["rmse_performance"].to_numpy(dtype=float),
        compare_df["rmse_filtered_performance"].to_numpy(dtype=float),
    )

    compare_df["rmse_over_filtered_ci_low"] = safe_divide(
        compare_df["rmse_performance_ci_low"].to_numpy(dtype=float),
        compare_df["rmse_filtered_performance_ci_high"].to_numpy(dtype=float),
    )

    compare_df["rmse_over_filtered_ci_high"] = safe_divide(
        compare_df["rmse_performance_ci_high"].to_numpy(dtype=float),
        compare_df["rmse_filtered_performance_ci_low"].to_numpy(dtype=float),
    )

    compare_df["percent_rmse_relative_to_filtered"] = (
        100.0 * (compare_df["rmse_over_filtered"] - 1.0)
    )

    compare_df["percent_ci_low"] = (
        100.0 * (compare_df["rmse_over_filtered_ci_low"] - 1.0)
    )

    compare_df["percent_ci_high"] = (
        100.0 * (compare_df["rmse_over_filtered_ci_high"] - 1.0)
    )

    compare_df = compare_df.merge(
        rmse_cfg_df[
            [
                "dataset",
                "method",
                "rmse_best_epi",
                "rmse_best_einn",
                "rmse_best_filter",
                "rmse_best_ti",
            ]
        ],
        on=["dataset", "method"],
        how="left",
    )

    compare_df = compare_df.merge(
        rmse_filtered_cfg_df[
            [
                "dataset",
                "method",
                "rmse_filtered_best_epi",
                "rmse_filtered_best_einn",
                "rmse_filtered_best_filter",
                "rmse_filtered_best_ti",
            ]
        ],
        on=["dataset", "method"],
        how="left",
    )

    return compare_df


def build_rmse_vs_filtered_method_summary(compare_df: pd.DataFrame):
    """
    Pool RMSE / RMSE filtered ratios across datasets for each method.

    Returns:
      method
      n_datasets
      pooled_rmse_over_filtered
      pooled_rmse_over_filtered_ci_low
      pooled_rmse_over_filtered_ci_high
      pooled_percent_rmse_relative_to_filtered
      pooled_percent_ci_low
      pooled_percent_ci_high
    """
    if compare_df.empty:
        return pd.DataFrame()

    rows = []

    for method, g in compare_df.groupby("method", dropna=False):
        pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
            means=g["rmse_over_filtered"].to_numpy(dtype=float),
            ci_lows=g["rmse_over_filtered_ci_low"].to_numpy(dtype=float),
            ci_highs=g["rmse_over_filtered_ci_high"].to_numpy(dtype=float),
            method=HORIZON_AGG_METHOD,
            clip_to_01=False,
        )

        rows.append(
            {
                "method": method,
                "n_datasets": g["dataset"].nunique(),
                "pooled_rmse_over_filtered": pooled_mean,
                "pooled_rmse_over_filtered_ci_low": pooled_ci_low,
                "pooled_rmse_over_filtered_ci_high": pooled_ci_high,
                "pooled_percent_rmse_relative_to_filtered": (
                    100.0 * (pooled_mean - 1.0)
                ),
                "pooled_percent_ci_low": (
                    100.0 * (pooled_ci_low - 1.0)
                ),
                "pooled_percent_ci_high": (
                    100.0 * (pooled_ci_high - 1.0)
                ),
            }
        )

    return pd.DataFrame(rows).sort_values("method").reset_index(drop=True)


def build_rmse_vs_filtered_overall_stats(compare_df: pd.DataFrame):
    """
    Optional extra stats, similar in spirit to the outbreak summaries.
    """
    if compare_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    method_stats_df = (
        compare_df
        .groupby("method", dropna=False)
        .agg(
            n_datasets=("dataset", "nunique"),
            mean_percent_difference=("percent_rmse_relative_to_filtered", "mean"),
            median_percent_difference=("percent_rmse_relative_to_filtered", "median"),
            min_percent_difference=("percent_rmse_relative_to_filtered", "min"),
            max_percent_difference=("percent_rmse_relative_to_filtered", "max"),
        )
        .reset_index()
        .sort_values("method")
    )

    overall_stats_df = pd.DataFrame(
        [
            {
                "n_rows": compare_df["percent_rmse_relative_to_filtered"].count(),
                "mean_percent_difference": (
                    compare_df["percent_rmse_relative_to_filtered"].mean()
                ),
                "median_percent_difference": (
                    compare_df["percent_rmse_relative_to_filtered"].median()
                ),
                "min_percent_difference": (
                    compare_df["percent_rmse_relative_to_filtered"].min()
                ),
                "max_percent_difference": (
                    compare_df["percent_rmse_relative_to_filtered"].max()
                ),
            }
        ]
    )

    return method_stats_df, overall_stats_df


def plot_method_percent_rmse_vs_filtered(
    summary_df: pd.DataFrame,
    method_to_color_idx: dict | None = None,
    save_filename: str | None = None,
):
    """
    Bar plot of percent RMSE vs RMSE filtered difference by method.

    Uses:
      pooled_rmse_over_filtered
      pooled_rmse_over_filtered_ci_low
      pooled_rmse_over_filtered_ci_high

    Converts those ratios to percentages:
      percent = 100 * (ratio - 1)

    Positive values mean RMSE is worse than RMSE filtered.
    Negative values mean RMSE is better than RMSE filtered.

    Colors are assigned by method_to_color_idx so they remain consistent
    with other plots even after sorting.
    """
    if summary_df.empty:
        print("No RMSE-vs-filtered summary to plot.")
        return

    needed_cols = [
        "method",
        "pooled_rmse_over_filtered",
        "pooled_rmse_over_filtered_ci_low",
        "pooled_rmse_over_filtered_ci_high",
    ]

    missing = [c for c in needed_cols if c not in summary_df.columns]
    if missing:
        raise ValueError(f"Missing expected columns in summary_df: {missing}")

    if method_to_color_idx is None:
        method_to_color_idx = build_method_color_index_map(summary_df)

    dfp = summary_df.copy()

    dfp["percent_rmse_relative_to_filtered"] = (
        100.0 * (dfp["pooled_rmse_over_filtered"] - 1.0)
    )

    dfp["percent_ci_low"] = (
        100.0 * (dfp["pooled_rmse_over_filtered_ci_low"] - 1.0)
    )

    dfp["percent_ci_high"] = (
        100.0 * (dfp["pooled_rmse_over_filtered_ci_high"] - 1.0)
    )

    # Put the most RMSE-worse methods first.
    # Color assignment remains stable because it uses method_to_color_idx.
    dfp = dfp.sort_values(
        "percent_rmse_relative_to_filtered",
        ascending=False,
    ).reset_index(drop=True)

    methods = dfp["method"].astype(str).to_list()
    vals = dfp["percent_rmse_relative_to_filtered"].to_numpy(dtype=float)

    yerr_low = vals - dfp["percent_ci_low"].to_numpy(dtype=float)
    yerr_high = dfp["percent_ci_high"].to_numpy(dtype=float) - vals

    yerr_low = np.where(np.isfinite(yerr_low), yerr_low, np.nan)
    yerr_high = np.where(np.isfinite(yerr_high), yerr_high, np.nan)

    yerr = np.vstack([yerr_low, yerr_high])

    x = np.arange(len(methods))

    fig, ax = plt.subplots(figsize=(8, 8))

    colors = [
        get_method_color(
            method,
            method_to_color_idx.get(method, i),
        )
        for i, method in enumerate(methods)
    ]

    ax.bar(
        x,
        vals,
        alpha=BAR_ALPHA,
        color=colors,
        yerr=yerr if SHOW_ERROR_BARS else None,
        ecolor=ERROR_BAR_COLOR,
        error_kw={
            "elinewidth": ERROR_BAR_LINEWIDTH,
            "capsize": ERROR_BAR_CAPSIZE,
            "capthick": ERROR_BAR_CAPTHICK,
        } if SHOW_ERROR_BARS else None,
    )

    ax.axhline(
        REFERENCE_PERCENT_Y,
        linestyle=PERF_REFERENCE_LINESTYLE,
        linewidth=PERF_REFERENCE_LINEWIDTH,
        color=PERF_REFERENCE_LINECOLOR,
    )

    ax.set_title(
        TITLE_RMSE_VS_FILTERED_PERCENT_COMPARISON,
        fontsize=TITLE_FONTSIZE,
    )

    ax.set_ylabel(
        Y_LABEL_RMSE_VS_FILTERED_PERCENT_COMPARISON,
        fontsize=AXIS_LABEL_FONTSIZE,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        methods,
        rotation=45,
        ha="right",
        fontsize=XTICK_FONTSIZE,
    )

    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)

    # Format y-axis as percentages.
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda y, _: f"{y:.0f}%")
    )

    if SHOW_GRID:
        ax.grid(True, axis=GRID_AXIS, alpha=GRID_ALPHA)

    plt.tight_layout()

    if SAVE_FIGURES and save_filename is not None:
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        save_path = SAVE_DIR / save_filename
        plt.savefig(
            save_path,
            dpi=SAVE_DPI,
            bbox_inches=SAVE_BBOX_INCHES,
        )
        print(f"Saved: {save_path}")

    if SHOW_FIGURES:
        plt.show()

    if CLOSE_FIGURES:
        plt.close(fig)


# =========================================================
# Build summary tables and plot
# =========================================================

rmse_vs_filtered_dataset_df = build_rmse_vs_filtered_dataset_table(RESULTS_DICT)

# Stable method -> color index mapping.
# This is built before the summary gets sorted for plotting.
rmse_vs_filtered_method_to_color_idx = build_method_color_index_map(
    rmse_vs_filtered_dataset_df,
    explicit_order=METHOD_COLOR_ORDER_FOR_RMSE_FILTERED_PLOT,
)

rmse_vs_filtered_method_summary_df = build_rmse_vs_filtered_method_summary(
    rmse_vs_filtered_dataset_df
)

rmse_vs_filtered_method_stats_df, rmse_vs_filtered_overall_stats_df = (
    build_rmse_vs_filtered_overall_stats(rmse_vs_filtered_dataset_df)
)

print("\nPer-dataset RMSE vs RMSE filtered comparison:")
display(rmse_vs_filtered_dataset_df)

print("\nPooled RMSE vs RMSE filtered summary by method:")
display(rmse_vs_filtered_method_summary_df)

print("\nMethod-level RMSE vs RMSE filtered stats:")
display(rmse_vs_filtered_method_stats_df)

print("\nOverall RMSE vs RMSE filtered stats:")
display(rmse_vs_filtered_overall_stats_df)

print(
    rmse_vs_filtered_method_stats_df["median_percent_difference"].median()
)

if PLOT_RMSE_VS_FILTERED_PERCENT_COMPARISON:
    plot_method_percent_rmse_vs_filtered(
        rmse_vs_filtered_method_summary_df,
        method_to_color_idx=rmse_vs_filtered_method_to_color_idx,
        save_filename=FILENAME_RMSE_VS_FILTERED_PERCENT_COMPARISON_PLOT,
    )

In [ ]:
# =========================================================
# Average increase summaries across datasets, for all metrics
# =========================================================

ORIGINAL_CONFIG = {
    "epi": False,
    "einn": False,
    "filter": False,
    "ti": False,
}

METRIC_LABELS = {
    # Standard point metrics
    "mse": "MSE",
    "mae": "MAE",
    "rmse": "RMSE",
    "medae": "medAE",
    "medse": "medSE",

    # Filtered point metrics
    "mse_filtered": "MSE filtered",
    "mae_filtered": "MAE filtered",
    "rmse_filtered": "RMSE filtered",
}


def get_metric_label(metric: str) -> str:
    return METRIC_LABELS.get(metric, metric)


def config_equals(df: pd.DataFrame, config: dict) -> pd.Series:
    mask = pd.Series(True, index=df.index)
    for col, val in config.items():
        mask &= df[col].eq(val)
    return mask


def build_all_config_perf_and_win_tables_for_metric(
    results_dict: dict,
    metric_col: str,
):
    """
    Builds config-level performance and win-rate tables for every
    dataset x method x config for one metric.

    Performance is error relative to repeat_last baseline.
    Lower performance is better.

    Win rate is aggregated across horizons.
    Higher win_rate is better.
    """
    all_perf_rows = []
    all_win_rows = []

    for dataset, df in results_dict.items():
        if df is None or len(df) == 0:
            continue

        df = standardize_config_cols(df)

        # -------------------------------------------------
        # Performance relative to baseline for this metric
        # -------------------------------------------------
        try:
            df_perf = compute_baseline_norm_rows_with_ci_for_metric(
                df_in=df,
                metric_col=metric_col,
                group_cols=group_cols,
            )
        except ValueError as e:
            print(f"Skipping {dataset}, metric={metric_col}: {e}")
            continue

        if not INCLUDE_REPEAT_LAST:
            df_perf = df_perf[df_perf["method"] != BASELINE_METHOD_NAME].copy()
            df = df[df["method"] != BASELINE_METHOD_NAME].copy()

        if df_perf.empty or df.empty:
            print(f"Skipping {dataset}, metric={metric_col}: no methods left after filtering")
            continue

        # -------------------------------------------------
        # Aggregate performance across horizons per config
        # -------------------------------------------------
        for keys, g in df_perf.groupby(["method"] + group_cols, dropna=False):
            method = keys[0]
            cfg = keys[1:]

            pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
                means=g["baseline_norm_mean"].to_numpy(dtype=float),
                ci_lows=g["baseline_norm_ci_low"].to_numpy(dtype=float),
                ci_highs=g["baseline_norm_ci_high"].to_numpy(dtype=float),
                method=HORIZON_AGG_METHOD,
                clip_to_01=False,
            )

            all_perf_rows.append(
                {
                    "metric": metric_col,
                    "metric_label": get_metric_label(metric_col),
                    "dataset": dataset,
                    "method": method,
                    "epi": cfg[0],
                    "einn": cfg[1],
                    "filter": cfg[2],
                    "ti": cfg[3],
                    "performance": pooled_mean,
                    "performance_ci_low": pooled_ci_low,
                    "performance_ci_high": pooled_ci_high,
                }
            )

        # -------------------------------------------------
        # Aggregate win rate across horizons per config
        # -------------------------------------------------
        for keys, g in df.groupby(["method"] + group_cols, dropna=False):
            method = keys[0]
            cfg = keys[1:]

            pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
                means=g["win_rate_mean"].to_numpy(dtype=float),
                ci_lows=g["win_rate_ci_low"].to_numpy(dtype=float),
                ci_highs=g["win_rate_ci_high"].to_numpy(dtype=float),
                method=HORIZON_AGG_METHOD,
                clip_to_01=True,
            )

            all_win_rows.append(
                {
                    "metric": metric_col,
                    "metric_label": get_metric_label(metric_col),
                    "dataset": dataset,
                    "method": method,
                    "epi": cfg[0],
                    "einn": cfg[1],
                    "filter": cfg[2],
                    "ti": cfg[3],
                    "win_rate": pooled_mean,
                    "win_rate_ci_low": pooled_ci_low,
                    "win_rate_ci_high": pooled_ci_high,
                }
            )

    all_perf_df = pd.DataFrame(all_perf_rows)
    all_win_df = pd.DataFrame(all_win_rows)

    return all_perf_df, all_win_df


def build_average_increase_summary_for_metric(
    results_dict: dict,
    metric_col: str,
):
    all_perf_df, all_win_df = build_all_config_perf_and_win_tables_for_metric(
        results_dict=results_dict,
        metric_col=metric_col,
    )

    if all_perf_df.empty or all_win_df.empty:
        empty_method_df = pd.DataFrame()
        empty_dataset_df = pd.DataFrame()
        return empty_method_df, empty_dataset_df, all_perf_df, all_win_df

    # -----------------------------------------------------
    # Original performance and best patched performance
    # -----------------------------------------------------
    original_perf_df = all_perf_df[config_equals(all_perf_df, ORIGINAL_CONFIG)].copy()
    original_perf_df = original_perf_df.rename(
        columns={
            "performance": "original_performance",
            "performance_ci_low": "original_performance_ci_low",
            "performance_ci_high": "original_performance_ci_high",
        }
    )

    best_perf_idx = all_perf_df.groupby(["metric", "dataset", "method"])["performance"].idxmin()
    best_perf_df_for_ratio = all_perf_df.loc[best_perf_idx].copy()
    best_perf_df_for_ratio = best_perf_df_for_ratio.rename(
        columns={
            "performance": "best_patched_performance",
            "performance_ci_low": "best_patched_performance_ci_low",
            "performance_ci_high": "best_patched_performance_ci_high",
            "epi": "best_perf_epi",
            "einn": "best_perf_einn",
            "filter": "best_perf_filter",
            "ti": "best_perf_ti",
        }
    )

    perf_ratio_df = original_perf_df.merge(
        best_perf_df_for_ratio[
            [
                "metric",
                "metric_label",
                "dataset",
                "method",
                "best_patched_performance",
                "best_patched_performance_ci_low",
                "best_patched_performance_ci_high",
                "best_perf_epi",
                "best_perf_einn",
                "best_perf_filter",
                "best_perf_ti",
            ]
        ],
        on=["metric", "metric_label", "dataset", "method"],
        how="inner",
    )

    perf_ratio_df["performance_increase"] = safe_divide(
        perf_ratio_df["original_performance"].to_numpy(dtype=float),
        perf_ratio_df["best_patched_performance"].to_numpy(dtype=float),
    )

    perf_ratio_df["performance_percent_increase"] = (
        perf_ratio_df["performance_increase"] - 1.0
    ) * 100.0

    # -----------------------------------------------------
    # Original win rate and best win rate
    # -----------------------------------------------------
    original_win_df = all_win_df[config_equals(all_win_df, ORIGINAL_CONFIG)].copy()
    original_win_df = original_win_df.rename(
        columns={
            "win_rate": "original_win_rate",
            "win_rate_ci_low": "original_win_rate_ci_low",
            "win_rate_ci_high": "original_win_rate_ci_high",
        }
    )

    best_win_idx = all_win_df.groupby(["metric", "dataset", "method"])["win_rate"].idxmax()
    best_win_df_for_ratio = all_win_df.loc[best_win_idx].copy()
    best_win_df_for_ratio = best_win_df_for_ratio.rename(
        columns={
            "win_rate": "best_win_rate",
            "win_rate_ci_low": "best_win_rate_ci_low",
            "win_rate_ci_high": "best_win_rate_ci_high",
            "epi": "best_win_epi",
            "einn": "best_win_einn",
            "filter": "best_win_filter",
            "ti": "best_win_ti",
        }
    )

    win_ratio_df = original_win_df.merge(
        best_win_df_for_ratio[
            [
                "metric",
                "metric_label",
                "dataset",
                "method",
                "best_win_rate",
                "best_win_rate_ci_low",
                "best_win_rate_ci_high",
                "best_win_epi",
                "best_win_einn",
                "best_win_filter",
                "best_win_ti",
            ]
        ],
        on=["metric", "metric_label", "dataset", "method"],
        how="inner",
    )

    win_ratio_df["win_rate_increase"] = safe_divide(
        win_ratio_df["best_win_rate"].to_numpy(dtype=float),
        win_ratio_df["original_win_rate"].to_numpy(dtype=float),
    )

    win_ratio_df["win_rate_percent_increase"] = (
        win_ratio_df["win_rate_increase"] - 1.0
    ) * 100.0

    # -----------------------------------------------------
    # Per-dataset combined table
    # -----------------------------------------------------
    per_dataset_increase_df = perf_ratio_df.merge(
        win_ratio_df,
        on=["metric", "metric_label", "dataset", "method"],
        how="outer",
        suffixes=("_perf", "_win"),
    )

    # -----------------------------------------------------
    # Mean across datasets per metric x method
    # -----------------------------------------------------
    method_mean_increase_df = (
        per_dataset_increase_df
        .groupby(["metric", "metric_label", "method"], dropna=False)
        .agg(
            n_datasets_performance=("performance_increase", "count"),
            mean_performance_increase=("performance_increase", "mean"),
            mean_performance_percent_increase=("performance_percent_increase", "mean"),
            n_datasets_win_rate=("win_rate_increase", "count"),
            mean_win_rate_increase=("win_rate_increase", "mean"),
            mean_win_rate_percent_increase=("win_rate_percent_increase", "mean"),
        )
        .reset_index()
        .sort_values(["metric", "method"])
    )

    return (
        method_mean_increase_df,
        per_dataset_increase_df,
        all_perf_df,
        all_win_df,
    )


def build_average_increase_summary_all_metrics(results_dict: dict):
    method_mean_rows = []
    per_dataset_rows = []
    all_perf_rows = []
    all_win_rows = []

    for metric_col in metric_cols:
        (
            method_mean_increase_df,
            per_dataset_increase_df,
            all_perf_df,
            all_win_df,
        ) = build_average_increase_summary_for_metric(
            results_dict=results_dict,
            metric_col=metric_col,
        )

        if not method_mean_increase_df.empty:
            method_mean_rows.append(method_mean_increase_df)

        if not per_dataset_increase_df.empty:
            per_dataset_rows.append(per_dataset_increase_df)

        if not all_perf_df.empty:
            all_perf_rows.append(all_perf_df)

        if not all_win_df.empty:
            all_win_rows.append(all_win_df)

    method_mean_increase_all_metrics_df = (
        pd.concat(method_mean_rows, ignore_index=True)
        if method_mean_rows
        else pd.DataFrame()
    )

    per_dataset_increase_all_metrics_df = (
        pd.concat(per_dataset_rows, ignore_index=True)
        if per_dataset_rows
        else pd.DataFrame()
    )

    all_config_perf_all_metrics_df = (
        pd.concat(all_perf_rows, ignore_index=True)
        if all_perf_rows
        else pd.DataFrame()
    )

    all_config_win_all_metrics_df = (
        pd.concat(all_win_rows, ignore_index=True)
        if all_win_rows
        else pd.DataFrame()
    )

    return (
        method_mean_increase_all_metrics_df,
        per_dataset_increase_all_metrics_df,
        all_config_perf_all_metrics_df,
        all_config_win_all_metrics_df,
    )


(
    method_mean_increase_df,
    per_dataset_increase_df,
    all_config_perf_df,
    all_config_win_df,
) = build_average_increase_summary_all_metrics(RESULTS_DICT)


print("\nMean increase across datasets by metric and method:")
display(method_mean_increase_df)

print("\nPer-dataset increase details by metric:")
display(per_dataset_increase_df)

print("\nMedian mean percent increases by metric:")
metric_medians_df = (
    method_mean_increase_df
    .groupby(["metric", "metric_label"], dropna=False)
    .agg(
        median_mean_performance_percent_increase=(
            "mean_performance_percent_increase",
            "median",
        ),
        median_mean_win_rate_percent_increase=(
            "mean_win_rate_percent_increase",
            "median",
        ),
    )
    .reset_index()
    .sort_values("metric")
)

display(metric_medians_df)

print("\nOverall medians across all metric-method rows:")
print(
    method_mean_increase_df["mean_performance_percent_increase"].median(),
    method_mean_increase_df["mean_win_rate_percent_increase"].median(),
)

### AVG ERROR

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.transforms import blended_transform_factory

# =========================================================
# Editable parameters
# =========================================================

metric_cols = [
    #"mse",
    #"mae",
    "rmse",
    #"mse_filtered",
    #"rmse_filtered",
    #"mae_filtered",
    #"medse",
    #"medae",
]

group_cols = ["epi", "einn", "filter", "ti"]
CONFIG_COLS = ["epi", "einn", "filter", "ti"]

BASELINE_METHOD_NAME = "repeat_last"
INCLUDE_REPEAT_LAST = False

# How to aggregate across horizons using existing CIs
# Options:
#   "inverse_variance"
#   "simple_mean"
HORIZON_AGG_METHOD = "inverse_variance"

# ---------------------------------------------------------
# Figure / layout
# ---------------------------------------------------------
FIGSIZE = (12, 6)
SHARE_Y = True

BAR_GROUP_WIDTH = 0.8
BAR_ALPHA = 0.9

SHOW_GRID = False
GRID_ALPHA = 0.25
GRID_AXIS = "y"

# Set to None for automatic limits
AVG_ERROR_YMIN = None
AVG_ERROR_YMAX = None

# Overall-average reference line
SHOW_OVERALL_AVERAGE_LINE = True
OVERALL_AVERAGE_LINESTYLE = "--"
OVERALL_AVERAGE_LINEWIDTH = 1.5
OVERALL_AVERAGE_LINECOLOR = "red"
OVERALL_AVERAGE_LABEL = "Overall average"

# ---------------------------------------------------------
# Error bars / CI style
# ---------------------------------------------------------
SHOW_ERROR_BARS = True
ERROR_BAR_COLOR = "black"
ERROR_BAR_LINEWIDTH = 1.2
ERROR_BAR_CAPSIZE = 4
ERROR_BAR_CAPTHICK = 1.2

# ---------------------------------------------------------
# Fonts
# ---------------------------------------------------------
TITLE_FONTSIZE = 24
AXIS_LABEL_FONTSIZE = 22
XTICK_FONTSIZE = 22
YTICK_FONTSIZE = 18

# ---------------------------------------------------------
# Text
# ---------------------------------------------------------
TITLE_ALL = "Non-outbreak periods"
TITLE_CONSENSUS = "Outbreak periods"
X_LABEL = ""
Y_LABEL = "Average error"

# ---------------------------------------------------------
# X tick / labels
# ---------------------------------------------------------
XTICK_ROTATION = 45
XTICK_HA = "right"

# ---------------------------------------------------------
# Legend
# Removed
# ---------------------------------------------------------
SHOW_LEGEND = False

# ---------------------------------------------------------
# Method colors
# Use shared 20-color fallback palette
# ---------------------------------------------------------
METHOD_COLORS = {
    # "AGCRN": "tab:blue",
    # "DCRNN": "tab:orange",
}

FALLBACK_COLOR_CYCLE = list(plt.get_cmap("tab20").colors)

# ---------------------------------------------------------
# Save / show
# ---------------------------------------------------------
SAVE_FIGURE = True
SAVE_DIR = Path(".")
SAVE_DPI = 300
SAVE_BBOX_INCHES = "tight"
FILENAME = "best_patch_avg_error_non_outbreak_vs_outbreak.png"

SHOW_FIGURE = True
CLOSE_FIGURE = True


# =========================================================
# Helpers
# =========================================================

def normalize_config_value(x):
    if pd.isna(x):
        return x
    if isinstance(x, str):
        if x == "True":
            return True
        if x == "False":
            return False
    return x


def standardize_config_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in CONFIG_COLS:
        if col in df.columns:
            df[col] = df[col].map(normalize_config_value)
    return df


def safe_divide(num, den):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)
    out = np.full(np.broadcast(num, den).shape, np.nan, dtype=float)
    valid = np.isfinite(num) & np.isfinite(den) & (den != 0)
    out[valid] = num[valid] / den[valid]
    return out


def ci_to_se(ci_low, ci_high):
    if not (np.isfinite(ci_low) and np.isfinite(ci_high)):
        return np.nan
    width = ci_high - ci_low
    if width <= 0:
        return np.nan
    return width / (2 * 1.96)


def aggregate_estimates_with_ci(
    means: np.ndarray,
    ci_lows: np.ndarray,
    ci_highs: np.ndarray,
    method: str = "inverse_variance",
):
    """
    Aggregate horizon-level estimates with existing CIs.

    Returns:
      pooled_mean, pooled_ci_low, pooled_ci_high
    """
    means = np.asarray(means, dtype=float)
    ci_lows = np.asarray(ci_lows, dtype=float)
    ci_highs = np.asarray(ci_highs, dtype=float)

    valid = np.isfinite(means) & np.isfinite(ci_lows) & np.isfinite(ci_highs)
    means = means[valid]
    ci_lows = ci_lows[valid]
    ci_highs = ci_highs[valid]

    if len(means) == 0:
        return np.nan, np.nan, np.nan

    if len(means) == 1:
        return means[0], ci_lows[0], ci_highs[0]

    if method == "simple_mean":
        return means.mean(), ci_lows.mean(), ci_highs.mean()

    if method == "inverse_variance":
        ses = np.array([ci_to_se(lo, hi) for lo, hi in zip(ci_lows, ci_highs)], dtype=float)
        valid_se = np.isfinite(ses) & (ses > 0)

        if not valid_se.any():
            return means.mean(), ci_lows.mean(), ci_highs.mean()

        means = means[valid_se]
        ses = ses[valid_se]

        weights = 1.0 / (ses ** 2)
        pooled_mean = np.sum(weights * means) / np.sum(weights)
        pooled_se = np.sqrt(1.0 / np.sum(weights))
        pooled_ci_low = pooled_mean - 1.96 * pooled_se
        pooled_ci_high = pooled_mean + 1.96 * pooled_se

        return pooled_mean, pooled_ci_low, pooled_ci_high

    raise ValueError(f"Unsupported HORIZON_AGG_METHOD={method}")


def get_method_color(method, idx):
    if method in METHOD_COLORS:
        return METHOD_COLORS[method]
    if len(FALLBACK_COLOR_CYCLE) > 0:
        return FALLBACK_COLOR_CYCLE[idx % len(FALLBACK_COLOR_CYCLE)]
    return None


def add_overall_average_line(ax, y_value, label):
    if not SHOW_OVERALL_AVERAGE_LINE or not np.isfinite(y_value):
        return

    ax.axhline(
        y_value,
        linestyle=OVERALL_AVERAGE_LINESTYLE,
        linewidth=OVERALL_AVERAGE_LINEWIDTH,
        color=OVERALL_AVERAGE_LINECOLOR,
    )
    """
    trans = blended_transform_factory(ax.transAxes, ax.transData)
    ax.text(
        0.98,
        y_value,
        label,
        transform=trans,
        ha="right",
        va="bottom",
        fontsize=YTICK_FONTSIZE,
        color=OVERALL_AVERAGE_LINECOLOR,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.5),
    )
    """


# =========================================================
# Best-config selection, based on performance vs baseline
# =========================================================

def compute_baseline_norm_rows_with_ci(df_in, metric_cols, group_cols):
    """
    Recompute baseline_norm_avg relative to repeat_last within each setting+horizon.

    Input dataframe is expected to have:
      <metric>_mean, <metric>_ci_low, <metric>_ci_high

    Returns row-level dataframe with:
      baseline_norm_avg_mean
      baseline_norm_avg_ci_low
      baseline_norm_avg_ci_high
    """
    df = df_in.copy()

    metric_mean_cols = [f"{m}_mean" for m in metric_cols]
    metric_low_cols = [f"{m}_ci_low" for m in metric_cols]
    metric_high_cols = [f"{m}_ci_high" for m in metric_cols]

    needed = metric_mean_cols + metric_low_cols + metric_high_cols
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")

    rows = []

    for _, df_group in df.groupby(group_cols, dropna=False):
        if not (df_group["method"] == BASELINE_METHOD_NAME).any():
            continue

        valid_horizons = (
            df_group.groupby("horizon")[metric_mean_cols]
            .apply(lambda x: x.notna().any().any())
        )
        valid_horizons = valid_horizons[valid_horizons].index

        df_sub = df_group[df_group["horizon"].isin(valid_horizons)].copy()
        if df_sub.empty:
            continue

        baseline_cols = ["horizon"] + metric_mean_cols + metric_low_cols + metric_high_cols
        baseline = (
            df_sub[df_sub["method"] == BASELINE_METHOD_NAME][baseline_cols]
            .drop_duplicates(subset=["horizon"])
            .set_index("horizon")
        )
        if baseline.empty:
            continue

        rename_map = {}
        for m in metric_cols:
            rename_map[f"{m}_mean"] = f"{m}_baseline_mean"
            rename_map[f"{m}_ci_low"] = f"{m}_baseline_ci_low"
            rename_map[f"{m}_ci_high"] = f"{m}_baseline_ci_high"

        baseline_renamed = baseline.rename(columns=rename_map)

        df_sub = df_sub.merge(
            baseline_renamed,
            left_on="horizon",
            right_index=True,
            how="left",
        )

        rel_mean_cols = []
        rel_low_cols = []
        rel_high_cols = []

        for m in metric_cols:
            mean_col = f"{m}_rel_mean"
            low_col = f"{m}_rel_ci_low"
            high_col = f"{m}_rel_ci_high"

            df_sub[mean_col] = safe_divide(
                df_sub[f"{m}_mean"].to_numpy(),
                df_sub[f"{m}_baseline_mean"].to_numpy(),
            )

            # Conservative CI propagation
            df_sub[low_col] = safe_divide(
                df_sub[f"{m}_ci_low"].to_numpy(),
                df_sub[f"{m}_baseline_ci_high"].to_numpy(),
            )
            df_sub[high_col] = safe_divide(
                df_sub[f"{m}_ci_high"].to_numpy(),
                df_sub[f"{m}_baseline_ci_low"].to_numpy(),
            )

            rel_mean_cols.append(mean_col)
            rel_low_cols.append(low_col)
            rel_high_cols.append(high_col)

        df_sub[rel_mean_cols] = df_sub[rel_mean_cols].replace([np.inf, -np.inf], np.nan)
        df_sub[rel_low_cols] = df_sub[rel_low_cols].replace([np.inf, -np.inf], np.nan)
        df_sub[rel_high_cols] = df_sub[rel_high_cols].replace([np.inf, -np.inf], np.nan)

        df_sub["baseline_norm_avg_mean"] = df_sub[rel_mean_cols].mean(axis=1, skipna=True)
        df_sub["baseline_norm_avg_ci_low"] = df_sub[rel_low_cols].mean(axis=1, skipna=True)
        df_sub["baseline_norm_avg_ci_high"] = df_sub[rel_high_cols].mean(axis=1, skipna=True)

        df_sub = df_sub.dropna(subset=["baseline_norm_avg_mean"]).copy()

        if not df_sub.empty:
            rows.append(df_sub)

    if not rows:
        raise ValueError("No valid combinations found with repeat_last baseline.")

    return pd.concat(rows, ignore_index=True)


def build_best_avg_error_df(results_dict: dict) -> pd.DataFrame:
    """
    For each dataset x method:
      1) select the best config using lowest performance relative to baseline
      2) aggregate avg_error across horizons for that chosen config

    Returns:
      dataset, method, avg_error, avg_error_ci_low, avg_error_ci_high
    """
    rows = []

    for dataset, df in results_dict.items():
        if df is None or len(df) == 0:
            continue

        df = standardize_config_cols(df)

        needed_avg_error_cols = ["avg_error_mean", "avg_error_ci_low", "avg_error_ci_high"]
        missing_avg_error = [c for c in needed_avg_error_cols if c not in df.columns]
        if missing_avg_error:
            print(f"Skipping {dataset}: missing avg_error columns {missing_avg_error}")
            continue

        # Use performance-vs-baseline to choose best config
        df_perf = compute_baseline_norm_rows_with_ci(
            df_in=df,
            metric_cols=metric_cols,
            group_cols=group_cols,
        )

        if not INCLUDE_REPEAT_LAST:
            df_perf = df_perf[df_perf["method"] != BASELINE_METHOD_NAME].copy()
            df = df[df["method"] != BASELINE_METHOD_NAME].copy()

        if df_perf.empty or df.empty:
            print(f"Skipping {dataset}: no methods left after filtering")
            continue

        config_perf_rows = []
        for keys, g in df_perf.groupby(["method"] + group_cols, dropna=False):
            method = keys[0]
            cfg = keys[1:]

            pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
                means=g["baseline_norm_avg_mean"].to_numpy(dtype=float),
                ci_lows=g["baseline_norm_avg_ci_low"].to_numpy(dtype=float),
                ci_highs=g["baseline_norm_avg_ci_high"].to_numpy(dtype=float),
                method=HORIZON_AGG_METHOD,
            )

            config_perf_rows.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "epi": cfg[0],
                    "einn": cfg[1],
                    "filter": cfg[2],
                    "ti": cfg[3],
                    "performance": pooled_mean,
                    "performance_ci_low": pooled_ci_low,
                    "performance_ci_high": pooled_ci_high,
                }
            )

        config_perf_df = pd.DataFrame(config_perf_rows)
        if config_perf_df.empty:
            print(f"Skipping {dataset}: no valid config-level performance rows")
            continue

        best_idx = config_perf_df.groupby("method")["performance"].idxmin()
        best_configs_df = config_perf_df.loc[best_idx].reset_index(drop=True)

        # Aggregate avg_error for the chosen config
        for _, best_row in best_configs_df.iterrows():
            method = best_row["method"]
            epi = best_row["epi"]
            einn = best_row["einn"]
            filt = best_row["filter"]
            ti = best_row["ti"]

            g_avg = df[
                (df["method"] == method)
                & (df["epi"] == epi)
                & (df["einn"] == einn)
                & (df["filter"] == filt)
                & (df["ti"] == ti)
            ].copy()

            if g_avg.empty:
                rows.append(
                    {
                        "dataset": dataset,
                        "method": method,
                        "avg_error": np.nan,
                        "avg_error_ci_low": np.nan,
                        "avg_error_ci_high": np.nan,
                    }
                )
                continue

            pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
                means=g_avg["avg_error_mean"].to_numpy(dtype=float),
                ci_lows=g_avg["avg_error_ci_low"].to_numpy(dtype=float),
                ci_highs=g_avg["avg_error_ci_high"].to_numpy(dtype=float),
                method=HORIZON_AGG_METHOD,
            )

            rows.append(
                {
                    "dataset": dataset,
                    "method": method,
                    "avg_error": pooled_mean,
                    "avg_error_ci_low": pooled_ci_low,
                    "avg_error_ci_high": pooled_ci_high,
                }
            )

    if not rows:
        return pd.DataFrame(
            columns=[
                "dataset",
                "method",
                "avg_error",
                "avg_error_ci_low",
                "avg_error_ci_high",
            ]
        )

    return pd.DataFrame(rows)


# =========================================================
# Plotting
# =========================================================

def plot_grouped_barplot_on_axis(
    ax,
    plot_df: pd.DataFrame,
    value_col: str,
    ci_low_col: str,
    ci_high_col: str,
    title: str,
    y_label: str,
):
    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    ax.set_xlabel(X_LABEL, fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel(y_label, fontsize=AXIS_LABEL_FONTSIZE)

    if plot_df.empty:
        return

    datasets_order = list(plot_df["dataset"].drop_duplicates())
    methods_order = list(plot_df["method"].drop_duplicates())

    x = np.arange(len(datasets_order))
    n_methods = len(methods_order)

    if n_methods == 0:
        return

    bar_width = BAR_GROUP_WIDTH / n_methods

    for i, method in enumerate(methods_order):
        vals = []
        yerr_low = []
        yerr_high = []

        for dataset in datasets_order:
            sub = plot_df[
                (plot_df["dataset"] == dataset)
                & (plot_df["method"] == method)
            ]

            if sub.empty:
                vals.append(np.nan)
                yerr_low.append(np.nan)
                yerr_high.append(np.nan)
            else:
                row = sub.iloc[0]
                val = row[value_col]
                ci_low = row[ci_low_col]
                ci_high = row[ci_high_col]

                vals.append(val)
                yerr_low.append(val - ci_low if pd.notna(val) and pd.notna(ci_low) else np.nan)
                yerr_high.append(ci_high - val if pd.notna(val) and pd.notna(ci_high) else np.nan)

        vals = np.asarray(vals, dtype=float)
        yerr = np.vstack([
            np.asarray(yerr_low, dtype=float),
            np.asarray(yerr_high, dtype=float),
        ])

        offsets = x - (BAR_GROUP_WIDTH / 2) + (i + 0.5) * bar_width
        color = get_method_color(method, i)

        ax.bar(
            offsets,
            vals,
            width=bar_width,
            alpha=BAR_ALPHA,
            color=color,
            yerr=yerr if SHOW_ERROR_BARS else None,
            ecolor=ERROR_BAR_COLOR,
            error_kw={
                "elinewidth": ERROR_BAR_LINEWIDTH,
                "capsize": ERROR_BAR_CAPSIZE,
                "capthick": ERROR_BAR_CAPTHICK,
            } if SHOW_ERROR_BARS else None,
        )

    overall_average = plot_df[value_col].mean(skipna=True)
    add_overall_average_line(ax, overall_average, OVERALL_AVERAGE_LABEL)

    ax.set_xticks(x)
    ax.set_xticklabels(
        datasets_order,
        rotation=XTICK_ROTATION,
        ha=XTICK_HA,
        fontsize=XTICK_FONTSIZE,
    )
    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)

    if AVG_ERROR_YMIN is not None or AVG_ERROR_YMAX is not None:
        ax.set_ylim(AVG_ERROR_YMIN, AVG_ERROR_YMAX)
    ax.set_ylim([-0.5, 0.1])
    if SHOW_GRID:
        ax.grid(True, axis=GRID_AXIS, alpha=GRID_ALPHA)


# =========================================================
# Build and plot
# =========================================================

plot_df_non_outbreak = build_best_avg_error_df(results_dfs_non_outbreak)
plot_df_consensus = build_best_avg_error_df(results_dfs_consensus)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, sharey=SHARE_Y)

plot_grouped_barplot_on_axis(
    ax=axes[0],
    plot_df=plot_df_non_outbreak,
    value_col="avg_error",
    ci_low_col="avg_error_ci_low",
    ci_high_col="avg_error_ci_high",
    title=TITLE_ALL,
    y_label=Y_LABEL,
)

plot_grouped_barplot_on_axis(
    ax=axes[1],
    plot_df=plot_df_consensus,
    value_col="avg_error",
    ci_low_col="avg_error_ci_low",
    ci_high_col="avg_error_ci_high",
    title=TITLE_CONSENSUS,
    y_label=Y_LABEL,
)

plt.tight_layout()

if SAVE_FIGURE:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    save_path = SAVE_DIR / FILENAME
    plt.savefig(save_path, dpi=SAVE_DPI, bbox_inches=SAVE_BBOX_INCHES)
    print(f"Saved: {save_path}")

if SHOW_FIGURE:
    plt.show()

if CLOSE_FIGURE:
    plt.close(fig)

In [ ]:
from matplotlib.patches import Patch

# =========================================================
# Average error by method:
# non-outbreaks vs outbreaks, two bars per method
# =========================================================

FILENAME_METHOD_AVG_ERROR = (
    "best_patch_avg_error_by_method_non_outbreak_vs_outbreak.png"
)

TITLE_METHOD_AVG_ERROR = "Average error by outbreak status"
Y_LABEL_METHOD_AVG_ERROR = "Average error"

NON_OUTBREAK_LABEL = "Non-outbreaks"
OUTBREAK_LABEL = "Outbreaks"

NON_OUTBREAK_ALPHA = 0.5
OUTBREAK_ALPHA = 1.0

BAR_WIDTH_TWO_PERIODS = 0.36
FIGSIZE_METHOD_AVG_ERROR = (8, 8)

# Optional: sort methods by outbreak average error
SORT_METHODS_BY_OUTBREAK = True
SORT_ASCENDING = True

# Hide x tick labels, but keep bars in place
HIDE_XTICK_LABELS = True


def make_nonnegative_yerr(val, ci_low, ci_high):
    """
    Matplotlib expects nonnegative asymmetric error sizes.
    """
    if pd.isna(val) or pd.isna(ci_low) or pd.isna(ci_high):
        return np.nan, np.nan

    low_err = max(val - ci_low, 0.0)
    high_err = max(ci_high - val, 0.0)

    return low_err, high_err


def build_method_avg_error_summary(
    plot_df: pd.DataFrame,
    period_label: str,
) -> pd.DataFrame:
    """
    Pool avg_error across datasets for each method.

    Input should be the output of build_best_avg_error_df(...):
      dataset, method, avg_error, avg_error_ci_low, avg_error_ci_high

    Returns one row per method for the given period.
    """
    if plot_df.empty:
        return pd.DataFrame(
            columns=[
                "period",
                "method",
                "n_datasets",
                "avg_error",
                "avg_error_ci_low",
                "avg_error_ci_high",
                "simple_mean_avg_error",
                "median_avg_error",
            ]
        )

    rows = []

    for method, g in plot_df.groupby("method", dropna=False):
        pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
            means=g["avg_error"].to_numpy(dtype=float),
            ci_lows=g["avg_error_ci_low"].to_numpy(dtype=float),
            ci_highs=g["avg_error_ci_high"].to_numpy(dtype=float),
            method=HORIZON_AGG_METHOD,
        )

        rows.append(
            {
                "period": period_label,
                "method": method,
                "n_datasets": g["dataset"].nunique(),
                "avg_error": pooled_mean,
                "avg_error_ci_low": pooled_ci_low,
                "avg_error_ci_high": pooled_ci_high,
                "simple_mean_avg_error": g["avg_error"].mean(skipna=True),
                "median_avg_error": g["avg_error"].median(skipna=True),
            }
        )

    return pd.DataFrame(rows)


def build_two_period_method_avg_error_plot_df(
    non_outbreak_method_df: pd.DataFrame,
    outbreak_method_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Merge non-outbreak and outbreak method summaries into one row per method.
    """
    non_df = non_outbreak_method_df.copy().rename(
        columns={
            "n_datasets": "n_datasets_non_outbreak",
            "avg_error": "avg_error_non_outbreak",
            "avg_error_ci_low": "avg_error_ci_low_non_outbreak",
            "avg_error_ci_high": "avg_error_ci_high_non_outbreak",
            "simple_mean_avg_error": "simple_mean_avg_error_non_outbreak",
            "median_avg_error": "median_avg_error_non_outbreak",
        }
    )

    out_df = outbreak_method_df.copy().rename(
        columns={
            "n_datasets": "n_datasets_outbreak",
            "avg_error": "avg_error_outbreak",
            "avg_error_ci_low": "avg_error_ci_low_outbreak",
            "avg_error_ci_high": "avg_error_ci_high_outbreak",
            "simple_mean_avg_error": "simple_mean_avg_error_outbreak",
            "median_avg_error": "median_avg_error_outbreak",
        }
    )

    non_df = non_df.drop(columns=["period"], errors="ignore")
    out_df = out_df.drop(columns=["period"], errors="ignore")

    plot_df = non_df.merge(
        out_df,
        on="method",
        how="outer",
    )

    if SORT_METHODS_BY_OUTBREAK and "avg_error_outbreak" in plot_df.columns:
        plot_df = plot_df.sort_values(
            "avg_error_outbreak",
            ascending=SORT_ASCENDING,
        )

    return plot_df.reset_index(drop=True)


def build_overall_avg_error_stats(
    plot_df_non_outbreak: pd.DataFrame,
    plot_df_outbreak: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compute overall average errors.

    Returns:
      overall_simple_stats_df:
        simple descriptive averages across dataset x method rows

      overall_pooled_stats_df:
        pooled average error using aggregate_estimates_with_ci
    """
    df_non = plot_df_non_outbreak.copy()
    df_non["period"] = NON_OUTBREAK_LABEL

    df_out = plot_df_outbreak.copy()
    df_out["period"] = OUTBREAK_LABEL

    combined_df = pd.concat([df_non, df_out], ignore_index=True)

    if combined_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    overall_simple_stats_df = (
        combined_df
        .groupby("period", dropna=False)
        .agg(
            n_rows=("avg_error", "count"),
            n_datasets=("dataset", "nunique"),
            n_methods=("method", "nunique"),
            mean_avg_error=("avg_error", "mean"),
            median_avg_error=("avg_error", "median"),
            min_avg_error=("avg_error", "min"),
            max_avg_error=("avg_error", "max"),
        )
        .reset_index()
    )

    pooled_rows = []

    for period, g in combined_df.groupby("period", dropna=False):
        pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
            means=g["avg_error"].to_numpy(dtype=float),
            ci_lows=g["avg_error_ci_low"].to_numpy(dtype=float),
            ci_highs=g["avg_error_ci_high"].to_numpy(dtype=float),
            method=HORIZON_AGG_METHOD,
        )

        pooled_rows.append(
            {
                "period": period,
                "pooled_avg_error": pooled_mean,
                "pooled_avg_error_ci_low": pooled_ci_low,
                "pooled_avg_error_ci_high": pooled_ci_high,
            }
        )

    overall_pooled_stats_df = pd.DataFrame(pooled_rows)

    return overall_simple_stats_df, overall_pooled_stats_df


def build_method_color_index_map(
    *dfs: pd.DataFrame,
    method_col: str = "method",
) -> dict:
    """
    Build a stable method -> color index mapping using first-seen order.

    This keeps method colors consistent with the original sequential order
    even if the final plotting table is sorted differently.
    """
    method_series = []

    for df in dfs:
        if df is not None and not df.empty and method_col in df.columns:
            method_series.append(df[method_col])

    if not method_series:
        return {}

    method_order = (
        pd.concat(method_series, ignore_index=True)
        .dropna()
        .drop_duplicates()
        .to_list()
    )

    return {
        method: idx
        for idx, method in enumerate(method_order)
    }


def plot_method_avg_error_two_periods(
    plot_df: pd.DataFrame,
    method_to_color_idx: dict,
    save_filename: str | None = None,
):
    """
    Two bars per method:
      - non-outbreaks: alpha 0.5
      - outbreaks: alpha 1.0

    No red dashed overall-average line.
    X-tick labels are hidden.
    Method colors are fixed by method_to_color_idx.
    """
    if plot_df.empty:
        print("No method-level average error data to plot.")
        return

    methods = plot_df["method"].astype(str).to_list()
    x = np.arange(len(methods))

    vals_non = plot_df["avg_error_non_outbreak"].to_numpy(dtype=float)
    vals_out = plot_df["avg_error_outbreak"].to_numpy(dtype=float)

    yerr_non_low = []
    yerr_non_high = []
    yerr_out_low = []
    yerr_out_high = []

    for _, row in plot_df.iterrows():
        low, high = make_nonnegative_yerr(
            row["avg_error_non_outbreak"],
            row["avg_error_ci_low_non_outbreak"],
            row["avg_error_ci_high_non_outbreak"],
        )
        yerr_non_low.append(low)
        yerr_non_high.append(high)

        low, high = make_nonnegative_yerr(
            row["avg_error_outbreak"],
            row["avg_error_ci_low_outbreak"],
            row["avg_error_ci_high_outbreak"],
        )
        yerr_out_low.append(low)
        yerr_out_high.append(high)

    yerr_non = np.vstack(
        [
            np.asarray(yerr_non_low, dtype=float),
            np.asarray(yerr_non_high, dtype=float),
        ]
    )

    yerr_out = np.vstack(
        [
            np.asarray(yerr_out_low, dtype=float),
            np.asarray(yerr_out_high, dtype=float),
        ]
    )

    fig, ax = plt.subplots(figsize=FIGSIZE_METHOD_AVG_ERROR)

    colors = [
        get_method_color(
            method,
            method_to_color_idx.get(method, i),
        )
        for i, method in enumerate(methods)
    ]

    # Non-outbreak bars: lighter opacity
    ax.bar(
        x - BAR_WIDTH_TWO_PERIODS / 2,
        vals_non,
        width=BAR_WIDTH_TWO_PERIODS,
        alpha=NON_OUTBREAK_ALPHA,
        color=colors,
        yerr=yerr_non if SHOW_ERROR_BARS else None,
        ecolor=ERROR_BAR_COLOR,
        error_kw={
            "elinewidth": ERROR_BAR_LINEWIDTH,
            "capsize": ERROR_BAR_CAPSIZE,
            "capthick": ERROR_BAR_CAPTHICK,
        } if SHOW_ERROR_BARS else None,
    )

    # Outbreak bars: full opacity
    ax.bar(
        x + BAR_WIDTH_TWO_PERIODS / 2,
        vals_out,
        width=BAR_WIDTH_TWO_PERIODS,
        alpha=OUTBREAK_ALPHA,
        color=colors,
        yerr=yerr_out if SHOW_ERROR_BARS else None,
        ecolor=ERROR_BAR_COLOR,
        error_kw={
            "elinewidth": ERROR_BAR_LINEWIDTH,
            "capsize": ERROR_BAR_CAPSIZE,
            "capthick": ERROR_BAR_CAPTHICK,
        } if SHOW_ERROR_BARS else None,
    )

    ax.set_title(
        TITLE_METHOD_AVG_ERROR,
        fontsize=TITLE_FONTSIZE,
    )

    ax.set_ylabel(
        Y_LABEL_METHOD_AVG_ERROR,
        fontsize=AXIS_LABEL_FONTSIZE,
    )

    ax.set_xticks(x)
    ax.set_yticklabels([])
    if False:
        ax.set_xticklabels([])
        ax.tick_params(axis="x", length=0)
    else:
        ax.set_xticklabels(
            methods,
            rotation=45,
            ha="right",
            fontsize=XTICK_FONTSIZE,
        )

    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)


    if SHOW_GRID:
        ax.grid(True, axis=GRID_AXIS, alpha=GRID_ALPHA)

    legend_handles = [
        Patch(
            facecolor="gray",
            alpha=NON_OUTBREAK_ALPHA,
            label=NON_OUTBREAK_LABEL,
        ),
        Patch(
            facecolor="gray",
            alpha=OUTBREAK_ALPHA,
            label=OUTBREAK_LABEL,
        ),
    ]

    ax.legend(
        handles=legend_handles,
        loc="best",
        fontsize=YTICK_FONTSIZE,
        frameon=True,
    )

    plt.tight_layout()

    if SAVE_FIGURE and save_filename is not None:
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        save_path = SAVE_DIR / save_filename
        plt.savefig(
            save_path,
            dpi=SAVE_DPI,
            bbox_inches=SAVE_BBOX_INCHES,
        )
        print(f"Saved: {save_path}")

    if SHOW_FIGURE:
        plt.show()

    if CLOSE_FIGURE:
        plt.close(fig)


# =========================================================
# Build tables and plot
# =========================================================

plot_df_non_outbreak = build_best_avg_error_df(results_dfs_non_outbreak)
plot_df_outbreak = build_best_avg_error_df(results_dfs_consensus)

# Stable color order from the original dataset-level tables,
# before method-level sorting.
method_to_color_idx = build_method_color_index_map(
    plot_df_non_outbreak,
    plot_df_outbreak,
)

method_avg_error_non_outbreak_df = build_method_avg_error_summary(
    plot_df=plot_df_non_outbreak,
    period_label=NON_OUTBREAK_LABEL,
)

method_avg_error_outbreak_df = build_method_avg_error_summary(
    plot_df=plot_df_outbreak,
    period_label=OUTBREAK_LABEL,
)

method_avg_error_two_period_plot_df = build_two_period_method_avg_error_plot_df(
    non_outbreak_method_df=method_avg_error_non_outbreak_df,
    outbreak_method_df=method_avg_error_outbreak_df,
)

overall_avg_error_simple_stats_df, overall_avg_error_pooled_stats_df = (
    build_overall_avg_error_stats(
        plot_df_non_outbreak=plot_df_non_outbreak,
        plot_df_outbreak=plot_df_outbreak,
    )
)

print("\nDataset x method average-error rows, non-outbreaks:")
display(plot_df_non_outbreak)

print("\nDataset x method average-error rows, outbreaks:")
display(plot_df_outbreak)

print("\nMethod-level average error, non-outbreaks:")
display(method_avg_error_non_outbreak_df)

print("\nMethod-level average error, outbreaks:")
display(method_avg_error_outbreak_df)

print("\nTwo-period method plot table:")
display(method_avg_error_two_period_plot_df)

print("\nOverall average-error simple stats:")
display(overall_avg_error_simple_stats_df)

print("\nOverall average-error pooled stats:")
display(overall_avg_error_pooled_stats_df)

plot_method_avg_error_two_periods(
    plot_df=method_avg_error_two_period_plot_df,
    method_to_color_idx=method_to_color_idx,
    save_filename=FILENAME_METHOD_AVG_ERROR,
)

### OUTBREAK EXAMPLE

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================
# User-configurable params
# =========================
dataset = "JHUcase"
horizon = 7
folder = Path(f"/net/dali/home/mscbio/rul98/EpiPatch/retrain0301_{dataset}")
intervals_path = Path(f"{dataset}_consensus_intervals.csv")

months_before = 1
months_after = 3

figsize = (10, 7)
dpi = 300
tight_layout = True

title_fontsize = 24
label_fontsize = 22
tick_fontsize = 18
legend_fontsize = 18
outbreak_text_fontsize = 18

truth_color = "black"
truth_linewidth = 2.8
truth_linestyle = "-"

repeat_last_color = "#ff0000"
repeat_last_linewidth = 2.0
repeat_last_alpha = 1.0

other_linewidth = 1.5
other_alpha = 0.95

ymin = 0
ymax = 12
xlabel = ""
ylabel = "Normalized disease prevalence"

show_title = True

show_legend = True
legend_loc = "upper left"
legend_bbox_to_anchor = (1.01, 1.0)
legend_frameon = False

tab20_name = "tab20"

# Optional explicit model color order.
# Leave as None to use alphabetical model order.
# y_true, repeat_last, and Naive are always excluded from this ordering.
METHOD_COLOR_ORDER = None

METHOD_COLORS = {
    # Optional manual overrides:
    # "AGCRN": "tab:blue",
    # "DCRNN": "tab:orange",
}

show_outbreak_bar = True
outbreak_bar_y = 0.985
outbreak_bar_linewidth = 3.0
outbreak_bar_pre_color = "black"
outbreak_bar_outbreak_color = "red"
outbreak_bar_post_color = "black"

show_outbreak_text = True
outbreak_text = "Outbreak"
outbreak_text_color = "black"
outbreak_text_y = 0.955
outbreak_text_ha = "center"
outbreak_text_va = "top"


# =========================
# Helpers
# =========================
def cfg_value_for_filename(x):
    if isinstance(x, (bool, np.bool_)):
        return "True" if bool(x) else "False"

    if isinstance(x, str):
        s = x.strip()
        if s.lower() == "true":
            return "True"
        if s.lower() == "false":
            return "False"
        if s == "1":
            return "True"
        if s == "0":
            return "False"
        return s

    if isinstance(x, (int, np.integer)):
        if x == 1:
            return "True"
        if x == 0:
            return "False"

    if isinstance(x, (float, np.floating)) and not pd.isna(x):
        if x == 1.0:
            return "True"
        if x == 0.0:
            return "False"

    return str(x)


def make_result_path(method, epi, einn, filt, ti):
    file_name = (
        f"retrain_{dataset}_{method}__horizon={horizon}"
        f"__epi={cfg_value_for_filename(epi)}"
        f"__einn={cfg_value_for_filename(einn)}"
        f"__filter={cfg_value_for_filename(filt)}"
        f"__ti={cfg_value_for_filename(ti)}.csv"
    )
    return folder / file_name


def get_existing_or_fallback_path(method, epi, einn, filt, ti):
    fp = make_result_path(method, epi, einn, filt, ti)

    if fp.exists():
        return fp

    fallback_fp = make_result_path(
        method=method,
        epi=False,
        einn=False,
        filt=False,
        ti=False,
    )

    if fallback_fp.exists():
        print(f"Missing best-config file: {fp}")
        print(f"Using all-False fallback: {fallback_fp}")
        return fallback_fp

    print(f"Missing file: {fp}")

    if fallback_fp != fp:
        print(f"Missing all-False fallback: {fallback_fp}")

    return None


def build_method_color_index_map(
    methods,
    explicit_order=None,
    excluded_methods=None,
):
    """
    Stable method -> color index mapping.

    y_true, repeat_last, and Naive should not consume colors.
    """
    if excluded_methods is None:
        excluded_methods = set()

    excluded_methods = {str(m) for m in excluded_methods}

    method_set = {
        str(m)
        for m in methods
        if str(m) not in excluded_methods
    }

    if explicit_order is not None:
        ordered = [
            str(m)
            for m in explicit_order
            if str(m) in method_set
        ]

        for m in sorted(method_set):
            if m not in ordered:
                ordered.append(m)
    else:
        ordered = sorted(method_set)

    return {
        method: idx
        for idx, method in enumerate(ordered)
    }


def get_prediction_method_color(method, method_to_color_idx):
    """
    y_true is plotted separately in black.
    repeat_last / Naive is red.
    All other methods use stable method color ordering.
    """
    method = str(method)

    if method == "repeat_last":
        return repeat_last_color

    if method in METHOD_COLORS:
        return METHOD_COLORS[method]

    idx = method_to_color_idx.get(method)

    if idx is None:
        idx = len(method_to_color_idx)

    tab20 = plt.get_cmap(tab20_name)
    return tab20(idx % 20)


# =========================
# Load intervals
# =========================
intervals_df = pd.read_csv(intervals_path)
intervals_df["state_code"] = intervals_df["state_code"].astype(str)
intervals_df["start"] = pd.to_datetime(intervals_df["start"])
intervals_df["end"] = pd.to_datetime(intervals_df["end"])


# =========================
# Load best-config files
# =========================
best_cfg_dataset = best_configs_df[best_configs_df["dataset"] == dataset].copy()

if best_cfg_dataset.empty:
    print(f"No best-config rows found for dataset={dataset}; trying repeat_last only.")

methods_in_cfg = (
    best_cfg_dataset["method"].astype(str).tolist()
    if not best_cfg_dataset.empty
    else []
)

if "repeat_last" not in methods_in_cfg:
    repeat_last_row = {
        "dataset": dataset,
        "method": "repeat_last",
        "epi": False,
        "einn": False,
        "filter": False,
        "ti": False,
    }

    best_cfg_dataset = pd.concat(
        [best_cfg_dataset, pd.DataFrame([repeat_last_row])],
        ignore_index=True,
    )


truth_parts = []
pred_parts = []

for _, cfg_row in best_cfg_dataset.iterrows():
    method = str(cfg_row["method"])

    epi = cfg_row["epi"]
    einn = cfg_row["einn"]
    filt = cfg_row["filter"]
    ti = cfg_row["ti"]

    fp = get_existing_or_fallback_path(
        method=method,
        epi=epi,
        einn=einn,
        filt=filt,
        ti=ti,
    )

    if fp is None:
        continue

    df = pd.read_csv(fp)

    if df.empty:
        print(f"Empty file skipped: {fp}")
        continue

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["state"] = df["state"].astype(str)

    truth_parts.append(df[["timestamp", "state", "y_true"]].copy())

    tmp = df[["timestamp", "state", "y_pred"]].copy()
    tmp["method"] = method
    pred_parts.append(tmp)


if len(pred_parts) == 0:
    raise ValueError(f"No usable prediction files found for {dataset}, horizon={horizon}")

if len(truth_parts) == 0:
    raise ValueError(f"No usable truth values found for {dataset}, horizon={horizon}")


truth_df = pd.concat(truth_parts, ignore_index=True)
truth_df = (
    truth_df
    .groupby(["timestamp", "state"], as_index=False)["y_true"]
    .mean()
    .sort_values(["state", "timestamp"])
    .reset_index(drop=True)
)

pred_df = pd.concat(pred_parts, ignore_index=True)
pred_df = (
    pred_df
    .groupby(["method", "timestamp", "state"], as_index=False)["y_pred"]
    .mean()
    .sort_values(["method", "state", "timestamp"])
    .reset_index(drop=True)
)


# =========================
# Find outbreak with highest average y_true
# =========================
outbreak_rows = []

for _, row in intervals_df.iterrows():
    state = row["state_code"]
    start = row["start"]
    end = row["end"]

    vals = truth_df.loc[
        (truth_df["state"] == state)
        & (truth_df["timestamp"] >= start)
        & (truth_df["timestamp"] <= end),
        "y_true",
    ]

    if len(vals) == 0:
        continue

    outbreak_rows.append(
        {
            "state": state,
            "start": start,
            "end": end,
            "avg_y_true": vals.mean(),
            "n_points": len(vals),
        }
    )

if len(outbreak_rows) == 0:
    raise ValueError("No outbreak intervals overlap with available truth data.")

outbreak_summary = (
    pd.DataFrame(outbreak_rows)
    .sort_values("avg_y_true", ascending=False)
    .reset_index(drop=True)
)

best = outbreak_summary.iloc[0]
selected_state = best["state"]
selected_start = best["start"]
selected_end = best["end"]
selected_avg = best["avg_y_true"]

print("Selected outbreak:")
print(best)


# =========================
# Restrict x-axis to window around outbreak start
# =========================
x_min = selected_start - pd.DateOffset(months=months_before)
x_max = selected_start + pd.DateOffset(months=months_after)

truth_state = truth_df[
    (truth_df["state"] == selected_state)
    & (truth_df["timestamp"] >= x_min)
    & (truth_df["timestamp"] <= x_max)
].sort_values("timestamp").copy()

pred_state = pred_df[
    (pred_df["state"] == selected_state)
    & (pred_df["timestamp"] >= x_min)
    & (pred_df["timestamp"] <= x_max)
].sort_values(["method", "timestamp"]).copy()


# =========================
# Method ordering, colors, and labels
# =========================
methods_present = pred_state["method"].astype(str).drop_duplicates().tolist()

if "model_names" in globals():
    ordered_methods = [
        str(m)
        for m in model_names
        if str(m) in methods_present
    ]

    if "repeat_last" in methods_present and "repeat_last" not in ordered_methods:
        ordered_methods.append("repeat_last")

    for m in methods_present:
        if m not in ordered_methods:
            ordered_methods.append(m)

    methods_present = ordered_methods


excluded_from_model_color_order = {
    "y_true",
    "repeat_last",
    "Naive",
}

method_to_color_idx = build_method_color_index_map(
    methods=methods_present,
    explicit_order=METHOD_COLOR_ORDER,
    excluded_methods=excluded_from_model_color_order,
)

color_map = {
    method: get_prediction_method_color(
        method=method,
        method_to_color_idx=method_to_color_idx,
    )
    for method in methods_present
}

legend_label_map = {
    "repeat_last": "Naive",
}


# =========================
# Plot
# =========================
fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

ax.plot(
    truth_state["timestamp"],
    truth_state["y_true"],
    color=truth_color,
    linewidth=truth_linewidth,
    linestyle=truth_linestyle,
    label="y_true",
    zorder=4,
)

for method in methods_present:
    dfm = pred_state[pred_state["method"].astype(str) == str(method)]

    if dfm.empty:
        continue

    is_repeat_last = method == "repeat_last"

    ax.plot(
        dfm["timestamp"],
        dfm["y_pred"],
        color=color_map[method],
        linewidth=repeat_last_linewidth if is_repeat_last else other_linewidth,
        alpha=repeat_last_alpha if is_repeat_last else other_alpha,
        label=legend_label_map.get(method, method),
        zorder=3 if is_repeat_last else 2,
    )


ax.set_xlim(x_min, x_max)
ax.set_ylim(bottom=ymin, top=ymax)


# =========================
# Top outbreak bar annotation
# =========================
if show_outbreak_bar:
    trans = ax.get_xaxis_transform()

    ax.plot(
        [x_min, selected_start],
        [outbreak_bar_y, outbreak_bar_y],
        color=outbreak_bar_pre_color,
        linewidth=outbreak_bar_linewidth,
        solid_capstyle="butt",
        transform=trans,
        clip_on=False,
        zorder=6,
    )

    ax.plot(
        [selected_start, selected_end],
        [outbreak_bar_y, outbreak_bar_y],
        color=outbreak_bar_outbreak_color,
        linewidth=outbreak_bar_linewidth,
        solid_capstyle="butt",
        transform=trans,
        clip_on=False,
        zorder=7,
    )

    ax.plot(
        [selected_end, x_max],
        [outbreak_bar_y, outbreak_bar_y],
        color=outbreak_bar_post_color,
        linewidth=outbreak_bar_linewidth,
        solid_capstyle="butt",
        transform=trans,
        clip_on=False,
        zorder=6,
    )

    if show_outbreak_text:
        outbreak_mid = selected_start + (selected_end - selected_start) / 2

        ax.text(
            outbreak_mid,
            outbreak_text_y,
            outbreak_text,
            color=outbreak_text_color,
            fontsize=outbreak_text_fontsize,
            ha=outbreak_text_ha,
            va=outbreak_text_va,
            transform=trans,
            clip_on=False,
            zorder=8,
        )


# =========================
# Labels, title, legend
# =========================
if show_title:
    ax.set_title(
        "JHUCase Missouri Outbreak",
        fontsize=title_fontsize,
    )

ax.set_xlabel(xlabel, fontsize=label_fontsize)
ax.set_ylabel(ylabel, fontsize=label_fontsize)

ax.tick_params(axis="y", labelsize=tick_fontsize)
ax.tick_params(axis="x", labelsize=tick_fontsize, rotation=45)

if show_legend:
    ax.legend(
        loc=legend_loc,
        bbox_to_anchor=legend_bbox_to_anchor,
        frameon=legend_frameon,
        fontsize=legend_fontsize,
    )

if tight_layout:
    plt.tight_layout()

plt.savefig("JHUCase_outbreak.png", dpi=300, bbox_inches="tight")
plt.show()

### TUSOAI

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


RESULTS_DICT = results_dfs
TUSOAI_DF = tusoai_df

metric_cols = [
    "mse",
    "mae",
    "rmse",
    "medae",
    "medse",
    "mse_filtered",
    "mae_filtered",
    "rmse_filtered",
]

METRIC_LABELS = {
    "mse": "MSE",
    "mae": "MAE",
    "rmse": "RMSE",
    "medae": "medAE",
    "medse": "medSE",
    "mse_filtered": "MSE filtered",
    "mae_filtered": "MAE filtered",
    "rmse_filtered": "RMSE filtered",
}

group_cols = ["epi", "einn", "filter", "ti"]
CONFIG_COLS = ["epi", "einn", "filter", "ti"]

BASELINE_METHOD_NAME = "repeat_last"

ARIMA_METHOD_ALIASES = {"arima"}
DLINEAR_METHOD_ALIASES = {"dlinear", "dlinear_model", "d_linear"}

TUSOAI_SPATIAL_METHODS = {"tusoai"}
TUSOAI_NOSPATIAL_METHODS = {"tusoai_simple_stgnn"}

HORIZON_AGG_METHOD = "inverse_variance"

DISPLAY_METHOD_ORDER = [
    "ARIMA",
    "Dlinear",
    "Best graph-based",
    "TusoAI (spatial)",
    "TusoAI (no spatial)",
]

FIGSIZE_WIN = (14, 8)
FIGSIZE_PERF = (14, 8)

BAR_GROUP_WIDTH = 0.8
BAR_ALPHA = 0.9

SHOW_GRID = False
GRID_ALPHA = 0.25
GRID_AXIS = "y"

WIN_YMIN = 0.0
WIN_YMAX = 1.0

PERF_YMIN = 0.0
PERF_YMAX = 2.0

WIN_REFERENCE_Y = 0.5
WIN_REFERENCE_LINESTYLE = "--"
WIN_REFERENCE_LINEWIDTH = 1.5
WIN_REFERENCE_LINECOLOR = "black"

PERF_REFERENCE_Y = 1.0
PERF_REFERENCE_LINESTYLE = "--"
PERF_REFERENCE_LINEWIDTH = 1.5
PERF_REFERENCE_LINECOLOR = "black"

SHOW_ERROR_BARS = True
ERROR_BAR_COLOR = "black"
ERROR_BAR_LINEWIDTH = 1.2
ERROR_BAR_CAPSIZE = 4
ERROR_BAR_CAPTHICK = 1.2

TITLE_FONTSIZE = 24
AXIS_LABEL_FONTSIZE = 24
XTICK_FONTSIZE = 22
YTICK_FONTSIZE = 18
LEGEND_FONTSIZE = 18

WIN_TITLE_TEMPLATE = "Win rate across datasets ({metric_label})"
PERF_TITLE_TEMPLATE = "Overall performance across datasets ({metric_label})"

X_LABEL = ""
WIN_Y_LABEL = "Win rate"
PERF_Y_LABEL = "Error relative to baseline"

WIN_REFERENCE_LABEL = "50%"
PERF_REFERENCE_LABEL = "repeat_last = 1"

XTICK_ROTATION = 45
XTICK_HA = "right"

SHOW_LEGEND = True
LEGEND_LOC = "upper left"
LEGEND_BBOX_TO_ANCHOR = (1.02, 1.0)
LEGEND_FRAMEON = True
LEGEND_TITLE = ""

METHOD_COLORS = {
    # "ARIMA": "tab:blue",
    # "Dlinear": "tab:orange",
    # "Best graph-based": "tab:green",
    # "TusoAI (spatial)": "tab:red",
    # "TusoAI (no spatial)": "tab:purple",
}

FALLBACK_COLOR_CYCLE = list(plt.get_cmap("tab20").colors)

SAVE_FIGURES = True
SAVE_DIR = Path(".")
SAVE_DPI = 300
SAVE_BBOX_INCHES = "tight"

WIN_FILENAME_TEMPLATE = "five_method_win_rate_across_datasets_{metric}.png"
PERF_FILENAME_TEMPLATE = "five_method_performance_across_datasets_{metric}.png"

SHOW_FIGURES = True
CLOSE_FIGURES = True


def get_metric_label(metric):
    return METRIC_LABELS.get(metric, metric)


def normalize_method_name(method):
    return str(method).strip().lower()


def is_arima(method):
    return normalize_method_name(method) in ARIMA_METHOD_ALIASES


def is_dlinear(method):
    return normalize_method_name(method) in DLINEAR_METHOD_ALIASES


def is_baseline(method):
    return normalize_method_name(method) == normalize_method_name(BASELINE_METHOD_NAME)


def is_tusoai_name(method):
    m = normalize_method_name(method)
    return m in TUSOAI_SPATIAL_METHODS or m in TUSOAI_NOSPATIAL_METHODS


def is_graph_based_candidate(method):
    return (
        not is_baseline(method)
        and not is_arima(method)
        and not is_dlinear(method)
        and not is_tusoai_name(method)
    )


def normalize_config_value(x):
    if pd.isna(x):
        return x

    if isinstance(x, (bool, np.bool_)):
        return bool(x)

    if isinstance(x, str):
        s = x.strip()
        if s.lower() == "true":
            return True
        if s.lower() == "false":
            return False
        if s == "1":
            return True
        if s == "0":
            return False
        return s

    if isinstance(x, (int, np.integer)):
        if x == 1:
            return True
        if x == 0:
            return False

    if isinstance(x, (float, np.floating)) and np.isfinite(x):
        if x == 1.0:
            return True
        if x == 0.0:
            return False

    return x


def standardize_config_cols(df):
    df = df.copy()
    for col in CONFIG_COLS:
        if col in df.columns:
            df[col] = df[col].map(normalize_config_value)
    return df


def safe_divide(num, den):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)
    out = np.full(np.broadcast(num, den).shape, np.nan, dtype=float)
    valid = np.isfinite(num) & np.isfinite(den) & (den != 0)
    out[valid] = num[valid] / den[valid]
    return out


def ci_to_se(ci_low, ci_high):
    if not (np.isfinite(ci_low) and np.isfinite(ci_high)):
        return np.nan
    width = ci_high - ci_low
    if width <= 0:
        return np.nan
    return width / (2 * 1.96)


def clip01(x):
    return np.clip(x, 0.0, 1.0) if np.isfinite(x) else x


def aggregate_estimates_with_ci(
    means,
    ci_lows,
    ci_highs,
    method="inverse_variance",
    clip_to_01=False,
):
    means = np.asarray(means, dtype=float)
    ci_lows = np.asarray(ci_lows, dtype=float)
    ci_highs = np.asarray(ci_highs, dtype=float)

    valid = np.isfinite(means) & np.isfinite(ci_lows) & np.isfinite(ci_highs)
    means = means[valid]
    ci_lows = ci_lows[valid]
    ci_highs = ci_highs[valid]

    if len(means) == 0:
        return np.nan, np.nan, np.nan

    if len(means) == 1:
        pooled_mean = means[0]
        pooled_ci_low = ci_lows[0]
        pooled_ci_high = ci_highs[0]

    elif method == "simple_mean":
        pooled_mean = means.mean()
        pooled_ci_low = ci_lows.mean()
        pooled_ci_high = ci_highs.mean()

    elif method == "inverse_variance":
        ses = np.array(
            [ci_to_se(lo, hi) for lo, hi in zip(ci_lows, ci_highs)],
            dtype=float,
        )
        valid_se = np.isfinite(ses) & (ses > 0)

        if not valid_se.any():
            pooled_mean = means.mean()
            pooled_ci_low = ci_lows.mean()
            pooled_ci_high = ci_highs.mean()
        else:
            means = means[valid_se]
            ses = ses[valid_se]
            weights = 1.0 / (ses ** 2)
            pooled_mean = np.sum(weights * means) / np.sum(weights)
            pooled_se = np.sqrt(1.0 / np.sum(weights))
            pooled_ci_low = pooled_mean - 1.96 * pooled_se
            pooled_ci_high = pooled_mean + 1.96 * pooled_se

    else:
        raise ValueError(f"Unsupported HORIZON_AGG_METHOD={method}")

    if clip_to_01:
        pooled_mean = clip01(pooled_mean)
        pooled_ci_low = clip01(pooled_ci_low)
        pooled_ci_high = clip01(pooled_ci_high)

    return pooled_mean, pooled_ci_low, pooled_ci_high


def get_method_color(method, idx):
    if method in METHOD_COLORS:
        return METHOD_COLORS[method]
    if len(FALLBACK_COLOR_CYCLE) > 0:
        return FALLBACK_COLOR_CYCLE[idx % len(FALLBACK_COLOR_CYCLE)]
    return None


def choose_best_row(candidates):
    candidates = candidates.copy()
    candidates = candidates[np.isfinite(candidates["performance"])]
    if candidates.empty:
        return None
    return candidates.loc[candidates["performance"].idxmin()]


def compute_baseline_norm_rows_with_ci(df_in, metric_cols_for_run, group_cols):
    df = df_in.copy()

    metric_mean_cols = [f"{m}_mean" for m in metric_cols_for_run]
    metric_low_cols = [f"{m}_ci_low" for m in metric_cols_for_run]
    metric_high_cols = [f"{m}_ci_high" for m in metric_cols_for_run]

    needed = (
        ["method", "horizon"]
        + group_cols
        + metric_mean_cols
        + metric_low_cols
        + metric_high_cols
    )
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")

    rows = []

    for _, df_group in df.groupby(group_cols, dropna=False):
        if not (df_group["method"].map(is_baseline)).any():
            continue

        valid_horizons = (
            df_group.groupby("horizon")[metric_mean_cols]
            .apply(lambda x: x.notna().any().any())
        )
        valid_horizons = valid_horizons[valid_horizons].index

        df_sub = df_group[df_group["horizon"].isin(valid_horizons)].copy()
        if df_sub.empty:
            continue

        baseline_cols = ["horizon"] + metric_mean_cols + metric_low_cols + metric_high_cols
        baseline = (
            df_sub[df_sub["method"].map(is_baseline)][baseline_cols]
            .drop_duplicates(subset=["horizon"])
            .set_index("horizon")
        )
        if baseline.empty:
            continue

        rename_map = {}
        for m in metric_cols_for_run:
            rename_map[f"{m}_mean"] = f"{m}_baseline_mean"
            rename_map[f"{m}_ci_low"] = f"{m}_baseline_ci_low"
            rename_map[f"{m}_ci_high"] = f"{m}_baseline_ci_high"

        baseline_renamed = baseline.rename(columns=rename_map)

        df_sub = df_sub.merge(
            baseline_renamed,
            left_on="horizon",
            right_index=True,
            how="left",
        )

        rel_mean_cols = []
        rel_low_cols = []
        rel_high_cols = []

        for m in metric_cols_for_run:
            mean_col = f"{m}_rel_mean"
            low_col = f"{m}_rel_ci_low"
            high_col = f"{m}_rel_ci_high"

            df_sub[mean_col] = safe_divide(
                df_sub[f"{m}_mean"].to_numpy(),
                df_sub[f"{m}_baseline_mean"].to_numpy(),
            )
            df_sub[low_col] = safe_divide(
                df_sub[f"{m}_ci_low"].to_numpy(),
                df_sub[f"{m}_baseline_ci_high"].to_numpy(),
            )
            df_sub[high_col] = safe_divide(
                df_sub[f"{m}_ci_high"].to_numpy(),
                df_sub[f"{m}_baseline_ci_low"].to_numpy(),
            )

            rel_mean_cols.append(mean_col)
            rel_low_cols.append(low_col)
            rel_high_cols.append(high_col)

        df_sub[rel_mean_cols] = df_sub[rel_mean_cols].replace([np.inf, -np.inf], np.nan)
        df_sub[rel_low_cols] = df_sub[rel_low_cols].replace([np.inf, -np.inf], np.nan)
        df_sub[rel_high_cols] = df_sub[rel_high_cols].replace([np.inf, -np.inf], np.nan)

        df_sub["baseline_norm_avg_mean"] = df_sub[rel_mean_cols].mean(axis=1, skipna=True)
        df_sub["baseline_norm_avg_ci_low"] = df_sub[rel_low_cols].mean(axis=1, skipna=True)
        df_sub["baseline_norm_avg_ci_high"] = df_sub[rel_high_cols].mean(axis=1, skipna=True)

        df_sub = df_sub.dropna(subset=["baseline_norm_avg_mean"]).copy()
        if not df_sub.empty:
            rows.append(df_sub)

    if not rows:
        raise ValueError("No valid combinations found with repeat_last baseline.")

    return pd.concat(rows, ignore_index=True)


def aggregate_existing_config_performance(df_perf, dataset):
    config_perf_rows = []

    for keys, g in df_perf.groupby(["method"] + group_cols, dropna=False):
        method = keys[0]
        cfg = keys[1:]

        pooled_mean, pooled_ci_low, pooled_ci_high = aggregate_estimates_with_ci(
            means=g["baseline_norm_avg_mean"].to_numpy(dtype=float),
            ci_lows=g["baseline_norm_avg_ci_low"].to_numpy(dtype=float),
            ci_highs=g["baseline_norm_avg_ci_high"].to_numpy(dtype=float),
            method=HORIZON_AGG_METHOD,
            clip_to_01=False,
        )

        config_perf_rows.append(
            {
                "dataset": dataset,
                "method": method,
                "epi": cfg[0],
                "einn": cfg[1],
                "filter": cfg[2],
                "ti": cfg[3],
                "performance": pooled_mean,
                "performance_ci_low": pooled_ci_low,
                "performance_ci_high": pooled_ci_high,
            }
        )

    return pd.DataFrame(config_perf_rows)


def aggregate_existing_win_rate_for_selected_config(df, selected_row):
    method = selected_row["method"]
    epi = selected_row["epi"]
    einn = selected_row["einn"]
    filt = selected_row["filter"]
    ti = selected_row["ti"]

    g_win = df[
        (df["method"] == method)
        & (df["epi"] == epi)
        & (df["einn"] == einn)
        & (df["filter"] == filt)
        & (df["ti"] == ti)
    ].copy()

    if g_win.empty:
        return np.nan, np.nan, np.nan

    return aggregate_estimates_with_ci(
        means=g_win["win_rate_mean"].to_numpy(dtype=float),
        ci_lows=g_win["win_rate_ci_low"].to_numpy(dtype=float),
        ci_highs=g_win["win_rate_ci_high"].to_numpy(dtype=float),
        method=HORIZON_AGG_METHOD,
        clip_to_01=True,
    )


def build_existing_five_bar_rows(results_dict, metric_col):
    perf_rows = []
    win_rows = []
    selected_rows = []
    metric_cols_for_run = [metric_col]

    for dataset, df in results_dict.items():
        if df is None or len(df) == 0:
            continue

        df = standardize_config_cols(df)

        try:
            df_perf = compute_baseline_norm_rows_with_ci(
                df_in=df,
                metric_cols_for_run=metric_cols_for_run,
                group_cols=group_cols,
            )
        except ValueError as e:
            print(f"Skipping {dataset}, metric={metric_col}: {e}")
            continue

        config_perf_df = aggregate_existing_config_performance(
            df_perf=df_perf,
            dataset=dataset,
        )

        if config_perf_df.empty:
            print(f"Skipping {dataset}, metric={metric_col}: no config-level performance rows")
            continue

        selections = []

        best_arima = choose_best_row(config_perf_df[config_perf_df["method"].map(is_arima)])
        if best_arima is not None:
            selections.append(("ARIMA", best_arima))

        best_dlinear = choose_best_row(config_perf_df[config_perf_df["method"].map(is_dlinear)])
        if best_dlinear is not None:
            selections.append(("Dlinear", best_dlinear))

        best_graph = choose_best_row(
            config_perf_df[config_perf_df["method"].map(is_graph_based_candidate)]
        )
        if best_graph is not None:
            selections.append(("Best graph-based", best_graph))

        for display_method, selected in selections:
            win_mean, win_low, win_high = aggregate_existing_win_rate_for_selected_config(
                df=df,
                selected_row=selected,
            )

            perf_rows.append(
                {
                    "dataset": dataset,
                    "metric": metric_col,
                    "display_method": display_method,
                    "method": selected["method"],
                    "performance": selected["performance"],
                    "performance_ci_low": selected["performance_ci_low"],
                    "performance_ci_high": selected["performance_ci_high"],
                }
            )
            win_rows.append(
                {
                    "dataset": dataset,
                    "metric": metric_col,
                    "display_method": display_method,
                    "method": selected["method"],
                    "win_rate": win_mean,
                    "win_rate_ci_low": win_low,
                    "win_rate_ci_high": win_high,
                }
            )
            selected_rows.append(
                {
                    "dataset": dataset,
                    "metric": metric_col,
                    "display_method": display_method,
                    "selected_method": selected["method"],
                    "epi": selected["epi"],
                    "einn": selected["einn"],
                    "filter": selected["filter"],
                    "ti": selected["ti"],
                    "performance": selected["performance"],
                    "performance_ci_low": selected["performance_ci_low"],
                    "performance_ci_high": selected["performance_ci_high"],
                    "win_rate": win_mean,
                    "win_rate_ci_low": win_low,
                    "win_rate_ci_high": win_high,
                }
            )

    return pd.DataFrame(perf_rows), pd.DataFrame(win_rows), pd.DataFrame(selected_rows)


def prepare_tusoai_df(tusoai_df, metric_cols_for_run):
    df = tusoai_df.copy()

    if "result_schema_version" not in df.columns and df.index.name == "result_schema_version":
        df = df.reset_index()

    required_base = [
        "dataset",
        "method",
        "horizon",
        "win_rate_mean",
        "win_rate_ci_low",
        "win_rate_ci_high",
    ]
    missing_base = [c for c in required_base if c not in df.columns]
    if missing_base:
        raise ValueError(f"tusoai_df is missing expected columns: {missing_base}")

    missing_metrics = []
    for m in metric_cols_for_run:
        for suffix in ["mean", "ci_low", "ci_high"]:
            col = f"{m}_{suffix}"
            if col not in df.columns:
                missing_metrics.append(col)

    if missing_metrics:
        raise ValueError(f"tusoai_df is missing expected metric columns: {missing_metrics}")

    df["method_norm"] = df["method"].map(normalize_method_name)
    return df


def get_tusoai_baseline_config_value(row, col):
    baseline_col = f"baseline_{col}"
    if baseline_col in row.index:
        return normalize_config_value(row[baseline_col])
    return False


def display_name_for_tusoai_method(method):
    m = normalize_method_name(method)
    if m in TUSOAI_SPATIAL_METHODS:
        return "TusoAI (spatial)"
    if m in TUSOAI_NOSPATIAL_METHODS:
        return "TusoAI (no spatial)"
    return None


def get_matching_repeat_last_rows_from_results(
    results_df,
    dataset,
    horizon,
    epi,
    einn,
    filt,
    ti,
):
    df = standardize_config_cols(results_df)

    sub = df[
        (df["method"].map(is_baseline))
        & (df["horizon"] == horizon)
        & (df["epi"] == epi)
        & (df["einn"] == einn)
        & (df["filter"] == filt)
        & (df["ti"] == ti)
    ].copy()

    if sub.empty:
        print(
            f"Missing repeat_last baseline in results_dfs for "
            f"dataset={dataset}, horizon={horizon}, "
            f"epi={epi}, einn={einn}, filter={filt}, ti={ti}"
        )

    return sub


def build_tusoai_long_df_for_shared_normalization(tusoai_df, results_dict, metric_col):
    metric_cols_for_run = [metric_col]
    tuso = prepare_tusoai_df(tusoai_df, metric_cols_for_run=metric_cols_for_run)
    long_parts = []

    for _, row in tuso.iterrows():
        dataset = row["dataset"]
        method = row["method"]
        horizon = row["horizon"]

        display_method = display_name_for_tusoai_method(method)
        if display_method is None:
            continue

        if dataset not in results_dict:
            print(f"Skipping TusoAI row because dataset is missing from results_dfs: {dataset}")
            continue

        epi = get_tusoai_baseline_config_value(row, "epi")
        einn = get_tusoai_baseline_config_value(row, "einn")
        filt = get_tusoai_baseline_config_value(row, "filter")
        ti = get_tusoai_baseline_config_value(row, "ti")

        tuso_row = {
            "dataset": dataset,
            "method": method,
            "horizon": horizon,
            "epi": epi,
            "einn": einn,
            "filter": filt,
            "ti": ti,
            "win_rate_mean": row["win_rate_mean"],
            "win_rate_ci_low": row["win_rate_ci_low"],
            "win_rate_ci_high": row["win_rate_ci_high"],
        }
        for m in metric_cols_for_run:
            tuso_row[f"{m}_mean"] = row[f"{m}_mean"]
            tuso_row[f"{m}_ci_low"] = row[f"{m}_ci_low"]
            tuso_row[f"{m}_ci_high"] = row[f"{m}_ci_high"]

        long_parts.append(pd.DataFrame([tuso_row]))

        baseline_rows = get_matching_repeat_last_rows_from_results(
            results_df=results_dict[dataset],
            dataset=dataset,
            horizon=horizon,
            epi=epi,
            einn=einn,
            filt=filt,
            ti=ti,
        )

        if not baseline_rows.empty:
            keep_cols = (
                ["dataset", "method", "horizon"]
                + group_cols
                + [f"{m}_mean" for m in metric_cols_for_run]
                + [f"{m}_ci_low" for m in metric_cols_for_run]
                + [f"{m}_ci_high" for m in metric_cols_for_run]
            )
            available_keep_cols = [c for c in keep_cols if c in baseline_rows.columns]
            baseline_rows = baseline_rows[available_keep_cols].copy()
            baseline_rows = baseline_rows.drop_duplicates(
                subset=["dataset", "method", "horizon"] + group_cols
            )
            baseline_rows["win_rate_mean"] = 0.5
            baseline_rows["win_rate_ci_low"] = 0.5
            baseline_rows["win_rate_ci_high"] = 0.5
            long_parts.append(baseline_rows)

    if not long_parts:
        raise ValueError(f"No usable TusoAI rows could be paired for metric={metric_col}.")

    long_df = pd.concat(long_parts, ignore_index=True)
    long_df = standardize_config_cols(long_df)
    long_df = long_df.drop_duplicates(
        subset=["dataset", "method", "horizon"] + group_cols,
        keep="first",
    )
    return long_df


def build_tusoai_five_bar_rows(tusoai_df, results_dict, metric_col):
    metric_cols_for_run = [metric_col]
    long_df = build_tusoai_long_df_for_shared_normalization(
        tusoai_df=tusoai_df,
        results_dict=results_dict,
        metric_col=metric_col,
    )

    perf_rows = []
    win_rows = []

    for dataset, df_dataset in long_df.groupby("dataset", dropna=False):
        try:
            df_perf = compute_baseline_norm_rows_with_ci(
                df_in=df_dataset,
                metric_cols_for_run=metric_cols_for_run,
                group_cols=group_cols,
            )
        except ValueError as e:
            print(f"Skipping TusoAI dataset={dataset}, metric={metric_col}: {e}")
            continue

        df_perf = df_perf[~df_perf["method"].map(is_baseline)].copy()

        for method, g_perf in df_perf.groupby("method", dropna=False):
            display_method = display_name_for_tusoai_method(method)
            if display_method is None:
                continue

            perf_mean, perf_low, perf_high = aggregate_estimates_with_ci(
                means=g_perf["baseline_norm_avg_mean"].to_numpy(dtype=float),
                ci_lows=g_perf["baseline_norm_avg_ci_low"].to_numpy(dtype=float),
                ci_highs=g_perf["baseline_norm_avg_ci_high"].to_numpy(dtype=float),
                method=HORIZON_AGG_METHOD,
                clip_to_01=False,
            )

            g_win = df_dataset[df_dataset["method"] == method].copy()

            win_mean, win_low, win_high = aggregate_estimates_with_ci(
                means=g_win["win_rate_mean"].to_numpy(dtype=float),
                ci_lows=g_win["win_rate_ci_low"].to_numpy(dtype=float),
                ci_highs=g_win["win_rate_ci_high"].to_numpy(dtype=float),
                method=HORIZON_AGG_METHOD,
                clip_to_01=True,
            )

            perf_rows.append(
                {
                    "dataset": dataset,
                    "metric": metric_col,
                    "display_method": display_method,
                    "method": method,
                    "performance": perf_mean,
                    "performance_ci_low": perf_low,
                    "performance_ci_high": perf_high,
                }
            )
            win_rows.append(
                {
                    "dataset": dataset,
                    "metric": metric_col,
                    "display_method": display_method,
                    "method": method,
                    "win_rate": win_mean,
                    "win_rate_ci_low": win_low,
                    "win_rate_ci_high": win_high,
                }
            )

    return pd.DataFrame(perf_rows), pd.DataFrame(win_rows)


def build_five_bar_tables(results_dict, tusoai_df, metric_col):
    existing_perf_df, existing_win_df, existing_selection_df = build_existing_five_bar_rows(
        results_dict=results_dict,
        metric_col=metric_col,
    )

    tusoai_perf_df, tusoai_win_df = build_tusoai_five_bar_rows(
        tusoai_df=tusoai_df,
        results_dict=results_dict,
        metric_col=metric_col,
    )

    perf_df = pd.concat([existing_perf_df, tusoai_perf_df], ignore_index=True)
    win_df = pd.concat([existing_win_df, tusoai_win_df], ignore_index=True)

    dataset_order = list(results_dict.keys())

    for d in perf_df["dataset"].drop_duplicates():
        if d not in dataset_order:
            dataset_order.append(d)

    for d in win_df["dataset"].drop_duplicates():
        if d not in dataset_order:
            dataset_order.append(d)

    perf_df["dataset"] = pd.Categorical(
        perf_df["dataset"],
        categories=dataset_order,
        ordered=True,
    )
    win_df["dataset"] = pd.Categorical(
        win_df["dataset"],
        categories=dataset_order,
        ordered=True,
    )

    perf_df["display_method"] = pd.Categorical(
        perf_df["display_method"],
        categories=DISPLAY_METHOD_ORDER,
        ordered=True,
    )
    win_df["display_method"] = pd.Categorical(
        win_df["display_method"],
        categories=DISPLAY_METHOD_ORDER,
        ordered=True,
    )

    perf_df = perf_df.sort_values(["dataset", "display_method"]).reset_index(drop=True)
    win_df = win_df.sort_values(["dataset", "display_method"]).reset_index(drop=True)

    return perf_df, win_df, existing_selection_df


def get_ordered_categories_present(series, categories):
    present = set(series.dropna().astype(str))
    return [c for c in categories if str(c) in present]


def plot_grouped_barplot(
    plot_df,
    value_col,
    ci_low_col,
    ci_high_col,
    title,
    y_label,
    reference_y,
    reference_label,
    reference_linestyle,
    reference_linewidth,
    reference_linecolor,
    ylim,
    figsize,
    save_filename=None,
):
    if plot_df.empty:
        print(f"No data to plot for: {title}")
        return

    if isinstance(plot_df["dataset"].dtype, pd.CategoricalDtype):
        datasets_order = get_ordered_categories_present(
            plot_df["dataset"],
            list(plot_df["dataset"].cat.categories),
        )
    else:
        datasets_order = list(plot_df["dataset"].drop_duplicates().astype(str))

    x = np.arange(len(datasets_order))
    n_methods = len(DISPLAY_METHOD_ORDER)
    bar_width = BAR_GROUP_WIDTH / n_methods

    fig, ax = plt.subplots(figsize=figsize)

    for i, display_method in enumerate(DISPLAY_METHOD_ORDER):
        vals = []
        yerr_low = []
        yerr_high = []

        for dataset in datasets_order:
            sub = plot_df[
                (plot_df["dataset"].astype(str) == str(dataset))
                & (plot_df["display_method"].astype(str) == str(display_method))
            ]

            if sub.empty:
                vals.append(np.nan)
                yerr_low.append(np.nan)
                yerr_high.append(np.nan)
            else:
                row = sub.iloc[0]
                val = row[value_col]
                ci_low = row[ci_low_col]
                ci_high = row[ci_high_col]

                vals.append(val)
                yerr_low.append(val - ci_low if pd.notna(val) and pd.notna(ci_low) else np.nan)
                yerr_high.append(ci_high - val if pd.notna(val) and pd.notna(ci_high) else np.nan)

        vals = np.asarray(vals, dtype=float)
        yerr = np.vstack(
            [
                np.asarray(yerr_low, dtype=float),
                np.asarray(yerr_high, dtype=float),
            ]
        )

        offsets = x - (BAR_GROUP_WIDTH / 2) + (i + 0.5) * bar_width
        color = get_method_color(display_method, i)

        ax.bar(
            offsets,
            vals,
            width=bar_width,
            label=display_method,
            alpha=BAR_ALPHA,
            color=color,
            yerr=yerr if SHOW_ERROR_BARS else None,
            ecolor=ERROR_BAR_COLOR,
            error_kw={
                "elinewidth": ERROR_BAR_LINEWIDTH,
                "capsize": ERROR_BAR_CAPSIZE,
                "capthick": ERROR_BAR_CAPTHICK,
            } if SHOW_ERROR_BARS else None,
        )

    ax.axhline(
        reference_y,
        linestyle=reference_linestyle,
        linewidth=reference_linewidth,
        color=reference_linecolor,
        label=reference_label if not SHOW_LEGEND else None,
    )

    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    ax.set_xlabel(X_LABEL, fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_ylabel(y_label, fontsize=AXIS_LABEL_FONTSIZE)

    ax.set_xticks(x)
    ax.set_xticklabels(
        datasets_order,
        rotation=XTICK_ROTATION,
        ha=XTICK_HA,
        fontsize=XTICK_FONTSIZE,
    )

    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE)
    ax.set_ylim(ylim)

    if SHOW_GRID:
        ax.grid(True, axis=GRID_AXIS, alpha=GRID_ALPHA)

    if SHOW_LEGEND:
        ax.legend(
            loc=LEGEND_LOC,
            bbox_to_anchor=LEGEND_BBOX_TO_ANCHOR,
            fontsize=LEGEND_FONTSIZE,
            frameon=LEGEND_FRAMEON,
            title=LEGEND_TITLE,
        )

    plt.tight_layout()

    if SAVE_FIGURES and save_filename is not None:
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        save_path = SAVE_DIR / save_filename
        plt.savefig(save_path, dpi=SAVE_DPI, bbox_inches=SAVE_BBOX_INCHES)
        print(f"Saved: {save_path}")

    if SHOW_FIGURES:
        plt.show()

    if CLOSE_FIGURES:
        plt.close(fig)


five_perf_dfs = {}
five_win_dfs = {}
best_existing_selection_dfs = {}

for metric_col in metric_cols:
    metric_label = get_metric_label(metric_col)
    print(f"\nMetric: {metric_label}")

    try:
        five_perf_df, five_win_df, best_existing_selection_df = build_five_bar_tables(
            results_dict=RESULTS_DICT,
            tusoai_df=TUSOAI_DF,
            metric_col=metric_col,
        )
    except Exception as e:
        print(f"Failed to build five-method tables for metric={metric_col}: {e}")
        continue

    five_perf_dfs[metric_col] = five_perf_df
    five_win_dfs[metric_col] = five_win_df
    best_existing_selection_dfs[metric_col] = best_existing_selection_df

    print("Selected existing methods/configs:")
    print(best_existing_selection_df)

    print("Final performance table:")
    print(five_perf_df)

    print("Final win-rate table:")
    print(five_win_df)

    plot_grouped_barplot(
        plot_df=five_win_df,
        value_col="win_rate",
        ci_low_col="win_rate_ci_low",
        ci_high_col="win_rate_ci_high",
        title=WIN_TITLE_TEMPLATE.format(metric_label=metric_label),
        y_label=WIN_Y_LABEL,
        reference_y=WIN_REFERENCE_Y,
        reference_label=WIN_REFERENCE_LABEL,
        reference_linestyle=WIN_REFERENCE_LINESTYLE,
        reference_linewidth=WIN_REFERENCE_LINEWIDTH,
        reference_linecolor=WIN_REFERENCE_LINECOLOR,
        ylim=(WIN_YMIN, WIN_YMAX),
        figsize=FIGSIZE_WIN,
        save_filename=WIN_FILENAME_TEMPLATE.format(metric=metric_col),
    )

    plot_grouped_barplot(
        plot_df=five_perf_df,
        value_col="performance",
        ci_low_col="performance_ci_low",
        ci_high_col="performance_ci_high",
        title=PERF_TITLE_TEMPLATE.format(metric_label=metric_label),
        y_label=PERF_Y_LABEL,
        reference_y=PERF_REFERENCE_Y,
        reference_label=PERF_REFERENCE_LABEL,
        reference_linestyle=PERF_REFERENCE_LINESTYLE,
        reference_linewidth=PERF_REFERENCE_LINEWIDTH,
        reference_linecolor=PERF_REFERENCE_LINECOLOR,
        ylim=(PERF_YMIN, PERF_YMAX),
        figsize=FIGSIZE_PERF,
        save_filename=PERF_FILENAME_TEMPLATE.format(metric=metric_col),
    )


In [ ]:
tusoai_df[tusoai_df['dataset'] == 'ILI2019']

In [ ]:
results_dfs['ILI2019']